# Name That LLM: Starter Baseline (TF-IDF + Logistic Regression)


In [1]:
import pathlib

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline

CANDIDATES = [
    pathlib.Path('/kaggle/input/competitions/name-that-llm'),
    pathlib.Path('/kaggle/input/name-that-llm'),
]
DATA_DIR = next(p for p in CANDIDATES if (p / 'train.csv').exists())

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train shape: {train_df.shape} | Unique models: {train_df["model"].nunique()} | Prompts: {train_df["prompt_id"].nunique()}')
print(f'Test shape: {test_df.shape}')
train_df.head(3)

Train shape: (2699, 4) | Unique models: 18 | Prompts: 150
Test shape: (3, 2)


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash


In [7]:
# Build Word + Character N-Gram TF-IDF + Logistic Regression Pipeline
model_pipeline = Pipeline(
    [
        (
            'features',
            FeatureUnion(
                [
                    (
                        'word_tfidf',
                        TfidfVectorizer(
                            ngram_range=(1, 2),
                            max_features=25000,
                            sublinear_tf=True,
                        ),
                    ),
                    (
                        'char_tfidf',
                        TfidfVectorizer(
                            analyzer='char',
                            ngram_range=(3, 5),
                            max_features=35000,
                            sublinear_tf=True,
                        ),
                    ),
                ]
            ),
        ),
        ('clf', LinearSVC(C=9)),
    ]
)

# 5-fold CV that keeps each prompt in a single fold (test prompts are unseen), scored like the leaderboard
cv_scores = cross_val_score(
    model_pipeline,
    train_df['text'].fillna(''),
    train_df['model'],
    groups=train_df['prompt_id'],
    cv=GroupKFold(n_splits=5),
    scoring='balanced_accuracy',
    n_jobs=-1,
)
print(f'5-Fold Grouped CV Balanced Accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

# Fit on full training set
model_pipeline.fit(train_df['text'].fillna(''), train_df['model'])

5-Fold Grouped CV Balanced Accuracy: 0.2960 +/- 0.0141


Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word_tfidf',
                                                 TfidfVectorizer(max_features=25000,
                                                                 ngram_range=(1,
                                                                              2),
                                                                 sublinear_tf=True)),
                                                ('char_tfidf',
                                                 TfidfVectorizer(analyzer='char',
                                                                 max_features=35000,
                                                                 ngram_range=(3,
                                                                              5),
                                                                 sublinear_tf=True))])),
                ('clf', LinearSVC(C=9))])

In [6]:
# Batch inference on test.csv (1 example row interactively; the full hidden test set during the Leaderboard Rerun)
test_preds = model_pipeline.predict(test_df['text'].fillna(''))

submission = pd.DataFrame(
    {
        'id': test_df['id'],
        'model': test_preds,
    }
)

submission.to_parquet('submission.parquet', index=False)
print(f'Saved submission.parquet with {len(submission)} rows.')
submission.head(10)

Saved submission.parquet with 3 rows.


,id,model
0,00353fb1c541,gemini-3.6-flash
1,005f5a9a6139,gpt-5.6-sol
2,0062a53d5487,gemini-3.7-flash


In [14]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Prepare data
X = train_df['text'].fillna('')
y = train_df['model']
groups = train_df['prompt_id']

# Same validation strategy as the original notebook
cv = GroupKFold(n_splits=5)

results = []

# Experiment with different C values
for c in [0.1, 1, 3, 9, 20]:

    model_pipeline = Pipeline([
        ('features', FeatureUnion([
            ('word_tfidf', TfidfVectorizer(
                ngram_range=(1, 2),
                max_features=25000,
                sublinear_tf=True,
            )),
            ('char_tfidf', TfidfVectorizer(
                analyzer='char',
                ngram_range=(3, 5),
                max_features=35000,
                sublinear_tf=True,
            )),
        ])),
        ('clf', LinearSVC(C=c)),
    ])

    # Calculate scores using the same method as the original notebook
    cv_scores = cross_val_score(
        model_pipeline,
        X,
        y,
        groups=groups,
        cv=cv,
        scoring='balanced_accuracy',
        n_jobs=-1,
    )

    results.append({
        'C': c,
        'Mean balanced accuracy': cv_scores.mean(),
        'Std deviation': cv_scores.std(),
    })

    print(
        f'C={c}: {cv_scores.mean():.4f} '
        f'+/- {cv_scores.std():.4f}'
    )

# Display results from best to worst
results_df = pd.DataFrame(results).sort_values(
    'Mean balanced accuracy',
    ascending=False,
).reset_index(drop=True)

display(results_df)

# Select the best C value
best_c = results_df.loc[0, 'C']
print(f'Best C: {best_c}')

# Train the selected pipeline on all training data
model_pipeline = Pipeline([
    ('features', FeatureUnion([
        ('word_tfidf', TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=25000,
            sublinear_tf=True,
        )),
        ('char_tfidf', TfidfVectorizer(
            analyzer='char',
            ngram_range=(3, 5),
            max_features=35000,
            sublinear_tf=True,
        )),
    ])),
    ('clf', LinearSVC(C=best_c)),
])

model_pipeline.fit(X, y)

print(f'Final model trained using C={best_c}')

C=0.1: 0.3045 +/- 0.0123
C=1: 0.2989 +/- 0.0119
C=3: 0.2986 +/- 0.0126
C=9: 0.2960 +/- 0.0141
C=20: 0.2978 +/- 0.0142


,C,Mean balanced accuracy,Std deviation
0,0.1,0.304470,0.012335
1,1.0,0.298927,0.011885
2,3.0,0.298557,0.012609
3,20.0,0.297816,0.014197
4,9.0,0.295964,0.014149


Best C: 0.1
Final model trained using C=0.1


In [15]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC


# 1. Prepare data
X = train_df["text"].fillna("")
y = train_df["model"]
groups = train_df["prompt_id"]

# Use exactly the same five grouped folds for every experiment
cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))


# 2. Define the feature configurations
configs = [
    {
        "name": "Baseline",
        "word_ngram": (1, 2),
        "char_analyzer": "char",
        "char_ngram": (3, 5),
    },
    {
        "name": "Word ngrams (1, 3)",
        "word_ngram": (1, 3),
        "char_analyzer": "char",
        "char_ngram": (3, 5),
    },
    {
        "name": "Character ngrams (2, 6)",
        "word_ngram": (1, 2),
        "char_analyzer": "char",
        "char_ngram": (2, 6),
    },
    {
        "name": "Character within words",
        "word_ngram": (1, 2),
        "char_analyzer": "char_wb",
        "char_ngram": (3, 5),
    },
]


# 3. Build the pipeline
def build_pipeline(config):
    return Pipeline([
        (
            "features",
            FeatureUnion([
                (
                    "word_tfidf",
                    TfidfVectorizer(
                        ngram_range=config["word_ngram"],
                        max_features=25000,
                        sublinear_tf=True,
                    ),
                ),
                (
                    "char_tfidf",
                    TfidfVectorizer(
                        analyzer=config["char_analyzer"],
                        ngram_range=config["char_ngram"],
                        max_features=35000,
                        sublinear_tf=True,
                    ),
                ),
            ]),
        ),
        ("clf", LinearSVC(C=0.1)),
    ])


# 4. Evaluate each configuration
results = []

for config in configs:
    print(f"\nEvaluating: {config['name']}")

    experiment = build_pipeline(config)

    scores = cross_val_score(
        experiment,
        X,
        y,
        groups=groups,
        cv=cv_splits,
        scoring="balanced_accuracy",
        n_jobs=-1,
    )

    results.append({
        "Experiment": config["name"],
        "Mean balanced accuracy": scores.mean(),
        "Std deviation": scores.std(),
        "Fold scores": [round(float(s), 4) for s in scores],
    })

    print(f"Fold scores: {scores.round(4)}")
    print(f"Mean: {scores.mean():.4f} +/- {scores.std():.4f}")


# 5. Compare results
results_df = pd.DataFrame(results).sort_values(
    "Mean balanced accuracy",
    ascending=False,
).reset_index(drop=True)

print("\n========== EXPERIMENT COMPARISON ==========")
display(results_df)

best_name = results_df.loc[0, "Experiment"]
best_config = next(
    config for config in configs
    if config["name"] == best_name
)

print(f"Best configuration: {best_name}")
print(
    f"Best mean balanced accuracy: "
    f"{results_df.loc[0, 'Mean balanced accuracy']:.4f}"
)


# 6. Train the selected model on all training data
model_pipeline = build_pipeline(best_config)
model_pipeline.fit(X, y)

print("Final model trained.")





Evaluating: Baseline
Fold scores: [0.2926 0.287  0.3111 0.3185 0.3131]
Mean: 0.3045 +/- 0.0123

Evaluating: Word ngrams (1, 3)
Fold scores: [0.287  0.2759 0.3185 0.3037 0.3094]
Mean: 0.2989 +/- 0.0154

Evaluating: Character ngrams (2, 6)
Fold scores: [0.2889 0.2741 0.3093 0.313  0.3132]
Mean: 0.2997 +/- 0.0156

Evaluating: Character within words
Fold scores: [0.2815 0.2907 0.2907 0.3019 0.289 ]
Mean: 0.2908 +/- 0.0065

========== EXPERIMENT COMPARISON ==========


,Experiment,Mean balanced accuracy,Std deviation,Fold scores
0,Baseline,0.304470,0.012335,"[0.2926, 0.287, 0.3111, 0.3185, 0.3131]"
1,"Character ngrams (2, 6)",0.299668,0.015632,"[0.2889, 0.2741, 0.3093, 0.313, 0.3132]"
2,"Word ngrams (1, 3)",0.298914,0.015404,"[0.287, 0.2759, 0.3185, 0.3037, 0.3094]"
3,Character within words,0.290766,0.006514,"[0.2815, 0.2907, 0.2907, 0.3019, 0.289]"


Best configuration: Baseline
Best mean balanced accuracy: 0.3045
Final model trained.


In [16]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

# Prepare data
X = train_df["text"].fillna("")
y = train_df["model"]
groups = train_df["prompt_id"]

# Keep exactly the same five grouped validation folds
cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))


# Build the same baseline pipeline
def build_pipeline(class_weight=None):
    return Pipeline([
        ("features", FeatureUnion([
            ("word_tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                max_features=25000,
                sublinear_tf=True,
            )),
            ("char_tfidf", TfidfVectorizer(
                analyzer="char",
                ngram_range=(3, 5),
                max_features=35000,
                sublinear_tf=True,
            )),
        ])),
        ("clf", LinearSVC(
            C=0.1,
            class_weight=class_weight,
        )),
    ])


# Compare normal training vs balanced class weights
results = []

for weight in [None, "balanced"]:
    print(f"\nTesting class_weight={weight}")

    model = build_pipeline(class_weight=weight)

    scores = cross_val_score(
        model,
        X,
        y,
        groups=groups,
        cv=cv_splits,
        scoring="balanced_accuracy",
        n_jobs=-1,
    )

    results.append({
        "class_weight": str(weight),
        "Mean balanced accuracy": scores.mean(),
        "Std deviation": scores.std(),
    })

    print(f"Fold scores: {scores.round(4)}")
    print(f"Mean: {scores.mean():.4f} +/- {scores.std():.4f}")


# Compare the results
results_df = pd.DataFrame(results).sort_values(
    "Mean balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)


Testing class_weight=None
Fold scores: [0.2926 0.287  0.3111 0.3185 0.3131]
Mean: 0.3045 +/- 0.0123

Testing class_weight=balanced
Fold scores: [0.2926 0.287  0.3111 0.3185 0.3131]
Mean: 0.3045 +/- 0.0123


,class_weight,Mean balanced accuracy,Std deviation
0,None,0.30447,0.012335
1,balanced,0.30447,0.012335


In [17]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

# 1. Prepare the data
X = train_df["text"].fillna("")
y = train_df["model"]
groups = train_df["prompt_id"]

# 2. Keep the same validation folds for every experiment
cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))

# Check class distribution
print("Training examples per LLM:")
display(train_df["model"].value_counts().sort_values())

# 3. Define experiments
configs = [
    {"name": "Baseline", "word": 1.0, "char": 1.0},
    {"name": "Word lighter", "word": 0.5, "char": 1.0},
    {"name": "Word stronger", "word": 2.0, "char": 1.0},
    {"name": "Character lighter", "word": 1.0, "char": 0.5},
    {"name": "Character stronger", "word": 1.0, "char": 2.0},
]

# 4. Build the model
def build_pipeline(config):
    return Pipeline([
        ("features", FeatureUnion(
            transformer_list=[
                ("word_tfidf", TfidfVectorizer(
                    ngram_range=(1, 2),
                    max_features=25000,
                    sublinear_tf=True,
                )),
                ("char_tfidf", TfidfVectorizer(
                    analyzer="char",
                    ngram_range=(3, 5),
                    max_features=35000,
                    sublinear_tf=True,
                )),
            ],
            transformer_weights={
                "word_tfidf": config["word"],
                "char_tfidf": config["char"],
            },
        )),
        ("clf", LinearSVC(C=0.1)),
    ])

# 5. Evaluate each configuration
results = []

for config in configs:
    print(f"\nTesting: {config['name']}")

    model = build_pipeline(config)

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv_splits,
        scoring="balanced_accuracy",
        n_jobs=-1,
    )

    results.append({
        "Experiment": config["name"],
        "Word weight": config["word"],
        "Character weight": config["char"],
        "Mean balanced accuracy": scores.mean(),
        "Std deviation": scores.std(),
    })

    print(f"Fold scores: {scores.round(4)}")
    print(f"Mean: {scores.mean():.4f} +/- {scores.std():.4f}")

# 6. Compare results
results_df = pd.DataFrame(results).sort_values(
    "Mean balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

# 7. Select and train the best configuration
best_name = results_df.loc[0, "Experiment"]
best_config = next(
    c for c in configs if c["name"] == best_name
)

model_pipeline = build_pipeline(best_config)
model_pipeline.fit(X, y)

print(f"Best configuration: {best_name}")
print(
    "Best balanced accuracy:",
    round(results_df.loc[0, "Mean balanced accuracy"], 4),
)

Training examples per LLM:


model
claude-fable-5            149
qwen3.8-max               150
gemini-3.6-flash          150
claude-sonnet-5           150
glm-5.3-flash             150
gemini-3.1-pro-preview    150
gemini-3.8-flash          150
gpt-5.6-terra             150
gpt-5.6-luna              150
claude-opus-4-8           150
gemini-3.7-flash          150
claude-opus-5             150
kimi-k3                   150
gpt-5.5                   150
glm-5.3                   150
claude-opus-4-6           150
gpt-6-astra               150
gpt-5.6-sol               150
Name: count, dtype: int64


Testing: Baseline
Fold scores: [0.2926 0.287  0.3111 0.3185 0.3131]
Mean: 0.3045 +/- 0.0123

Testing: Word lighter
Fold scores: [0.3019 0.2611 0.3056 0.3185 0.3131]
Mean: 0.3000 +/- 0.0203

Testing: Word stronger
Fold scores: [0.2685 0.2944 0.287  0.3037 0.2835]
Mean: 0.2874 +/- 0.0117

Testing: Character lighter
Fold scores: [0.2648 0.2907 0.2963 0.3019 0.2983]
Mean: 0.2904 +/- 0.0133

Testing: Character stronger
Fold scores: [0.3037 0.2778 0.3037 0.3315 0.315 ]
Mean: 0.3063 +/- 0.0175


,Experiment,Word weight,Character weight,Mean balanced accuracy,Std deviation
0,Character stronger,1.0,2.0,0.306335,0.017532
1,Baseline,1.0,1.0,0.304470,0.012335
2,Word lighter,0.5,1.0,0.300026,0.020302
3,Character lighter,1.0,0.5,0.290396,0.013287
4,Word stronger,2.0,1.0,0.287433,0.011727


Best configuration: Character stronger
Best balanced accuracy: 0.3063


In [18]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

# Prepare data
X = train_df["text"].fillna("")
y = train_df["model"]
groups = train_df["prompt_id"]

# Same five-fold grouped validation
cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))

# Try different regularization strengths
C_VALUES = [0.03, 0.1, 0.3, 1.0]
results = []

for c in C_VALUES:
    print(f"\nTesting C={c}")

    model = Pipeline([
        ("features", FeatureUnion(
            transformer_list=[
                ("word_tfidf", TfidfVectorizer(
                    ngram_range=(1, 2),
                    max_features=25000,
                    sublinear_tf=True,
                )),
                ("char_tfidf", TfidfVectorizer(
                    analyzer="char",
                    ngram_range=(3, 5),
                    max_features=35000,
                    sublinear_tf=True,
                )),
            ],
            transformer_weights={
                "word_tfidf": 1.0,
                "char_tfidf": 2.0,
            },
        )),
        ("clf", LinearSVC(C=c)),
    ])

    scores = cross_val_score(
        model,
        X,
        y,
        groups=groups,
        cv=cv_splits,
        scoring="balanced_accuracy",
        n_jobs=-1,
    )

    results.append({
        "C": c,
        "Mean balanced accuracy": scores.mean(),
        "Std deviation": scores.std(),
        "Fold scores": scores.round(4).tolist(),
    })

    print(f"Fold scores: {scores.round(4)}")
    print(f"Mean: {scores.mean():.4f} +/- {scores.std():.4f}")

# Compare results
results_df = pd.DataFrame(results).sort_values(
    "Mean balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)


Testing C=0.03
Fold scores: [0.3056 0.2667 0.3074 0.3259 0.3112]
Mean: 0.3034 +/- 0.0197

Testing C=0.1
Fold scores: [0.3037 0.2778 0.3037 0.3315 0.315 ]
Mean: 0.3063 +/- 0.0175

Testing C=0.3
Fold scores: [0.2963 0.2852 0.2981 0.3315 0.3132]
Mean: 0.3049 +/- 0.0160

Testing C=1.0
Fold scores: [0.2907 0.287  0.2944 0.3259 0.3113]
Mean: 0.3019 +/- 0.0146


,C,Mean balanced accuracy,Std deviation,Fold scores
0,0.10,0.306335,0.017532,"[0.3037, 0.2778, 0.3037, 0.3315, 0.315]"
1,0.30,0.304853,0.016019,"[0.2963, 0.2852, 0.2981, 0.3315, 0.3132]"
2,0.03,0.303359,0.019692,"[0.3056, 0.2667, 0.3074, 0.3259, 0.3112]"
3,1.00,0.301890,0.014606,"[0.2907, 0.287, 0.2944, 0.3259, 0.3113]"


In [20]:
import re
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score


# ============================================================
# 1. PREPARE DATA AND VALIDATION FOLDS
# ============================================================

X = train_df["text"].fillna("")
y = train_df["model"]
groups = train_df["prompt_id"]

labels = np.sort(y.unique())

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))


# ============================================================
# 2. CREATE A STYLISTIC FEATURE EXTRACTOR
# ============================================================

class StyleFeatureExtractor(BaseEstimator, TransformerMixin):

    punctuation = [
        ",", ";", ":", ".", "!", "?", "-", "—", "–",
        "'", "’", '"', "“", "”", "(", ")", "[", "]",
        "{", "}", "/", "\\", "*", "_", "#", "`", "…", "•"
    ]

    phrases = [
        "as an ai",
        "as a language model",
        "certainly",
        "of course",
        "here is",
        "here's",
        "let's",
        "in conclusion",
        "in summary",
        "to summarize",
        "for example",
        "for instance",
        "note that",
        "it is important",
        "it's important",
        "overall",
        "however",
        "therefore",
        "moreover",
        "furthermore",
        "firstly",
        "secondly",
        "hope this helps",
    ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.asarray(
            [self.extract_features(text) for text in X],
            dtype=np.float32,
        )

    def extract_features(self, value):

        text = "" if value is None else str(value)
        lower = text.lower()

        char_count = max(len(text), 1)

        words = re.findall(r"\b\w+\b", lower)
        word_count = max(len(words), 1)

        word_lengths = np.asarray(
            [len(word) for word in words],
            dtype=np.float32,
        )

        sentences = [
            s.strip()
            for s in re.split(r"[.!?]+", text)
            if s.strip()
        ]

        sentence_lengths = np.asarray(
            [
                len(re.findall(r"\b\w+\b", sentence))
                for sentence in sentences
            ],
            dtype=np.float32,
        )

        lines = text.splitlines() or [text]

        paragraphs = [
            p for p in re.split(r"\n\s*\n", text)
            if p.strip()
        ]

        # Basic length and vocabulary statistics
        features = [
            np.log1p(len(text)),
            np.log1p(len(words)),
            np.log1p(len(sentences)),
            np.log1p(len(lines)),
            np.log1p(len(paragraphs)),

            float(word_lengths.mean()) if word_lengths.size else 0.0,
            float(word_lengths.std()) if word_lengths.size else 0.0,
            float(word_lengths.max()) if word_lengths.size else 0.0,

            len(set(words)) / word_count,

            sum(len(w) <= 3 for w in words) / word_count,
            sum(len(w) >= 8 for w in words) / word_count,

            float(sentence_lengths.mean())
            if sentence_lengths.size else 0.0,

            float(sentence_lengths.std())
            if sentence_lengths.size else 0.0,

            float(sentence_lengths.max())
            if sentence_lengths.size else 0.0,
        ]

        # Character composition
        features.extend([
            sum(c.isupper() for c in text) / char_count,
            sum(c.islower() for c in text) / char_count,
            sum(c.isdigit() for c in text) / char_count,
            sum(c.isspace() for c in text) / char_count,
            sum(ord(c) > 127 for c in text) / char_count,
        ])

        # Typography: frequency of punctuation per 1,000 characters
        for mark in self.punctuation:
            features.append(
                text.count(mark) / char_count * 1000
            )

        # Formatting and list indicators
        bullet_lines = sum(
            bool(re.match(r"^\s*(?:[-*•]\s+|\d+[.)]\s+)", line))
            for line in lines
        )

        features.extend([
            bullet_lines / max(len(lines), 1),
            float("\n" in text),
            float("```" in text),
            float("##" in text),
            float(text.rstrip().endswith(("!", "?", "."))),
        ])

        # Common stylistic phrase indicators
        for phrase in self.phrases:
            features.append(float(phrase in lower))

        return features


# ============================================================
# 3. BUILD THREE SPECIALIZED CLASSIFIERS
# ============================================================

# Model A: word vocabulary and phrases
word_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=25000,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        C=1.0,
        max_iter=1500,
    )),
])


# Model B: character patterns, including individual characters
char_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(1, 5),
        max_features=100000,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        C=1.0,
        max_iter=1500,
    )),
])


# Model C: writing style and typography
style_model = Pipeline([
    ("style_features", StyleFeatureExtractor()),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        C=1.0,
        max_iter=1500,
    )),
])


# ============================================================
# 4. GENERATE OUT-OF-FOLD PROBABILITIES
# ============================================================

# Each prediction is made by a model that did not train on
# that example or on other examples from the same prompt fold.

print("Evaluating word model...")
word_oof = cross_val_predict(
    word_model, X, y,
    cv=cv_splits,
    method="predict_proba",
    n_jobs=1,
)

print("Evaluating character model...")
char_oof = cross_val_predict(
    char_model, X, y,
    cv=cv_splits,
    method="predict_proba",
    n_jobs=1,
)

print("Evaluating style model...")
style_oof = cross_val_predict(
    style_model, X, y,
    cv=cv_splits,
    method="predict_proba",
    n_jobs=1,
)


# ============================================================
# 5. BLEND MODEL PROBABILITIES
# ============================================================

def geometric_blend(probabilities, weights):

    log_prob = np.zeros_like(
        probabilities[0],
        dtype=np.float64,
    )

    for prob, weight in zip(probabilities, weights):
        if weight > 0:
            log_prob += weight * np.log(
                np.clip(prob, 1e-12, 1.0)
            )

    # Normalize probabilities for each passage
    log_prob -= log_prob.max(axis=1, keepdims=True)

    combined = np.exp(log_prob)

    return combined / combined.sum(axis=1, keepdims=True)


oof_predictions = {
    "Word TF-IDF": word_oof,
    "Character TF-IDF": char_oof,
    "Style features": style_oof,

    "Word + character": geometric_blend(
        [word_oof, char_oof],
        [0.5, 0.5],
    ),

    "All three equally": geometric_blend(
        [word_oof, char_oof, style_oof],
        [1/3, 1/3, 1/3],
    ),

    "Style at 20%": geometric_blend(
        [word_oof, char_oof, style_oof],
        [0.4, 0.4, 0.2],
    ),
}


# ============================================================
# 6. COMPARE USING BALANCED ACCURACY
# ============================================================

results = []

for name, probabilities in oof_predictions.items():

    predictions = labels[np.argmax(probabilities, axis=1)]

    score = balanced_accuracy_score(y, predictions)

    results.append({
        "Method": name,
        "Balanced accuracy": score,
    })

results_df = pd.DataFrame(results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

best_method = results_df.loc[0, "Method"]

print("Best method:", best_method)
print(
    "Best grouped-CV balanced accuracy:",
    round(results_df.loc[0, "Balanced accuracy"], 4),
)


# ============================================================
# 7. FIT ALL THREE MODELS ON FULL TRAINING DATA
# ============================================================

word_model.fit(X, y)
char_model.fit(X, y)
style_model.fit(X, y)


# ============================================================
# 8. PREDICT TEST DATA WITH THE BEST OOF METHOD
# ============================================================

X_test = test_df["text"].fillna("")

word_test = word_model.predict_proba(X_test)
char_test = char_model.predict_proba(X_test)
style_test = style_model.predict_proba(X_test)

test_predictions = {
    "Word TF-IDF": word_test,
    "Character TF-IDF": char_test,
    "Style features": style_test,

    "Word + character": geometric_blend(
        [word_test, char_test],
        [0.5, 0.5],
    ),

    "All three equally": geometric_blend(
        [word_test, char_test, style_test],
        [1/3, 1/3, 1/3],
    ),

    "Style at 20%": geometric_blend(
        [word_test, char_test, style_test],
        [0.4, 0.4, 0.2],
    ),
}

final_predictions = labels[
    np.argmax(test_predictions[best_method], axis=1)
]



Evaluating word model...
Evaluating character model...
Evaluating style model...


,Method,Balanced accuracy
0,Style at 20%,0.348548
1,All three equally,0.325593
2,Character TF-IDF,0.312617
3,Word + character,0.300758
4,Style features,0.285585
5,Word TF-IDF,0.262242


Best method: Style at 20%
Best grouped-CV balanced accuracy: 0.3485


In [21]:
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score


# Combine probabilities using a weighted arithmetic mean
def arithmetic_blend(probabilities, weights):
    combined = sum(
        weight * prob
        for prob, weight in zip(probabilities, weights)
    )

    return combined / combined.sum(axis=1, keepdims=True)


# Combine probabilities using a weighted geometric mean
def geometric_blend(probabilities, weights):
    log_prob = np.zeros_like(
        probabilities[0],
        dtype=np.float64,
    )

    for prob, weight in zip(probabilities, weights):
        log_prob += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    # Numerical stability
    log_prob -= log_prob.max(axis=1, keepdims=True)

    combined = np.exp(log_prob)

    return combined / combined.sum(axis=1, keepdims=True)


# OOF probabilities generated using the same GroupKFold splits
oof_probabilities = [word_oof, char_oof, style_oof]

# Try a small, predefined set of weight combinations
experiments = [
    ("Geometric", (0.20, 0.60, 0.20)),
    ("Geometric", (0.30, 0.50, 0.20)),
    ("Geometric", (0.40, 0.40, 0.20)),
    ("Geometric", (0.50, 0.30, 0.20)),
    ("Geometric", (0.60, 0.20, 0.20)),
    ("Geometric", (0.45, 0.45, 0.10)),
    ("Geometric", (0.35, 0.35, 0.30)),
    ("Geometric", (0.30, 0.30, 0.40)),
    ("Geometric", (0.25, 0.25, 0.50)),
    ("Arithmetic", (0.40, 0.40, 0.20)),
    ("Arithmetic", (0.30, 0.50, 0.20)),
    ("Arithmetic", (0.50, 0.30, 0.20)),
    ("Arithmetic", (1/3, 1/3, 1/3)),
]

results = []

for blend_type, weights in experiments:

    if blend_type == "Geometric":
        probabilities = geometric_blend(
            oof_probabilities, weights
        )
    else:
        probabilities = arithmetic_blend(
            oof_probabilities, weights
        )

    predictions = labels[
        np.argmax(probabilities, axis=1)
    ]

    score = balanced_accuracy_score(y, predictions)

    results.append({
        "Blend": blend_type,
        "Word weight": weights[0],
        "Character weight": weights[1],
        "Style weight": weights[2],
        "Balanced accuracy": score,
    })

results_df = pd.DataFrame(results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

print(
    "Best blend score:",
    round(results_df.loc[0, "Balanced accuracy"], 4)
)

,Blend,Word weight,Character weight,Style weight,Balanced accuracy
0,Geometric,0.450000,0.450000,0.100000,0.357808
1,Geometric,0.200000,0.600000,0.200000,0.354479
2,Geometric,0.300000,0.500000,0.200000,0.351514
3,Geometric,0.400000,0.400000,0.200000,0.348548
4,Geometric,0.500000,0.300000,0.200000,0.344842
5,Geometric,0.600000,0.200000,0.200000,0.342622
6,Geometric,0.350000,0.350000,0.300000,0.328558
7,Geometric,0.300000,0.300000,0.400000,0.317808
8,Arithmetic,0.300000,0.500000,0.200000,0.317069
9,Arithmetic,0.500000,0.300000,0.200000,0.317069


Best blend score: 0.3578


In [22]:
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

def geometric_blend(probabilities, weights):
    log_prob = np.zeros_like(probabilities[0], dtype=float)

    for prob, weight in zip(probabilities, weights):
        log_prob += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    log_prob -= log_prob.max(axis=1, keepdims=True)
    combined = np.exp(log_prob)

    return combined / combined.sum(axis=1, keepdims=True)


# Reuse the existing out-of-fold probabilities
probabilities = [word_oof, char_oof, style_oof]

results = []

# Search combinations close to the current best
for style_weight in [0.05, 0.10, 0.15]:
    for word_weight in [0.35, 0.40, 0.45, 0.50, 0.55]:

        char_weight = 1.0 - word_weight - style_weight

        weights = [
            word_weight,
            char_weight,
            style_weight,
        ]

        blended = geometric_blend(probabilities, weights)

        predictions = labels[np.argmax(blended, axis=1)]

        score = balanced_accuracy_score(y, predictions)

        results.append({
            "Word weight": word_weight,
            "Character weight": char_weight,
            "Style weight": style_weight,
            "Balanced accuracy": score,
        })

results_df = pd.DataFrame(results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df.head(10))

,Word weight,Character weight,Style weight,Balanced accuracy
0,0.40,0.50,0.10,0.358919
1,0.45,0.45,0.10,0.357808
2,0.35,0.55,0.10,0.357069
3,0.50,0.40,0.10,0.355958
4,0.35,0.50,0.15,0.355585
5,0.55,0.35,0.10,0.354104
6,0.35,0.60,0.05,0.354096
7,0.40,0.55,0.05,0.352988
8,0.40,0.45,0.15,0.351884
9,0.50,0.35,0.15,0.351879


In [23]:
import numpy as np
import pandas as pd

from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import balanced_accuracy_score


# --------------------------------------------------
# 1. Build a new model: TF-IDF + calibrated Linear SVM
# --------------------------------------------------

svm_pipeline = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=25000,
            sublinear_tf=True,
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(1, 5),
            max_features=100000,
            sublinear_tf=True,
        )),
    ])),
    ("clf", LinearSVC(C=0.1)),
])

# Calibration converts decision scores into estimated probabilities
try:
    calibrated_svm = CalibratedClassifierCV(
        estimator=svm_pipeline,
        method="sigmoid",
        cv=3,
    )
except TypeError:
    # Compatibility with older scikit-learn versions
    calibrated_svm = CalibratedClassifierCV(
        base_estimator=svm_pipeline,
        method="sigmoid",
        cv=3,
    )


# --------------------------------------------------
# 2. Generate grouped out-of-fold probabilities
# --------------------------------------------------

print("Evaluating calibrated SVM...")

svm_oof = cross_val_predict(
    calibrated_svm,
    X,
    y,
    cv=cv_splits,
    method="predict_proba",
    n_jobs=1,
)

svm_predictions = labels[np.argmax(svm_oof, axis=1)]

svm_score = balanced_accuracy_score(y, svm_predictions)

print(f"Calibrated SVM balanced accuracy: {svm_score:.4f}")


# --------------------------------------------------
# 3. Compare it with the current best blend
# --------------------------------------------------

def geometric_blend(probabilities, weights):
    log_prob = np.zeros_like(
        probabilities[0],
        dtype=np.float64,
    )

    for prob, weight in zip(probabilities, weights):
        log_prob += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    log_prob -= log_prob.max(axis=1, keepdims=True)
    combined = np.exp(log_prob)

    return combined / combined.sum(axis=1, keepdims=True)


old_models = [word_oof, char_oof, style_oof]
old_weights = [0.40, 0.50, 0.10]

results = [{
    "Method": "Existing best blend",
    "SVM weight": 0.0,
    "Balanced accuracy": balanced_accuracy_score(
        y,
        labels[np.argmax(
            geometric_blend(old_models, old_weights),
            axis=1
        )]
    ),
}]

# Add the SVM at different weights
for svm_weight in [0.10, 0.20, 0.30, 0.40]:

    remaining = 1.0 - svm_weight

    weights = [
        old_weights[0] * remaining,
        old_weights[1] * remaining,
        old_weights[2] * remaining,
        svm_weight,
    ]

    probabilities = geometric_blend(
        [word_oof, char_oof, style_oof, svm_oof],
        weights,
    )

    predictions = labels[np.argmax(probabilities, axis=1)]

    score = balanced_accuracy_score(y, predictions)

    results.append({
        "Method": "Blend with calibrated SVM",
        "SVM weight": svm_weight,
        "Balanced accuracy": score,
    })


results_df = pd.DataFrame(results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)


Evaluating calibrated SVM...
Calibrated SVM balanced accuracy: 0.2696


,Method,SVM weight,Balanced accuracy
0,Existing best blend,0.0,0.358919
1,Blend with calibrated SVM,0.1,0.355578
2,Blend with calibrated SVM,0.2,0.349279
3,Blend with calibrated SVM,0.3,0.343346
4,Blend with calibrated SVM,0.4,0.330378


In [24]:
import numpy as np
import pandas as pd

from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import balanced_accuracy_score


# 1. Existing best ensemble
def geometric_blend(probabilities, weights):
    log_prob = np.zeros_like(
        probabilities[0],
        dtype=np.float64,
    )

    for prob, weight in zip(probabilities, weights):
        log_prob += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    log_prob -= log_prob.max(axis=1, keepdims=True)
    combined = np.exp(log_prob)

    return combined / combined.sum(axis=1, keepdims=True)


base_oof = geometric_blend(
    [word_oof, char_oof, style_oof],
    [0.40, 0.50, 0.10],
)

base_predictions = labels[np.argmax(base_oof, axis=1)]
base_score = balanced_accuracy_score(y, base_predictions)

print(f"Existing ensemble: {base_score:.4f}")


# 2. Build Complement Naive Bayes using word + character TF-IDF
def build_nb_pipeline(alpha):
    return Pipeline([
        ("features", FeatureUnion([
            ("word_tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                max_features=25000,
                sublinear_tf=True,
            )),
            ("char_tfidf", TfidfVectorizer(
                analyzer="char",
                ngram_range=(3, 5),
                max_features=35000,
                sublinear_tf=True,
            )),
        ])),
        ("clf", ComplementNB(alpha=alpha)),
    ])


# 3. Evaluate different smoothing strengths
results = []

results.append({
    "Method": "Existing ensemble",
    "Alpha": None,
    "NB weight": 0.0,
    "Balanced accuracy": base_score,
})

for alpha in [0.01, 0.1, 0.5, 1.0]:

    print(f"\nEvaluating ComplementNB alpha={alpha}")

    nb_model = build_nb_pipeline(alpha)

    nb_oof = cross_val_predict(
        nb_model,
        X,
        y,
        cv=cv_splits,
        method="predict_proba",
        n_jobs=-1,
    )

    # Score ComplementNB alone
    nb_predictions = labels[np.argmax(nb_oof, axis=1)]
    nb_score = balanced_accuracy_score(y, nb_predictions)

    results.append({
        "Method": "ComplementNB alone",
        "Alpha": alpha,
        "NB weight": 1.0,
        "Balanced accuracy": nb_score,
    })

    print(f"ComplementNB score: {nb_score:.4f}")

    # Blend ComplementNB with the existing ensemble
    for nb_weight in [0.1, 0.2, 0.3]:

        blended = geometric_blend(
            [base_oof, nb_oof],
            [1.0 - nb_weight, nb_weight],
        )

        predictions = labels[np.argmax(blended, axis=1)]
        score = balanced_accuracy_score(y, predictions)

        results.append({
            "Method": "Ensemble + ComplementNB",
            "Alpha": alpha,
            "NB weight": nb_weight,
            "Balanced accuracy": score,
        })


# 4. Compare all experiments
results_df = pd.DataFrame(results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

print("\nBest experiment:")
display(results_df.head(5))

Existing ensemble: 0.3589

Evaluating ComplementNB alpha=0.01
ComplementNB score: 0.2337

Evaluating ComplementNB alpha=0.1
ComplementNB score: 0.2452

Evaluating ComplementNB alpha=0.5
ComplementNB score: 0.2570

Evaluating ComplementNB alpha=1.0
ComplementNB score: 0.2607


,Method,Alpha,NB weight,Balanced accuracy
0,Ensemble + ComplementNB,1.00,0.2,0.360027
1,Existing ensemble,NaN,0.0,0.358919
2,Ensemble + ComplementNB,0.10,0.1,0.357437
3,Ensemble + ComplementNB,1.00,0.1,0.357064
4,Ensemble + ComplementNB,0.50,0.1,0.356694
5,Ensemble + ComplementNB,0.01,0.1,0.356326
6,Ensemble + ComplementNB,0.50,0.2,0.356324
7,Ensemble + ComplementNB,0.01,0.2,0.354840
8,Ensemble + ComplementNB,0.10,0.2,0.354469
9,Ensemble + ComplementNB,1.00,0.3,0.354096



Best experiment:


,Method,Alpha,NB weight,Balanced accuracy
0,Ensemble + ComplementNB,1.0,0.2,0.360027
1,Existing ensemble,NaN,0.0,0.358919
2,Ensemble + ComplementNB,0.1,0.1,0.357437
3,Ensemble + ComplementNB,1.0,0.1,0.357064
4,Ensemble + ComplementNB,0.5,0.1,0.356694


In [25]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score


# =====================================================
# 1. MAP EACH MODEL TO ITS LAB
# =====================================================

MODEL_TO_FAMILY = {
    "gpt-5.5": "OpenAI",
    "gpt-5.6-luna": "OpenAI",
    "gpt-5.6-sol": "OpenAI",
    "gpt-5.6-terra": "OpenAI",
    "gpt-6-astra": "OpenAI",

    "gemini-3.1-pro-preview": "Google",
    "gemini-3.6-flash": "Google",
    "gemini-3.7-flash": "Google",
    "gemini-3.8-flash": "Google",

    "claude-fable-5": "Anthropic",
    "claude-opus-4-6": "Anthropic",
    "claude-opus-4-8": "Anthropic",
    "claude-opus-5": "Anthropic",
    "claude-sonnet-5": "Anthropic",

    "glm-5.3": "ZAI",
    "glm-5.3-flash": "ZAI",

    "qwen3.8-max": "Qwen",
    "kimi-k3": "Moonshot",
}


# =====================================================
# 2. PREPARE DATA
# =====================================================

X = train_df["text"].fillna("").reset_index(drop=True)
y = train_df["model"].reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)

unknown_models = set(y.unique()) - set(MODEL_TO_FAMILY)

if unknown_models:
    raise ValueError(f"Missing family mappings: {unknown_models}")

all_models = np.array(sorted(y.unique()))
model_to_col = {
    model: i for i, model in enumerate(all_models)
}

families = sorted(set(MODEL_TO_FAMILY.values()))

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))


# =====================================================
# 3. CREATE TEXT FEATURES
# =====================================================

def build_features():
    return FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=25000,
            sublinear_tf=True,
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(1, 5),
            max_features=100000,
            sublinear_tf=True,
        )),
    ])


# =====================================================
# 4. RUN TWO-STAGE GROUPED CROSS-VALIDATION
# =====================================================

oof_probs = np.zeros((len(X), len(all_models)))
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv_splits, start=1):

    print(f"\n========== FOLD {fold}/5 ==========")

    X_train_text = X.iloc[train_idx]
    X_valid_text = X.iloc[valid_idx]

    y_train = y.iloc[train_idx].to_numpy()
    y_valid = y.iloc[valid_idx].to_numpy()

    # Learn features only from this fold's training texts
    features = build_features()

    X_train_features = features.fit_transform(X_train_text)
    X_valid_features = features.transform(X_valid_text)

    # Map training labels to their families
    y_train_family = np.array([
        MODEL_TO_FAMILY[model] for model in y_train
    ])

    # ---------------------------------------------
    # STAGE 1: PREDICT THE LAB
    # ---------------------------------------------

    family_model = LogisticRegression(
        C=1.0,
        max_iter=2000,
    )

    family_model.fit(
        X_train_features,
        y_train_family,
    )

    family_probs = family_model.predict_proba(
        X_valid_features
    )

    family_to_col = {
        family: i
        for i, family in enumerate(family_model.classes_)
    }

    # ---------------------------------------------
    # STAGE 2: PREDICT THE EXACT MODEL WITHIN EACH LAB
    # ---------------------------------------------

    for family in families:

        # Probability that each validation text belongs to this lab
        p_family = family_probs[
            :,
            family_to_col[family]
        ]

        # Select training rows belonging to this lab
        family_mask = np.array([
            MODEL_TO_FAMILY[model] == family
            for model in y_train
        ])

        X_family = X_train_features[family_mask]
        y_family = y_train[family_mask]

        family_labels = np.unique(y_family)

        # If the lab has only one candidate model,
        # its conditional probability is 1.
        if len(family_labels) == 1:

            model_label = family_labels[0]
            col = model_to_col[model_label]

            oof_probs[valid_idx, col] = p_family

        else:
            # Train a specialized classifier for this lab
            local_model = LogisticRegression(
                C=1.0,
                max_iter=2000,
            )

            local_model.fit(X_family, y_family)

            local_probs = local_model.predict_proba(
                X_valid_features
            )

            # Final probability:
            # P(exact model) = P(lab) * P(model | lab)
            for j, model_label in enumerate(
                local_model.classes_
            ):

                col = model_to_col[model_label]

                oof_probs[valid_idx, col] = (
                    p_family * local_probs[:, j]
                )

    # Evaluate this fold using all 18 model labels
    fold_predictions = all_models[
        np.argmax(oof_probs[valid_idx], axis=1)
    ]

    score = balanced_accuracy_score(
        y_valid,
        fold_predictions,
    )

    fold_scores.append(score)

    print(f"Fold balanced accuracy: {score:.4f}")


# =====================================================
# 5. DISPLAY FINAL CROSS-VALIDATION SCORE
# =====================================================

print("\n========== FINAL RESULTS ==========")

for i, score in enumerate(fold_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print(
    f"\nTwo-stage 5-fold Grouped CV Balanced Accuracy: "
    f"{np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}"
)

pooled_score = balanced_accuracy_score(
    y,
    all_models[np.argmax(oof_probs, axis=1)],
)

print(f"Pooled out-of-fold balanced accuracy: {pooled_score:.4f}")

print("\nPrevious best ensemble benchmark: 0.3589")


========== FOLD 1/5 ==========
Fold balanced accuracy: 0.2815

========== FOLD 2/5 ==========
Fold balanced accuracy: 0.3074

========== FOLD 3/5 ==========
Fold balanced accuracy: 0.3389

========== FOLD 4/5 ==========
Fold balanced accuracy: 0.3630

========== FOLD 5/5 ==========
Fold balanced accuracy: 0.3169

========== FINAL RESULTS ==========
Fold 1: 0.2815
Fold 2: 0.3074
Fold 3: 0.3389
Fold 4: 0.3630
Fold 5: 0.3169

Two-stage 5-fold Grouped CV Balanced Accuracy: 0.3215 +/- 0.0277
Pooled out-of-fold balanced accuracy: 0.3215

Previous best ensemble benchmark: 0.3589


In [26]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from sklearn.model_selection import GroupKFold
from sklearn.metrics import adjusted_rand_score

X = train_df["text"].fillna("")
y = train_df["model"]
groups = train_df["prompt_id"]

cv = GroupKFold(n_splits=5)

results = []

for fold, (_, valid_idx) in enumerate(
    cv.split(X, y, groups), start=1
):
    texts = X.iloc[valid_idx].tolist()
    true_prompts = groups.iloc[valid_idx].to_numpy()

    # Approximately 18 responses per prompt
    n_clusters = max(1, round(len(texts) / 18))

    # Represent passages using words and word pairs
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=40000,
        min_df=2,
        stop_words="english",
        sublinear_tf=True,
    )

    features = vectorizer.fit_transform(texts)

    # Calculate similarity between every pair of passages
    similarity = cosine_similarity(features)

    # Convert similarity into distance
    distance = np.clip(1.0 - similarity, 0.0, 2.0)
    np.fill_diagonal(distance, 0.0)

    # Cluster passages that appear to answer similar prompts
    clusterer = AgglomerativeClustering(
        n_clusters=n_clusters,
        metric="precomputed",
        linkage="average",
    )

    predicted_clusters = clusterer.fit_predict(distance)

    # Diagnostic only: prompt IDs are NOT used in clustering
    ari = adjusted_rand_score(
        true_prompts,
        predicted_clusters,
    )

    results.append({
        "Fold": fold,
        "Number of clusters": n_clusters,
        "Adjusted Rand Index": ari,
    })

    print(
        f"Fold {fold}: "
        f"clusters={n_clusters}, ARI={ari:.4f}"
    )

results_df = pd.DataFrame(results)
display(results_df)

print(
    "Mean clustering ARI:",
    round(results_df["Adjusted Rand Index"].mean(), 4),
)

Fold 1: clusters=30, ARI=1.0000
Fold 2: clusters=30, ARI=1.0000
Fold 3: clusters=30, ARI=1.0000
Fold 4: clusters=30, ARI=1.0000
Fold 5: clusters=30, ARI=0.9573


,Fold,Number of clusters,Adjusted Rand Index
0,1,30,1.000000
1,2,30,1.000000
2,3,30,1.000000
3,4,30,1.000000
4,5,30,0.957254


Mean clustering ARI: 0.9915


In [27]:
import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold


# 1. Existing best ensemble
labels = np.sort(y.unique())

def geometric_blend(probabilities, weights):
    log_prob = np.zeros_like(
        probabilities[0], dtype=np.float64
    )

    for prob, weight in zip(probabilities, weights):
        log_prob += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    log_prob -= log_prob.max(axis=1, keepdims=True)
    combined = np.exp(log_prob)

    return combined / combined.sum(axis=1, keepdims=True)


oof_probs = geometric_blend(
    [word_oof, char_oof, style_oof],
    [0.40, 0.50, 0.10],
)

baseline_predictions = labels[np.argmax(oof_probs, axis=1)]
constrained_predictions = baseline_predictions.copy()


# 2. Recover clusters separately inside each validation fold
cv = GroupKFold(n_splits=5)

fold_results = []
cluster_sizes_seen = []

for fold, (_, valid_idx) in enumerate(
    cv.split(X, y, groups), start=1
):
    texts = X.iloc[valid_idx].tolist()

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=40000,
        min_df=2,
        stop_words="english",
        sublinear_tf=True,
    )

    features = vectorizer.fit_transform(texts)

    similarity = (features @ features.T).toarray()
    distance = np.maximum(0.0, 1.0 - similarity)
    np.fill_diagonal(distance, 0.0)

    n_clusters = round(len(valid_idx) / 18)

    clusterer = AgglomerativeClustering(
        n_clusters=n_clusters,
        metric="precomputed",
        linkage="average",
    )

    cluster_ids = clusterer.fit_predict(distance)

    # 3. Assign model labels uniquely within valid clusters
    updated = 0

    for cluster_id in np.unique(cluster_ids):
        local_rows = np.where(cluster_ids == cluster_id)[0]
        global_rows = valid_idx[local_rows]

        size = len(global_rows)
        cluster_sizes_seen.append(size)

        # Training data has approximately 18 responses per prompt.
        # Test-set structure may differ, so skip unexpected sizes.
        if size not in (17, 18):
            continue

        log_probs = np.log(
            np.clip(oof_probs[global_rows], 1e-12, 1.0)
        )

        # If one response is missing, add a dummy row.
        # Its assigned label represents the missing model.
        if size == 17:
            log_probs = np.vstack([
                log_probs,
                np.zeros((1, len(labels))),
            ])

        # Find the assignment maximizing total log probability
        row_ind, col_ind = linear_sum_assignment(-log_probs)

        for r, c in zip(row_ind, col_ind):
            if r < size:
                constrained_predictions[global_rows[r]] = labels[c]

        updated += 1

    baseline_fold_score = balanced_accuracy_score(
        y.iloc[valid_idx],
        baseline_predictions[valid_idx],
    )

    constrained_fold_score = balanced_accuracy_score(
        y.iloc[valid_idx],
        constrained_predictions[valid_idx],
    )

    fold_results.append({
        "Fold": fold,
        "Baseline BA": baseline_fold_score,
        "Constrained BA": constrained_fold_score,
        "Clusters constrained": updated,
    })

    print(
        f"Fold {fold}: "
        f"baseline={baseline_fold_score:.4f}, "
        f"constrained={constrained_fold_score:.4f}, "
        f"clusters used={updated}/{n_clusters}"
    )


# 4. Compare results
results_df = pd.DataFrame(fold_results)
display(results_df)

print(
    "\nBaseline pooled BA:",
    round(
        balanced_accuracy_score(y, baseline_predictions), 4
    ),
)

print(
    "Constrained pooled BA:",
    round(
        balanced_accuracy_score(y, constrained_predictions), 4
    ),
)

print(
    "Observed cluster sizes:",
    pd.Series(cluster_sizes_seen).value_counts().sort_index().to_dict(),
)

Fold 1: baseline=0.3481, constrained=0.5037, clusters used=30/30
Fold 2: baseline=0.3222, constrained=0.4556, clusters used=30/30
Fold 3: baseline=0.4037, constrained=0.4944, clusters used=30/30
Fold 4: baseline=0.3815, constrained=0.5019, clusters used=30/30
Fold 5: baseline=0.3391, constrained=0.4542, clusters used=27/30


,Fold,Baseline BA,Constrained BA,Clusters constrained
0,1,0.348148,0.503704,30
1,2,0.322222,0.455556,30
2,3,0.403704,0.494444,30
3,4,0.381481,0.501852,30
4,5,0.339080,0.454215,27



Baseline pooled BA: 0.3589
Constrained pooled BA: 0.482
Observed cluster sizes: {5: 1, 13: 1, 17: 1, 18: 146, 36: 1}


In [2]:
# ============================================================
# 1. IMPORTS AND DATA
# ============================================================

import pathlib
import warnings
import numpy as np
import pandas as pd
import torch
import transformers

from packaging.version import Version
from tqdm.auto import tqdm

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore", category=UserWarning)

# Load data if not already loaded
if "train_df" not in globals() or "test_df" not in globals():

    CANDIDATES = [
        pathlib.Path("/kaggle/input/competitions/name-that-llm"),
        pathlib.Path("/kaggle/input/name-that-llm"),
    ]

    DATA_DIR = next(
        (p for p in CANDIDATES if (p / "train.csv").exists()),
        None,
    )

    if DATA_DIR is None:
        raise FileNotFoundError(
            "Could not find train.csv. Attach the competition dataset."
        )

    train_df = pd.read_csv(DATA_DIR / "train.csv")
    test_df = pd.read_csv(DATA_DIR / "test.csv")

X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].astype(str).reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)

labels = np.sort(y.unique())

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))

print("Training rows:", len(X))
print("Test rows:", len(test_df))
print("Classes:", len(labels))


# ============================================================
# 2. LOCATE OR DOWNLOAD QWEN3 EMBEDDING
# ============================================================

from pathlib import Path

model_dir = None

# First, look through attached Kaggle model inputs.
for config_file in Path("/kaggle/input").rglob("config.json"):

    candidate = config_file.parent
    path_text = str(candidate).lower()

    is_qwen_embedding = (
        "qwen" in path_text
        and "embedding" in path_text
        and any(token in path_text for token in [
            "0.6b", "0-6b", "0_6b", "06b"
        ])
    )

    if is_qwen_embedding:
        model_dir = candidate
        break

# If it is not attached, download it using KaggleHub.
if model_dir is None:
    try:
        import kagglehub

        model_dir = Path(
            kagglehub.model_download(
                "qwen-lm/qwen-3-embedding/transformers/0.6b"
            )
        )

    except Exception as exc:
        raise RuntimeError(
            "Could not find or download Qwen3 Embedding 0.6B.\n"
            "Enable Internet in Kaggle settings, or attach the model "
            "using Add Input -> Models -> Qwen 3 Embedding -> Transformers 0.6b."
        ) from exc

if not (Path(model_dir) / "config.json").exists():
    raise FileNotFoundError(
        f"Model directory has no config.json: {model_dir}"
    )

print("Model directory:", model_dir)


# ============================================================
# 3. LOAD THE PRETRAINED MODEL
# ============================================================

from transformers import AutoTokenizer, AutoModel

if Version(transformers.__version__) < Version("4.51.0"):
    raise RuntimeError(
        f"Transformers version {transformers.__version__} is too old. "
        "Qwen3 requires transformers >= 4.51.0."
    )

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    dtype = torch.float16
    batch_size = 4
else:
    dtype = torch.float32
    batch_size = 2
    print("Warning: CPU inference may be slow.")

tokenizer = AutoTokenizer.from_pretrained(
    str(model_dir),
    local_files_only=True,
    padding_side="left",
)

tokenizer.padding_side = "left"

model = AutoModel.from_pretrained(
    str(model_dir),
    local_files_only=True,
    torch_dtype=dtype,
)

model.to(device)
model.eval()

print("Device:", device)
print("Model loaded successfully.")


# ============================================================
# 4. CONVERT TEXT INTO EMBEDDINGS
# ============================================================

def encode_texts(texts, batch_size=4, max_length=1024):

    embeddings_list = []

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Creating embeddings",
    ):

        batch_texts = texts[start:start + batch_size]

        tokens = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        tokens = {
            key: value.to(device)
            for key, value in tokens.items()
        }

        with torch.inference_mode():

            outputs = model(**tokens)

            # Left padding means the final position is the
            # final actual token for each passage.
            batch_embeddings = outputs.last_hidden_state[:, -1, :]

            # Normalize each passage embedding.
            batch_embeddings = torch.nn.functional.normalize(
                batch_embeddings,
                p=2,
                dim=1,
            )

        embeddings_list.append(
            batch_embeddings.float().cpu().numpy()
        )

    if not embeddings_list:
        return np.empty((0, model.config.hidden_size), dtype=np.float32)

    return np.concatenate(embeddings_list, axis=0)


all_texts = X.tolist() + test_df["text"].fillna("").astype(str).tolist()

all_embeddings = encode_texts(
    all_texts,
    batch_size=batch_size,
    max_length=1024,
)

X_embed = all_embeddings[:len(X)]
X_test_embed = all_embeddings[len(X):]

del all_embeddings

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Training embeddings:", X_embed.shape)
print("Test embeddings:", X_test_embed.shape)


# ============================================================
# 5. BASELINE MODELS — REUSE OOF OR BUILD THEM
# ============================================================

def build_word_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=25000,
            sublinear_tf=True,
        )),
        ("clf", LogisticRegression(
            C=1.0,
            max_iter=1500,
        )),
    ])


def build_char_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(1, 5),
            max_features=100000,
            sublinear_tf=True,
        )),
        ("clf", LogisticRegression(
            C=1.0,
            max_iter=1500,
        )),
    ])


class StyleFeatureExtractor(BaseEstimator, TransformerMixin):

    punctuation = [
        ",", ";", ":", ".", "!", "?", "-", "—", "–",
        "'", "’", '"', "“", "”", "(", ")", "[", "]",
        "{", "}", "/", "\\", "*", "_", "#", "`", "…", "•"
    ]

    phrases = [
        "as an ai", "as a language model", "certainly",
        "of course", "here is", "here's", "let's",
        "in conclusion", "in summary", "to summarize",
        "for example", "for instance", "note that",
        "it is important", "it's important", "overall",
        "however", "therefore", "moreover", "furthermore",
        "firstly", "secondly", "hope this helps",
    ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.asarray(
            [self.extract_features(text) for text in X],
            dtype=np.float32,
        )

    def extract_features(self, value):

        text = "" if value is None else str(value)
        lower = text.lower()

        char_count = max(len(text), 1)
        words = __import__("re").findall(r"\b\w+\b", lower)
        word_count = max(len(words), 1)

        word_lengths = np.asarray(
            [len(w) for w in words],
            dtype=np.float32,
        )

        import re

        sentences = [
            s.strip()
            for s in re.split(r"[.!?]+", text)
            if s.strip()
        ]

        sentence_lengths = np.asarray(
            [
                len(re.findall(r"\b\w+\b", s))
                for s in sentences
            ],
            dtype=np.float32,
        )

        lines = text.splitlines() or [text]

        paragraphs = [
            p for p in re.split(r"\n\s*\n", text)
            if p.strip()
        ]

        features = [
            np.log1p(len(text)),
            np.log1p(len(words)),
            np.log1p(len(sentences)),
            np.log1p(len(lines)),
            np.log1p(len(paragraphs)),
            float(word_lengths.mean()) if word_lengths.size else 0.0,
            float(word_lengths.std()) if word_lengths.size else 0.0,
            float(word_lengths.max()) if word_lengths.size else 0.0,
            len(set(words)) / word_count,
            sum(len(w) <= 3 for w in words) / word_count,
            sum(len(w) >= 8 for w in words) / word_count,
            float(sentence_lengths.mean()) if sentence_lengths.size else 0.0,
            float(sentence_lengths.std()) if sentence_lengths.size else 0.0,
            float(sentence_lengths.max()) if sentence_lengths.size else 0.0,
        ]

        features.extend([
            sum(c.isupper() for c in text) / char_count,
            sum(c.islower() for c in text) / char_count,
            sum(c.isdigit() for c in text) / char_count,
            sum(c.isspace() for c in text) / char_count,
            sum(ord(c) > 127 for c in text) / char_count,
        ])

        for mark in self.punctuation:
            features.append(text.count(mark) / char_count * 1000)

        bullet_lines = sum(
            bool(re.match(r"^\s*(?:[-*•]\s+|\d+[.)]\s+)", line))
            for line in lines
        )

        features.extend([
            bullet_lines / max(len(lines), 1),
            float("\n" in text),
            float("```" in text),
            float("##" in text),
            float(text.rstrip().endswith(("!", "?", "."))),
        ])

        for phrase in self.phrases:
            features.append(float(phrase in lower))

        return features


def build_style_model():
    return Pipeline([
        ("style_features", StyleFeatureExtractor()),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(
            C=1.0,
            max_iter=1500,
        )),
    ])


have_baseline_oof = all(
    name in globals()
    for name in ["word_oof", "char_oof", "style_oof"]
)

if have_baseline_oof:
    have_baseline_oof = all(
        np.asarray(globals()[name]).shape == (len(X), len(labels))
        for name in ["word_oof", "char_oof", "style_oof"]
    )

if not have_baseline_oof:

    print("Existing baseline OOF predictions not found. Rebuilding...")

    print("Generating word-model OOF predictions...")
    word_oof = cross_val_predict(
        build_word_model(), X, y,
        cv=cv_splits,
        method="predict_proba",
        n_jobs=1,
    )

    print("Generating character-model OOF predictions...")
    char_oof = cross_val_predict(
        build_char_model(), X, y,
        cv=cv_splits,
        method="predict_proba",
        n_jobs=1,
    )

    print("Generating style-model OOF predictions...")
    style_oof = cross_val_predict(
        build_style_model(), X, y,
        cv=cv_splits,
        method="predict_proba",
        n_jobs=1,
    )

print("Baseline OOF probabilities are ready.")


# ============================================================
# 6. EVALUATE EMBEDDING CLASSIFIER
# ============================================================

embedding_results = []

best_embedding_score = -1
best_embedding_c = None
best_embedding_oof = None

for c in [0.03, 0.1, 0.3, 1.0, 3.0]:

    print(f"\nEvaluating embedding Logistic Regression C={c}")

    classifier = LogisticRegression(
        C=c,
        max_iter=2000,
    )

    oof_probs = cross_val_predict(
        classifier,
        X_embed,
        y,
        cv=cv_splits,
        method="predict_proba",
        n_jobs=1,
    )

    predictions = labels[np.argmax(oof_probs, axis=1)]

    score = balanced_accuracy_score(y, predictions)

    embedding_results.append({
        "C": c,
        "Balanced accuracy": score,
    })

    print(f"Balanced accuracy: {score:.4f}")

    if score > best_embedding_score:
        best_embedding_score = score
        best_embedding_c = c
        best_embedding_oof = oof_probs.copy()


embedding_results_df = pd.DataFrame(embedding_results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

print("\n========== EMBEDDING RESULTS ==========")
display(embedding_results_df)

print("Best C:", best_embedding_c)
print("Best embedding score:", round(best_embedding_score, 4))


# ============================================================
# 7. COMPARE AGAINST THE EXISTING ENSEMBLE
# ============================================================

def geometric_blend(probabilities, weights):

    log_probs = np.zeros_like(
        probabilities[0],
        dtype=np.float64,
    )

    for probs, weight in zip(probabilities, weights):
        log_probs += weight * np.log(
            np.clip(probs, 1e-12, 1.0)
        )

    log_probs -= log_probs.max(axis=1, keepdims=True)

    combined = np.exp(log_probs)

    return combined / combined.sum(axis=1, keepdims=True)


def rank_scores(probabilities):

    from scipy.stats import rankdata

    ranks = rankdata(
        probabilities,
        axis=1,
        method="average",
    )

    return (ranks - 1) / (probabilities.shape[1] - 1)


baseline_oof = geometric_blend(
    [word_oof, char_oof, style_oof],
    [0.40, 0.50, 0.10],
)

baseline_score = balanced_accuracy_score(
    y,
    labels[np.argmax(baseline_oof, axis=1)],
)

print(f"\nExisting ensemble score: {baseline_score:.4f}")

baseline_ranks = rank_scores(baseline_oof)
embedding_ranks = rank_scores(best_embedding_oof)

blend_results = [{
    "Method": "Existing ensemble",
    "Embedding weight": 0.0,
    "Balanced accuracy": baseline_score,
}]

blend_results.append({
    "Method": "Embedding alone",
    "Embedding weight": 1.0,
    "Balanced accuracy": best_embedding_score,
})

for embedding_weight in [0.20, 0.35, 0.50, 0.65]:

    combined = (
        (1 - embedding_weight) * baseline_ranks
        + embedding_weight * embedding_ranks
    )

    predictions = labels[np.argmax(combined, axis=1)]

    score = balanced_accuracy_score(y, predictions)

    blend_results.append({
        "Method": "Rank blend",
        "Embedding weight": embedding_weight,
        "Balanced accuracy": score,
    })


blend_results_df = pd.DataFrame(blend_results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

print("\n========== BLEND COMPARISON ==========")
display(blend_results_df)

best_embedding_weight = float(
    blend_results_df.iloc[0]["Embedding weight"]
)

print("Selected embedding weight:", best_embedding_weight)


# ============================================================
# 8. FIT ALL MODELS ON THE COMPLETE TRAINING DATA
# ============================================================

word_model = build_word_model()
char_model = build_char_model()
style_model = build_style_model()

print("\nTraining final baseline models...")

word_model.fit(X, y)
char_model.fit(X, y)
style_model.fit(X, y)

print("Training final embedding classifier...")

embedding_model = LogisticRegression(
    C=best_embedding_c,
    max_iter=2000,
)

embedding_model.fit(X_embed, y)


# ============================================================
# 9. GENERATE TEST PREDICTIONS
# ============================================================

test_texts = test_df["text"].fillna("").astype(str)

word_test = word_model.predict_proba(test_texts)
char_test = char_model.predict_proba(test_texts)
style_test = style_model.predict_proba(test_texts)

baseline_test = geometric_blend(
    [word_test, char_test, style_test],
    [0.40, 0.50, 0.10],
)

embedding_test = embedding_model.predict_proba(X_test_embed)

baseline_test_ranks = rank_scores(baseline_test)
embedding_test_ranks = rank_scores(embedding_test)

final_scores = (
    (1 - best_embedding_weight) * baseline_test_ranks
    + best_embedding_weight * embedding_test_ranks
)

final_predictions = labels[np.argmax(final_scores, axis=1)]


# ============================================================
# 10. SAVE SUBMISSION
# ============================================================

submission = pd.DataFrame({
    "id": test_df["id"],
    "model": final_predictions,
})

submission.to_parquet(
    "/kaggle/working/submission.parquet",
    index=False,
)

print("\nSubmission saved successfully.")
print("Rows:", len(submission))
display(submission.head(10))

Training rows: 2699
Test rows: 3
Classes: 18
Model directory: /kaggle/input/models/qwen-lm/qwen-3-embedding/transformers/0.6b/1


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Device: cuda
Model loaded successfully.


Creating embeddings:   0%|          | 0/676 [00:00<?, ?it/s]

...

,C,Balanced accuracy
0,3.00,0.167427
1,1.00,0.164822
2,0.30,0.154074
3,0.10,0.141111
4,0.03,0.135926


Best C: 3.0
Best embedding score: 0.1674

Existing ensemble score: 0.3600

========== BLEND COMPARISON ==========


,Method,Embedding weight,Balanced accuracy
0,Existing ensemble,0.00,0.360032
1,Rank blend,0.20,0.354119
2,Rank blend,0.35,0.323005
3,Rank blend,0.50,0.299302
4,Rank blend,0.65,0.263731
5,Embedding alone,1.00,0.167427


Selected embedding weight: 0.0

Training final baseline models...
Training final embedding classifier...

Submission saved successfully.
Rows: 3


,id,model
0,00353fb1c541,gemini-3.6-flash
1,005f5a9a6139,gpt-5.6-sol
2,0062a53d5487,gemini-3.7-flash


In [3]:
# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import random
import torch

from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score

from scipy.stats import rankdata


# ============================================================
# 2. CONFIGURATION
# ============================================================

MODEL_NAME = "google-bert/bert-base-cased"

MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = DEVICE.type == "cuda"

print("Device:", DEVICE)

if DEVICE.type != "cuda":
    print(
        "WARNING: BERT fine-tuning on CPU will be slow. "
        "Enable a GPU in Kaggle settings."
    )


# ============================================================
# 3. PREPARE DATA
# ============================================================

X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].astype(str).reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

labels = label_encoder.classes_
NUM_LABELS = len(labels)

print("Training examples:", len(X))
print("Number of classes:", NUM_LABELS)

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y_encoded, groups))

# Tokenizer does not use labels, so we can tokenize once.
# BERT will process at most 256 tokens per passage in this run.

# If the model is attached as a Kaggle input, use its local path.
local_model_paths = [
    p.parent
    for p in Path("/kaggle/input").rglob("config.json")
    if "bert-base-cased" in str(p.parent).lower()
]

if local_model_paths:
    MODEL_NAME = str(local_model_paths[0])

print("Model source:", MODEL_NAME)


# ============================================================
# 4. LOAD TOKENIZER AND TOKENIZE TEXT
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=Path(MODEL_NAME).exists(),
)

encoded = tokenizer(
    X.tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

input_ids = encoded["input_ids"]
attention_masks = encoded["attention_mask"]

print("Token tensor shape:", input_ids.shape)


# ============================================================
# 5. FUNCTION TO CREATE A FRESH BERT CLASSIFIER
# ============================================================

def build_bert():

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label={
            i: str(label)
            for i, label in enumerate(labels)
        },
        label2id={
            str(label): i
            for i, label in enumerate(labels)
        },
    )

    return model.to(DEVICE)


# ============================================================
# 6. FIVE-FOLD GROUPED CROSS-VALIDATION
# ============================================================

bert_oof_probs = np.zeros(
    (len(X), NUM_LABELS),
    dtype=np.float32,
)

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    cv_splits, start=1
):

    print(f"\n{'=' * 45}")
    print(f"FOLD {fold}/5")
    print(f"{'=' * 45}")

    # Reproducibility
    seed = 42 + fold
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # --------------------------------------------------------
    # Create the fold-specific datasets
    # --------------------------------------------------------

    train_indices = torch.tensor(
        train_idx, dtype=torch.long
    )
    valid_indices = torch.tensor(
        valid_idx, dtype=torch.long
    )

    train_dataset = TensorDataset(
        input_ids[train_indices],
        attention_masks[train_indices],
        torch.tensor(
            y_encoded[train_idx],
            dtype=torch.long,
        ),
    )

    valid_dataset = TensorDataset(
        input_ids[valid_indices],
        attention_masks[valid_indices],
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        pin_memory=USE_AMP,
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=32,
        shuffle=False,
        pin_memory=USE_AMP,
    )

    # --------------------------------------------------------
    # Load pretrained BERT and initialize classification head
    # --------------------------------------------------------

    model = build_bert()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=0.01,
    )

    total_steps = len(train_loader) * EPOCHS

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.06),
        num_training_steps=total_steps,
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_AMP
    )

    # --------------------------------------------------------
    # Fine-tune BERT
    # --------------------------------------------------------

    for epoch in range(EPOCHS):

        model.train()
        total_loss = 0.0

        for batch in train_loader:

            batch_input_ids = batch[0].to(DEVICE)
            batch_attention = batch[1].to(DEVICE)
            batch_labels = batch[2].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(
                enabled=USE_AMP
            ):

                outputs = model(
                    input_ids=batch_input_ids,
                    attention_mask=batch_attention,
                    labels=batch_labels,
                )

                loss = outputs.loss

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()

        mean_loss = total_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{EPOCHS} "
            f"| Training loss: {mean_loss:.4f}"
        )

    # --------------------------------------------------------
    # Predict the validation fold
    # --------------------------------------------------------

    model.eval()
    fold_probabilities = []

    with torch.no_grad():

        for batch in valid_loader:

            batch_input_ids = batch[0].to(DEVICE)
            batch_attention = batch[1].to(DEVICE)

            with torch.cuda.amp.autocast(
                enabled=USE_AMP
            ):

                outputs = model(
                    input_ids=batch_input_ids,
                    attention_mask=batch_attention,
                )

            probabilities = torch.softmax(
                outputs.logits.float(),
                dim=-1,
            )

            fold_probabilities.append(
                probabilities.cpu().numpy()
            )

    fold_probabilities = np.concatenate(
        fold_probabilities,
        axis=0,
    )

    bert_oof_probs[valid_idx] = fold_probabilities

    fold_predictions = labels[
        np.argmax(fold_probabilities, axis=1)
    ]

    score = balanced_accuracy_score(
        y.iloc[valid_idx],
        fold_predictions,
    )

    fold_scores.append(score)

    print(f"Fold balanced accuracy: {score:.4f}")

    del model, optimizer, scheduler, scaler
    del train_loader, valid_loader
    del train_dataset, valid_dataset

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 7. REPORT BERT PERFORMANCE
# ============================================================

print("\n========== BERT CROSS-VALIDATION RESULTS ==========")

for i, score in enumerate(fold_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print(
    f"\nMean fold balanced accuracy: "
    f"{np.mean(fold_scores):.4f} "
    f"+/- {np.std(fold_scores):.4f}"
)

bert_oof_predictions = labels[
    np.argmax(bert_oof_probs, axis=1)
]

bert_pooled_score = balanced_accuracy_score(
    y,
    bert_oof_predictions,
)

print(
    f"Pooled out-of-fold balanced accuracy: "
    f"{bert_pooled_score:.4f}"
)

print("Previous best ensemble: 0.3589")


# ============================================================
# 8. OPTIONAL: COMPARE BERT WITH EXISTING ENSEMBLE
# ============================================================

def geometric_blend(probabilities, weights):

    log_probs = np.zeros_like(
        probabilities[0],
        dtype=np.float64,
    )

    for prob, weight in zip(probabilities, weights):
        log_probs += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    log_probs -= log_probs.max(axis=1, keepdims=True)

    combined = np.exp(log_probs)

    return combined / combined.sum(axis=1, keepdims=True)


def rank_scores(probabilities):

    ranks = rankdata(
        probabilities,
        axis=1,
        method="average",
    )

    return (ranks - 1) / (probabilities.shape[1] - 1)


if all(
    name in globals()
    for name in ["word_oof", "char_oof", "style_oof"]
):

    old_ensemble = geometric_blend(
        [word_oof, char_oof, style_oof],
        [0.40, 0.50, 0.10],
    )

    old_score = balanced_accuracy_score(
        y,
        labels[np.argmax(old_ensemble, axis=1)],
    )

    blend_results = [
        {
            "Method": "Existing ensemble",
            "BERT weight": 0.0,
            "Balanced accuracy": old_score,
        },
        {
            "Method": "BERT alone",
            "BERT weight": 1.0,
            "Balanced accuracy": bert_pooled_score,
        },
    ]

    old_ranks = rank_scores(old_ensemble)
    bert_ranks = rank_scores(bert_oof_probs)

    for bert_weight in [0.10, 0.20, 0.30, 0.40]:

        combined_ranks = (
            (1 - bert_weight) * old_ranks
            + bert_weight * bert_ranks
        )

        predictions = labels[
            np.argmax(combined_ranks, axis=1)
        ]

        score = balanced_accuracy_score(y, predictions)

        blend_results.append({
            "Method": "Rank blend",
            "BERT weight": bert_weight,
            "Balanced accuracy": score,
        })

    blend_results_df = pd.DataFrame(
        blend_results
    ).sort_values(
        "Balanced accuracy",
        ascending=False,
    ).reset_index(drop=True)

    print("\n========== ENSEMBLE COMPARISON ==========")
    display(blend_results_df)

else:

    print(
        "\nExisting ensemble OOF arrays were not found. "
        "BERT was evaluated independently."
    )

Device: cuda
Training examples: 2699
Number of classes: 18
Model source: google-bert/bert-base-cased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykern

Epoch 1/3 | Training loss: 2.7627
Epoch 2/3 | Training loss: 2.4719
Epoch 3/3 | Training loss: 2.2872


/tmp/ipykernel_49/2215965741.py:288: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Fold balanced accuracy: 0.2169

========== BERT CROSS-VALIDATION RESULTS ==========
Fold 1: 0.2537
Fold 2: 0.1426
Fold 3: 0.2333
Fold 4: 0.1685
Fold 5: 0.2169

Mean fold balanced accuracy: 0.2030 +/- 0.0413
Pooled out-of-fold balanced accuracy: 0.2030
Previous best ensemble: 0.3589

Existing ensemble OOF arrays were not found. BERT was evaluated independently.


In [4]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin


# --------------------------------------------------
# 1. Prepare data and fixed validation folds
# --------------------------------------------------

X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))


# --------------------------------------------------
# 2. Extract stylistic features
# --------------------------------------------------

class StyleFeatures(BaseEstimator, TransformerMixin):

    punctuation = [
        ",", ";", ":", ".", "!", "?", "-", "—", "–",
        "'", "’", '"', "“", "”", "(", ")", "[", "]",
        "{", "}", "/", "\\", "*", "_", "#", "`", "…", "•"
    ]

    phrases = [
        "certainly", "of course", "for example",
        "for instance", "in summary", "in conclusion",
        "however", "therefore", "moreover",
        "furthermore", "it is important", "overall",
        "hope this helps", "let's", "here is",
        "as a language model", "note that",
    ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.asarray(
            [self.extract(text) for text in X],
            dtype=np.float32,
        )

    def extract(self, text):
        text = str(text)
        lower = text.lower()
        length = max(len(text), 1)

        words = re.findall(r"\b\w+\b", lower)
        word_count = max(len(words), 1)

        word_lengths = [len(w) for w in words]

        sentences = [
            s.strip() for s in re.split(r"[.!?]+", text)
            if s.strip()
        ]

        sentence_lengths = [
            len(re.findall(r"\b\w+\b", s))
            for s in sentences
        ]

        lines = text.splitlines() or [text]
        paragraphs = [
            p for p in re.split(r"\n\s*\n", text) if p.strip()
        ]

        features = [
            np.log1p(len(text)),
            np.log1p(len(words)),
            np.log1p(len(sentences)),
            np.log1p(len(lines)),
            np.log1p(len(paragraphs)),
            np.mean(word_lengths) if word_lengths else 0,
            np.std(word_lengths) if word_lengths else 0,
            np.max(word_lengths) if word_lengths else 0,
            len(set(words)) / word_count,
            sum(len(w) <= 3 for w in words) / word_count,
            sum(len(w) >= 8 for w in words) / word_count,
            np.mean(sentence_lengths) if sentence_lengths else 0,
            np.std(sentence_lengths) if sentence_lengths else 0,
            np.max(sentence_lengths) if sentence_lengths else 0,
            sum(c.isupper() for c in text) / length,
            sum(c.islower() for c in text) / length,
            sum(c.isdigit() for c in text) / length,
            sum(c.isspace() for c in text) / length,
            sum(ord(c) > 127 for c in text) / length,
        ]

        # Punctuation frequencies
        features.extend(
            text.count(mark) * 1000 / length
            for mark in self.punctuation
        )

        # Formatting indicators
        bullet_count = sum(
            bool(re.match(r"^\s*(?:[-*•]\s+|\d+[.)]\s+)", line))
            for line in lines
        )

        features.extend([
            bullet_count / max(len(lines), 1),
            float("\n" in text),
            float("```" in text),
            float("##" in text),
            float(text.rstrip().endswith(("!", "?", "."))),
        ])

        features.extend(
            float(phrase in lower)
            for phrase in self.phrases
        )

        return features


# --------------------------------------------------
# 3. Build multi-view SVM
# --------------------------------------------------

def build_model(C, style_weight):

    return Pipeline([
        ("features", FeatureUnion(
            transformer_list=[
                ("word", TfidfVectorizer(
                    ngram_range=(1, 2),
                    max_features=25000,
                    sublinear_tf=True,
                )),

                ("char", TfidfVectorizer(
                    analyzer="char",
                    ngram_range=(1, 5),
                    max_features=100000,
                    sublinear_tf=True,
                )),

                ("style", Pipeline([
                    ("extract", StyleFeatures()),
                    ("scale", StandardScaler()),
                ])),
            ],
            transformer_weights={
                "word": 1.0,
                "char": 1.0,
                "style": style_weight,
            },
        )),
        ("classifier", LinearSVC(C=C)),
    ])


# --------------------------------------------------
# 4. Compare nine configurations
# --------------------------------------------------

results = []

for C in [0.5, 1.5, 3.0]:
    for style_weight in [0.1, 0.2, 0.3]:

        print(f"\nTesting C={C}, style_weight={style_weight}")

        model = build_model(C, style_weight)

        scores = cross_val_score(
            model,
            X,
            y,
            groups=groups,
            cv=cv_splits,
            scoring="balanced_accuracy",
            n_jobs=-1,
        )

        results.append({
            "C": C,
            "Style weight": style_weight,
            "Mean balanced accuracy": scores.mean(),
            "Std deviation": scores.std(),
            "Fold scores": scores.round(4).tolist(),
        })

        print(f"Fold scores: {scores.round(4)}")
        print(f"Mean: {scores.mean():.4f} +/- {scores.std():.4f}")


# --------------------------------------------------
# 5. Display the best configuration
# --------------------------------------------------

results_df = pd.DataFrame(results).sort_values(
    "Mean balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

print("\nBest configuration:")
display(results_df.head(1))

print("\nCurrent ensemble benchmark: 0.3589")


Testing C=0.5, style_weight=0.1
Fold scores: [0.363  0.3426 0.3852 0.3704 0.3465]
Mean: 0.3615 +/- 0.0157

Testing C=0.5, style_weight=0.2
Fold scores: [0.3556 0.3444 0.3815 0.3593 0.3446]
Mean: 0.3571 +/- 0.0135

Testing C=0.5, style_weight=0.3
Fold scores: [0.3611 0.3407 0.3741 0.35   0.3372]
Mean: 0.3526 +/- 0.0135

Testing C=1.5, style_weight=0.1
Fold scores: [0.3778 0.3556 0.3815 0.3685 0.3466]
Mean: 0.3660 +/- 0.0132

Testing C=1.5, style_weight=0.2
Fold scores: [0.3759 0.3426 0.3722 0.3611 0.3372]
Mean: 0.3578 +/- 0.0155

Testing C=1.5, style_weight=0.3


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Fold scores: [0.3722 0.3444 0.3741 0.3648 0.3372]
Mean: 0.3586 +/- 0.0150

Testing C=3.0, style_weight=0.1
Fold scores: [0.3759 0.3519 0.3778 0.3667 0.3447]
Mean: 0.3634 +/- 0.0131

Testing C=3.0, style_weight=0.2
Fold scores: [0.3759 0.3463 0.3741 0.3593 0.341 ]
Mean: 0.3593 +/- 0.0141

Testing C=3.0, style_weight=0.3


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Fold scores: [0.3759 0.3426 0.3741 0.3667 0.3317]
Mean: 0.3582 +/- 0.0178


,C,Style weight,Mean balanced accuracy,Std deviation,Fold scores
0,1.5,0.1,0.365977,0.013206,"[0.3778, 0.3556, 0.3815, 0.3685, 0.3466]"
1,3.0,0.1,0.363384,0.013095,"[0.3759, 0.3519, 0.3778, 0.3667, 0.3447]"
2,0.5,0.1,0.361520,0.015650,"[0.363, 0.3426, 0.3852, 0.3704, 0.3465]"
3,3.0,0.2,0.359310,0.014134,"[0.3759, 0.3463, 0.3741, 0.3593, 0.341]"
4,1.5,0.3,0.358557,0.014972,"[0.3722, 0.3444, 0.3741, 0.3648, 0.3372]"
5,3.0,0.3,0.358186,0.017805,"[0.3759, 0.3426, 0.3741, 0.3667, 0.3317]"
6,1.5,0.2,0.357816,0.015505,"[0.3759, 0.3426, 0.3722, 0.3611, 0.3372]"
7,0.5,0.2,0.357075,0.013543,"[0.3556, 0.3444, 0.3815, 0.3593, 0.3446]"
8,0.5,0.3,0.352631,0.013544,"[0.3611, 0.3407, 0.3741, 0.35, 0.3372]"



Best configuration:


,C,Style weight,Mean balanced accuracy,Std deviation,Fold scores
0,1.5,0.1,0.365977,0.013206,"[0.3778, 0.3556, 0.3815, 0.3685, 0.3466]"



Current ensemble benchmark: 0.3589


In [3]:
import re
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin


class StyleFeatures(BaseEstimator, TransformerMixin):

    punctuation = [
        ",", ";", ":", ".", "!", "?", "-", "—", "–",
        "'", "’", '"', "“", "”", "(", ")", "[", "]",
        "{", "}", "/", "\\", "*", "_", "#", "`", "…", "•"
    ]

    phrases = [
        "certainly", "of course", "for example",
        "for instance", "in summary", "in conclusion",
        "however", "therefore", "moreover",
        "furthermore", "it is important", "overall",
        "hope this helps", "let's", "here is",
        "as a language model", "note that",
    ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.asarray(
            [self.extract(text) for text in X],
            dtype=np.float32,
        )

    def extract(self, text):
        text = str(text)
        lower = text.lower()
        length = max(len(text), 1)

        words = re.findall(r"\b\w+\b", lower)
        word_count = max(len(words), 1)
        word_lengths = [len(w) for w in words]

        sentences = [
            s.strip()
            for s in re.split(r"[.!?]+", text)
            if s.strip()
        ]

        sentence_lengths = [
            len(re.findall(r"\b\w+\b", s))
            for s in sentences
        ]

        lines = text.splitlines() or [text]
        paragraphs = [
            p for p in re.split(r"\n\s*\n", text)
            if p.strip()
        ]

        features = [
            np.log1p(len(text)),
            np.log1p(len(words)),
            np.log1p(len(sentences)),
            np.log1p(len(lines)),
            np.log1p(len(paragraphs)),
            np.mean(word_lengths) if word_lengths else 0,
            np.std(word_lengths) if word_lengths else 0,
            np.max(word_lengths) if word_lengths else 0,
            len(set(words)) / word_count,
            sum(len(w) <= 3 for w in words) / word_count,
            sum(len(w) >= 8 for w in words) / word_count,
            np.mean(sentence_lengths) if sentence_lengths else 0,
            np.std(sentence_lengths) if sentence_lengths else 0,
            np.max(sentence_lengths) if sentence_lengths else 0,
            sum(c.isupper() for c in text) / length,
            sum(c.islower() for c in text) / length,
            sum(c.isdigit() for c in text) / length,
            sum(c.isspace() for c in text) / length,
            sum(ord(c) > 127 for c in text) / length,
        ]

        # Punctuation frequencies
        features.extend(
            text.count(mark) * 1000 / length
            for mark in self.punctuation
        )

        # Formatting indicators
        bullet_count = sum(
            bool(re.match(
                r"^\s*(?:[-*•]\s+|\d+[.)]\s+)", line
            ))
            for line in lines
        )

        features.extend([
            bullet_count / max(len(lines), 1),
            float("\n" in text),
            float("```" in text),
            float("##" in text),
            float(text.rstrip().endswith(("!", "?", "."))),
        ])

        # Common phrase indicators
        features.extend(
            float(phrase in lower)
            for phrase in self.phrases
        )

        return features


print("StyleFeatures is now defined. Rerun the experiment cell.")

StyleFeatures is now defined. Rerun the experiment cell.


In [5]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold, cross_val_score

# Prepare data
X = train_df["text"].fillna("").astype(str)
y = train_df["model"]
groups = train_df["prompt_id"]

# Same grouped 5-fold validation
cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))

# Character n-gram configurations to compare
char_ranges = [(1, 5), (2, 5), (3, 5)]

results = []

for char_range in char_ranges:
    print(f"\nTesting character n-grams: {char_range}")

    model = Pipeline([
        ("features", FeatureUnion(
            transformer_list=[
                ("word", TfidfVectorizer(
                    ngram_range=(1, 2),
                    max_features=25000,
                    sublinear_tf=True,
                )),
                ("char", TfidfVectorizer(
                    analyzer="char",
                    ngram_range=char_range,
                    max_features=100000,
                    sublinear_tf=True,
                )),
                ("style", Pipeline([
                    ("extract", StyleFeatures()),
                    ("scale", StandardScaler()),
                ])),
            ],
            transformer_weights={
                "word": 1.0,
                "char": 1.0,
                "style": 0.1,
            },
        )),
        ("classifier", LinearSVC(
            C=1.5,
            max_iter=10000,
        )),
    ])

    scores = cross_val_score(
        model,
        X,
        y,
        groups=groups,
        cv=cv_splits,
        scoring="balanced_accuracy",
        n_jobs=-1,
    )

    results.append({
        "Character n-grams": str(char_range),
        "Mean balanced accuracy": scores.mean(),
        "Std deviation": scores.std(),
        "Fold scores": scores.round(4).tolist(),
    })

    print(f"Fold scores: {scores.round(4)}")
    print(f"Mean: {scores.mean():.4f} +/- {scores.std():.4f}")

results_df = pd.DataFrame(results).sort_values(
    "Mean balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

print("\nCurrent benchmark: 0.3660")


Testing character n-grams: (1, 5)
Fold scores: [0.3648 0.3519 0.3815 0.3722 0.3447]
Mean: 0.3630 +/- 0.0133

Testing character n-grams: (2, 5)
Fold scores: [0.3685 0.35   0.3815 0.3722 0.3466]
Mean: 0.3638 +/- 0.0134

Testing character n-grams: (3, 5)
Fold scores: [0.3648 0.3463 0.3759 0.3722 0.3466]
Mean: 0.3612 +/- 0.0126


,Character n-grams,Mean balanced accuracy,Std deviation,Fold scores
0,"(2, 5)",0.363755,0.013370,"[0.3685, 0.35, 0.3815, 0.3722, 0.3466]"
1,"(1, 5)",0.363014,0.013335,"[0.3648, 0.3519, 0.3815, 0.3722, 0.3447]"
2,"(3, 5)",0.361162,0.012555,"[0.3648, 0.3463, 0.3759, 0.3722, 0.3466]"



Current benchmark: 0.3660


In [6]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

# Prepare data
X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)

labels = np.sort(y.unique())

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))

# Rebuild the best-performing configuration
model = Pipeline([
    ("features", FeatureUnion(
        transformer_list=[
            ("word", TfidfVectorizer(
                ngram_range=(1, 2),
                max_features=25000,
                sublinear_tf=True,
            )),
            ("char", TfidfVectorizer(
                analyzer="char",
                ngram_range=(1, 5),
                max_features=100000,
                sublinear_tf=True,
            )),
            ("style", Pipeline([
                ("extract", StyleFeatures()),
                ("scale", StandardScaler()),
            ])),
        ],
        transformer_weights={
            "word": 1.0,
            "char": 1.0,
            "style": 0.1,
        },
    )),
    ("classifier", LinearSVC(
        C=1.5,
        max_iter=10000,
    )),
])

# Obtain out-of-fold predictions
print("Generating out-of-fold predictions...")

oof_predictions = cross_val_predict(
    model,
    X,
    y,
    cv=cv_splits,
    method="predict",
    n_jobs=-1,
)

print(
    "Pooled balanced accuracy:",
    round(balanced_accuracy_score(y, oof_predictions), 4),
)

# Build confusion matrix: rows are actual classes,
# columns are predicted classes.
cm = confusion_matrix(
    y,
    oof_predictions,
    labels=labels,
)

# Calculate recall per LLM
class_counts = cm.sum(axis=1)
class_correct = np.diag(cm)

class_recall = np.divide(
    class_correct,
    class_counts,
    out=np.zeros(len(labels), dtype=float),
    where=class_counts != 0,
)

recall_df = pd.DataFrame({
    "LLM": labels,
    "Correct": class_correct,
    "Total": class_counts,
    "Recall": class_recall,
}).sort_values("Recall").reset_index(drop=True)

print("\n========== HARDEST LLMs TO IDENTIFY ==========")
display(recall_df.head(10))

# Find the most frequent directional confusions
confusions = []

for i, actual_model in enumerate(labels):
    for j, predicted_model in enumerate(labels):
        if i != j and cm[i, j] > 0:
            confusions.append({
                "Actual LLM": actual_model,
                "Predicted LLM": predicted_model,
                "Misclassified passages": int(cm[i, j]),
            })

confusion_df = pd.DataFrame(confusions).sort_values(
    "Misclassified passages",
    ascending=False,
).reset_index(drop=True)

print("\n========== MOST COMMON CONFUSIONS ==========")
display(confusion_df.head(20))


Generating out-of-fold predictions...
Pooled balanced accuracy: 0.363

========== HARDEST LLMs TO IDENTIFY ==========


,LLM,Correct,Total,Recall
0,claude-fable-5,16,149,0.107383
1,gpt-5.6-sol,22,150,0.146667
2,glm-5.3,23,150,0.153333
3,glm-5.3-flash,28,150,0.186667
4,claude-opus-4-8,37,150,0.246667
5,gpt-5.6-terra,40,150,0.266667
6,kimi-k3,43,150,0.286667
7,gpt-5.6-luna,47,150,0.313333
8,gemini-3.8-flash,50,150,0.333333
9,gpt-5.5,53,150,0.353333



========== MOST COMMON CONFUSIONS ==========


,Actual LLM,Predicted LLM,Misclassified passages
0,gemini-3.8-flash,gemini-3.7-flash,49
1,gemini-3.7-flash,gemini-3.8-flash,48
2,glm-5.3,glm-5.3-flash,35
3,glm-5.3,claude-opus-5,31
4,glm-5.3-flash,glm-5.3,30
5,gpt-5.6-sol,gpt-5.6-luna,29
6,gemini-3.6-flash,gemini-3.7-flash,28
7,gemini-3.8-flash,gemini-3.6-flash,24
8,gpt-5.6-luna,gpt-6-astra,24
9,gpt-5.6-terra,gpt-5.5,23


In [7]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import GroupKFold, cross_val_score

X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)

# Same grouped validation folds
cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))


def build_pipeline(classifier):
    return Pipeline([
        ("features", FeatureUnion(
            transformer_list=[
                ("word", TfidfVectorizer(
                    ngram_range=(1, 2),
                    max_features=25000,
                    sublinear_tf=True,
                )),
                ("char", TfidfVectorizer(
                    analyzer="char",
                    ngram_range=(1, 5),
                    max_features=100000,
                    sublinear_tf=True,
                )),
                ("style", Pipeline([
                    ("extract", StyleFeatures()),
                    ("scale", StandardScaler()),
                ])),
            ],
            transformer_weights={
                "word": 1.0,
                "char": 1.0,
                "style": 0.1,
            },
        )),
        ("classifier", classifier),
    ])


experiments = {
    "LinearSVC (one-vs-rest)": LinearSVC(
        C=1.5,
        max_iter=10000,
    ),
    "SVC (one-vs-one)": SVC(
        C=1.5,
        kernel="linear",
        decision_function_shape="ovo",
    ),
}

results = []

for name, classifier in experiments.items():
    print(f"\nEvaluating {name}...")

    model = build_pipeline(classifier)

    scores = cross_val_score(
        model,
        X,
        y,
        groups=groups,
        cv=cv_splits,
        scoring="balanced_accuracy",
        n_jobs=-1,
    )

    results.append({
        "Model": name,
        "Mean balanced accuracy": scores.mean(),
        "Std deviation": scores.std(),
        "Fold scores": scores.round(4).tolist(),
    })

    print(f"Fold scores: {scores.round(4)}")
    print(f"Mean: {scores.mean():.4f} +/- {scores.std():.4f}")

results_df = pd.DataFrame(results).sort_values(
    "Mean balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)


Evaluating LinearSVC (one-vs-rest)...
Fold scores: [0.3648 0.3519 0.3815 0.3722 0.3447]
Mean: 0.3630 +/- 0.0133

Evaluating SVC (one-vs-one)...
Fold scores: [0.3537 0.337  0.4    0.3426 0.3374]
Mean: 0.3541 +/- 0.0237


,Model,Mean balanced accuracy,Std deviation,Fold scores
0,LinearSVC (one-vs-rest),0.363014,0.013335,"[0.3648, 0.3519, 0.3815, 0.3722, 0.3447]"
1,SVC (one-vs-one),0.354138,0.023710,"[0.3537, 0.337, 0.4, 0.3426, 0.3374]"


In [8]:
import numpy as np
import pandas as pd

from scipy.stats import rankdata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score

# 1. Prepare data and fixed validation folds
X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)

labels = np.sort(y.unique())

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))


# 2. Build the three individual Logistic Regression models
word_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=25000,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(C=1.0, max_iter=1500)),
])

char_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(1, 5),
        max_features=100000,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(C=1.0, max_iter=1500)),
])

style_model = Pipeline([
    ("features", StyleFeatures()),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(C=1.0, max_iter=1500)),
])


# 3. Get out-of-fold probabilities for each model
print("Evaluating word model...")
word_oof = cross_val_predict(
    word_model, X, y,
    cv=cv_splits,
    method="predict_proba",
    n_jobs=1,
)

print("Evaluating character model...")
char_oof = cross_val_predict(
    char_model, X, y,
    cv=cv_splits,
    method="predict_proba",
    n_jobs=1,
)

print("Evaluating style model...")
style_oof = cross_val_predict(
    style_model, X, y,
    cv=cv_splits,
    method="predict_proba",
    n_jobs=1,
)


# 4. Combine Logistic Regression predictions
def geometric_blend(probabilities, weights):
    log_probs = np.zeros_like(
        probabilities[0], dtype=np.float64
    )

    for prob, weight in zip(probabilities, weights):
        log_probs += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    log_probs -= log_probs.max(axis=1, keepdims=True)
    combined = np.exp(log_probs)

    return combined / combined.sum(axis=1, keepdims=True)


baseline_oof = geometric_blend(
    [word_oof, char_oof, style_oof],
    [0.40, 0.50, 0.10],
)

baseline_score = balanced_accuracy_score(
    y, labels[np.argmax(baseline_oof, axis=1)]
)

print(f"\nLogistic ensemble: {baseline_score:.4f}")


# 5. Build the multi-view LinearSVC
svm_model = Pipeline([
    ("features", FeatureUnion(
        transformer_list=[
            ("word", TfidfVectorizer(
                ngram_range=(1, 2),
                max_features=25000,
                sublinear_tf=True,
            )),
            ("char", TfidfVectorizer(
                analyzer="char",
                ngram_range=(1, 5),
                max_features=100000,
                sublinear_tf=True,
            )),
            ("style", Pipeline([
                ("extract", StyleFeatures()),
                ("scale", StandardScaler()),
            ])),
        ],
        transformer_weights={
            "word": 1.0,
            "char": 1.0,
            "style": 0.1,
        },
    )),
    ("clf", LinearSVC(C=1.5, max_iter=10000)),
])


# 6. Generate out-of-fold SVM decision scores
print("\nEvaluating multi-view LinearSVC...")

svm_oof = cross_val_predict(
    svm_model, X, y,
    cv=cv_splits,
    method="decision_function",
    n_jobs=1,
)

svm_score = balanced_accuracy_score(
    y, labels[np.argmax(svm_oof, axis=1)]
)

print(f"LinearSVC: {svm_score:.4f}")


# 7. Normalize rankings before blending
def rank_scores(scores):
    ranks = rankdata(
        scores, axis=1, method="average"
    )
    return (ranks - 1) / (scores.shape[1] - 1)


baseline_ranks = rank_scores(baseline_oof)
svm_ranks = rank_scores(svm_oof)

results = []

for svm_weight in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:

    combined = (
        (1 - svm_weight) * baseline_ranks
        + svm_weight * svm_ranks
    )

    predictions = labels[np.argmax(combined, axis=1)]

    score = balanced_accuracy_score(y, predictions)

    results.append({
        "SVM weight": svm_weight,
        "Logistic ensemble weight": 1 - svm_weight,
        "Balanced accuracy": score,
    })

results_df = pd.DataFrame(results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

print(
    "\nBest blend score:",
    round(results_df.iloc[0]["Balanced accuracy"], 4),
)

Evaluating word model...
Evaluating character model...
Evaluating style model...

Logistic ensemble: 0.3589

Evaluating multi-view LinearSVC...
LinearSVC: 0.3630


,SVM weight,Logistic ensemble weight,Balanced accuracy
0,0.6,0.4,0.370047
1,0.8,0.2,0.366339
2,1.0,0.0,0.363003
3,0.4,0.6,0.360776
4,0.2,0.8,0.359289
5,0.0,1.0,0.358919



Best blend score: 0.37


In [9]:
import numpy as np
import pandas as pd

from scipy.stats import rankdata
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score

# Prepare data
X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)
labels = np.sort(y.unique())

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))


# Convert each model's scores into normalized within-row ranks
def rank_scores(scores):
    ranks = rankdata(scores, axis=1, method="average")
    return (ranks - 1) / (scores.shape[1] - 1)


# Rebuild the existing Logistic Regression ensemble
def geometric_blend(probabilities, weights):
    log_probs = np.zeros_like(probabilities[0], dtype=np.float64)

    for prob, weight in zip(probabilities, weights):
        log_probs += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    log_probs -= log_probs.max(axis=1, keepdims=True)
    combined = np.exp(log_probs)

    return combined / combined.sum(axis=1, keepdims=True)


baseline_oof = geometric_blend(
    [word_oof, char_oof, style_oof],
    [0.40, 0.50, 0.10],
)

baseline_ranks = rank_scores(baseline_oof)

results = []

# Tune C while keeping all other settings fixed
for c in [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]:

    print(f"\nTesting C={c}")

    svm_model = Pipeline([
        ("features", FeatureUnion(
            transformer_list=[
                ("word", TfidfVectorizer(
                    ngram_range=(1, 2),
                    max_features=25000,
                    sublinear_tf=True,
                )),
                ("char", TfidfVectorizer(
                    analyzer="char",
                    ngram_range=(1, 5),
                    max_features=100000,
                    sublinear_tf=True,
                )),
                ("style", Pipeline([
                    ("extract", StyleFeatures()),
                    ("scale", StandardScaler()),
                ])),
            ],
            transformer_weights={
                "word": 1.0,
                "char": 1.0,
                "style": 0.1,
            },
        )),
        ("classifier", LinearSVC(
            C=c,
            max_iter=10000,
        )),
    ])

    # Generate predictions only from models that exclude
    # the corresponding validation fold.
    svm_oof = cross_val_predict(
        svm_model,
        X,
        y,
        cv=cv_splits,
        method="decision_function",
        n_jobs=-1,
    )

    svm_ranks = rank_scores(svm_oof)

    # Compare the SVM alone and the fixed 60/40 blend
    for svm_weight in [0.0, 0.6, 1.0]:

        combined = (
            svm_weight * svm_ranks
            + (1 - svm_weight) * baseline_ranks
        )

        predictions = labels[np.argmax(combined, axis=1)]

        fold_scores = [
            balanced_accuracy_score(
                y.iloc[valid_idx],
                predictions[valid_idx],
            )
            for _, valid_idx in cv_splits
        ]

        results.append({
            "C": c,
            "SVM weight": svm_weight,
            "Logistic weight": 1 - svm_weight,
            "Mean fold balanced accuracy": np.mean(fold_scores),
            "Pooled balanced accuracy": balanced_accuracy_score(
                y, predictions
            ),
            "Fold scores": np.round(fold_scores, 4).tolist(),
        })

        print(
            f"SVM weight={svm_weight:.1f} | "
            f"Mean fold score={np.mean(fold_scores):.4f}"
        )


# Compare configurations
results_df = pd.DataFrame(results).sort_values(
    "Mean fold balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

print("\nCurrent benchmark: 0.3700")


Testing C=0.5
SVM weight=0.0 | Mean fold score=0.3589
SVM weight=0.6 | Mean fold score=0.3663
SVM weight=1.0 | Mean fold score=0.3593

Testing C=1.0
SVM weight=0.0 | Mean fold score=0.3589
SVM weight=0.6 | Mean fold score=0.3704
SVM weight=1.0 | Mean fold score=0.3619

Testing C=1.5
SVM weight=0.0 | Mean fold score=0.3589
SVM weight=0.6 | Mean fold score=0.3701
SVM weight=1.0 | Mean fold score=0.3630

Testing C=2.0
SVM weight=0.0 | Mean fold score=0.3589
SVM weight=0.6 | Mean fold score=0.3697
SVM weight=1.0 | Mean fold score=0.3612

Testing C=3.0
SVM weight=0.0 | Mean fold score=0.3589
SVM weight=0.6 | Mean fold score=0.3719
SVM weight=1.0 | Mean fold score=0.3634

Testing C=5.0
SVM weight=0.0 | Mean fold score=0.3589
SVM weight=0.6 | Mean fold score=0.3697
SVM weight=1.0 | Mean fold score=0.3619


,C,SVM weight,Logistic weight,Mean fold balanced accuracy,Pooled balanced accuracy,Fold scores
0,3.0,0.6,0.4,0.371916,0.371899,"[0.3741, 0.3556, 0.3926, 0.3889, 0.3485]"
1,1.0,0.6,0.4,0.370434,0.370418,"[0.3741, 0.35, 0.3889, 0.3907, 0.3485]"
2,1.5,0.6,0.4,0.370064,0.370047,"[0.3741, 0.3519, 0.3889, 0.3907, 0.3448]"
3,2.0,0.6,0.4,0.369693,0.369677,"[0.3722, 0.3519, 0.3889, 0.3889, 0.3466]"
4,5.0,0.6,0.4,0.369693,0.369677,"[0.3741, 0.3556, 0.3907, 0.3815, 0.3466]"
5,0.5,0.6,0.4,0.366347,0.366334,"[0.3574, 0.3389, 0.3981, 0.3907, 0.3466]"
6,3.0,1.0,0.0,0.363384,0.363373,"[0.3667, 0.3519, 0.3778, 0.3722, 0.3484]"
7,1.5,1.0,0.0,0.363014,0.363003,"[0.3648, 0.3519, 0.3815, 0.3722, 0.3447]"
8,5.0,1.0,0.0,0.361903,0.361892,"[0.3685, 0.35, 0.3741, 0.3685, 0.3484]"
9,1.0,1.0,0.0,0.361903,0.361889,"[0.3648, 0.3481, 0.3778, 0.3722, 0.3466]"



Current benchmark: 0.3700


In [10]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold, cross_val_score

# Prepare data
X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)

# Preserve the same grouped validation folds
cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))

# Change only word and character feature weights
configs = [
    (0.5, 1.0),
    (0.75, 1.0),
    (1.0, 1.0),
    (1.25, 1.0),
    (1.5, 1.0),
    (1.0, 0.75),
    (1.0, 0.5),
]

results = []

for word_weight, char_weight in configs:

    print(
        f"\nWord weight={word_weight}, "
        f"Character weight={char_weight}"
    )

    model = Pipeline([
        ("features", FeatureUnion(
            transformer_list=[
                ("word", TfidfVectorizer(
                    ngram_range=(1, 2),
                    max_features=25000,
                    sublinear_tf=True,
                )),
                ("char", TfidfVectorizer(
                    analyzer="char",
                    ngram_range=(1, 5),
                    max_features=100000,
                    sublinear_tf=True,
                )),
                ("style", Pipeline([
                    ("extract", StyleFeatures()),
                    ("scale", StandardScaler()),
                ])),
            ],
            transformer_weights={
                "word": word_weight,
                "char": char_weight,
                "style": 0.1,
            },
        )),
        ("classifier", LinearSVC(
            C=3.0,
            max_iter=10000,
        )),
    ])

    scores = cross_val_score(
        model,
        X,
        y,
        groups=groups,
        cv=cv_splits,
        scoring="balanced_accuracy",
        n_jobs=-1,
    )

    results.append({
        "Word weight": word_weight,
        "Character weight": char_weight,
        "Mean balanced accuracy": scores.mean(),
        "Std deviation": scores.std(),
        "Fold scores": scores.round(4).tolist(),
    })

    print(f"Scores: {scores.round(4)}")
    print(f"Mean: {scores.mean():.4f} +/- {scores.std():.4f}")

results_df = pd.DataFrame(results).sort_values(
    "Mean balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

print("\nCurrent benchmark: 0.3719")


Word weight=0.5, Character weight=1.0
Scores: [0.3833 0.3444 0.3796 0.3704 0.3522]
Mean: 0.3660 +/- 0.0152

Word weight=0.75, Character weight=1.0
Scores: [0.3778 0.3556 0.3833 0.3722 0.3521]
Mean: 0.3682 +/- 0.0123

Word weight=1.0, Character weight=1.0
Scores: [0.3667 0.3519 0.3778 0.3722 0.3484]
Mean: 0.3634 +/- 0.0114

Word weight=1.25, Character weight=1.0
Scores: [0.3685 0.3407 0.35   0.3574 0.3428]
Mean: 0.3519 +/- 0.0102

Word weight=1.5, Character weight=1.0
Scores: [0.3537 0.3407 0.3537 0.363  0.3373]
Mean: 0.3497 +/- 0.0094

Word weight=1.0, Character weight=0.75
Scores: [0.3611 0.3463 0.3722 0.3611 0.3484]
Mean: 0.3578 +/- 0.0095

Word weight=1.0, Character weight=0.5
Scores: [0.3556 0.3519 0.3648 0.3556 0.341 ]
Mean: 0.3538 +/- 0.0077


,Word weight,Character weight,Mean balanced accuracy,Std deviation,Fold scores
0,0.75,1.00,0.368199,0.012295,"[0.3778, 0.3556, 0.3833, 0.3722, 0.3521]"
1,0.50,1.00,0.365990,0.015239,"[0.3833, 0.3444, 0.3796, 0.3704, 0.3522]"
2,1.00,1.00,0.363384,0.011432,"[0.3667, 0.3519, 0.3778, 0.3722, 0.3484]"
3,1.00,0.75,0.357829,0.009493,"[0.3611, 0.3463, 0.3722, 0.3611, 0.3484]"
4,1.00,0.50,0.353755,0.007684,"[0.3556, 0.3519, 0.3648, 0.3556, 0.341]"
5,1.25,1.00,0.351903,0.010164,"[0.3685, 0.3407, 0.35, 0.3574, 0.3428]"
6,1.50,1.00,0.349681,0.009404,"[0.3537, 0.3407, 0.3537, 0.363, 0.3373]"



Current benchmark: 0.3719


In [11]:
import numpy as np
import pandas as pd

from scipy.stats import rankdata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score


# 1. Prepare data
X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)

labels = np.sort(y.unique())

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))


# 2. Create the existing Logistic Regression ensemble
def geometric_blend(probabilities, weights):
    log_probs = np.zeros_like(
        probabilities[0], dtype=np.float64
    )

    for prob, weight in zip(probabilities, weights):
        log_probs += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    log_probs -= log_probs.max(axis=1, keepdims=True)
    combined = np.exp(log_probs)

    return combined / combined.sum(axis=1, keepdims=True)


baseline_oof = geometric_blend(
    [word_oof, char_oof, style_oof],
    [0.40, 0.50, 0.10],
)


# 3. Build the improved SVM
svm_model = Pipeline([
    ("features", FeatureUnion(
        transformer_list=[
            ("word", TfidfVectorizer(
                ngram_range=(1, 2),
                max_features=25000,
                sublinear_tf=True,
            )),
            ("char", TfidfVectorizer(
                analyzer="char",
                ngram_range=(2, 5),
                max_features=100000,
                sublinear_tf=True,
            )),
            ("style", Pipeline([
                ("extract", StyleFeatures()),
                ("scale", StandardScaler()),
            ])),
        ],
        transformer_weights={
            "word": 0.75,
            "char": 1.0,
            "style": 0.1,
        },
    )),
    ("classifier", LinearSVC(
        C=3.0,
        max_iter=10000,
    )),
])


# 4. Generate out-of-fold SVM scores
print("Generating improved SVM predictions...")

svm_oof = cross_val_predict(
    svm_model,
    X,
    y,
    cv=cv_splits,
    method="decision_function",
    n_jobs=-1,
)

# Convert each model's scores to comparable within-row ranks
def rank_scores(scores):
    ranks = rankdata(
        scores,
        axis=1,
        method="average",
    )
    return (ranks - 1) / (scores.shape[1] - 1)


baseline_ranks = rank_scores(baseline_oof)
svm_ranks = rank_scores(svm_oof)


# 5. Compare different blending weights
results = []

for svm_weight in [0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:

    combined = (
        svm_weight * svm_ranks
        + (1 - svm_weight) * baseline_ranks
    )

    predictions = labels[np.argmax(combined, axis=1)]

    fold_scores = [
        balanced_accuracy_score(
            y.iloc[valid_idx],
            predictions[valid_idx],
        )
        for _, valid_idx in cv_splits
    ]

    results.append({
        "SVM weight": svm_weight,
        "Logistic ensemble weight": 1 - svm_weight,
        "Mean fold balanced accuracy": np.mean(fold_scores),
        "Std deviation": np.std(fold_scores),
        "Fold scores": np.round(fold_scores, 4).tolist(),
    })

results_df = pd.DataFrame(results).sort_values(
    "Mean fold balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

print("\nPrevious best blended score: 0.3719")

Generating improved SVM predictions...


,SVM weight,Logistic ensemble weight,Mean fold balanced accuracy,Std deviation,Fold scores
0,0.7,0.3,0.374496,0.015348,"[0.3907, 0.3611, 0.3889, 0.3796, 0.3521]"
1,0.6,0.4,0.373027,0.018449,"[0.3815, 0.3556, 0.3907, 0.3907, 0.3466]"
2,0.5,0.5,0.372656,0.032102,"[0.3704, 0.3481, 0.4111, 0.4056, 0.3281]"
3,0.8,0.2,0.371903,0.014618,"[0.3852, 0.3593, 0.387, 0.3778, 0.3503]"
4,1.0,0.0,0.367088,0.014048,"[0.3796, 0.3556, 0.3833, 0.3704, 0.3466]"
5,0.4,0.6,0.365236,0.026417,"[0.3556, 0.3352, 0.4019, 0.3907, 0.3428]"



Previous best blended score: 0.3719


In [12]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
)


# 1. Combine the latest out-of-fold predictions
# using the best weights found so far.

baseline_ranks = rank_scores(baseline_oof)
svm_ranks = rank_scores(svm_oof)

blended_scores = (
    0.7 * svm_ranks
    + 0.3 * baseline_ranks
)

predictions = labels[
    np.argmax(blended_scores, axis=1)
]


# 2. Check overall and fold-wise balanced accuracy

print(
    "Pooled balanced accuracy:",
    round(balanced_accuracy_score(y, predictions), 4),
)

fold_results = []

for fold, (_, valid_idx) in enumerate(cv_splits, start=1):
    score = balanced_accuracy_score(
        y.iloc[valid_idx],
        predictions[valid_idx],
    )

    fold_results.append({
        "Fold": fold,
        "Balanced accuracy": score,
    })

display(pd.DataFrame(fold_results))


# 3. Build the confusion matrix

cm = confusion_matrix(y, predictions, labels=labels)

class_totals = cm.sum(axis=1)
correct = np.diag(cm)

recall = np.divide(
    correct,
    class_totals,
    out=np.zeros(len(labels), dtype=float),
    where=class_totals != 0,
)

recall_df = pd.DataFrame({
    "LLM": labels,
    "Correct": correct,
    "Total": class_totals,
    "Recall": recall,
}).sort_values(
    "Recall",
    ascending=True,
).reset_index(drop=True)

print("\nHardest LLMs to identify:")
display(recall_df.head(10))


# 4. Find the most common model confusions

confusions = []

for i, actual in enumerate(labels):
    for j, predicted in enumerate(labels):
        if i != j and cm[i, j] > 0:
            confusions.append({
                "Actual LLM": actual,
                "Predicted LLM": predicted,
                "Errors": int(cm[i, j]),
            })

confusion_df = pd.DataFrame(confusions).sort_values(
    "Errors",
    ascending=False,
).reset_index(drop=True)

print("\nMost common confusions:")
display(confusion_df.head(20))


Pooled balanced accuracy: 0.3745


,Fold,Balanced accuracy
0,1,0.390741
1,2,0.361111
2,3,0.388889
3,4,0.379630
4,5,0.352107



Hardest LLMs to identify:


,LLM,Correct,Total,Recall
0,claude-fable-5,17,149,0.114094
1,gpt-5.6-sol,24,150,0.160000
2,glm-5.3,25,150,0.166667
3,glm-5.3-flash,30,150,0.200000
4,claude-opus-4-8,38,150,0.253333
5,gpt-5.6-terra,40,150,0.266667
6,kimi-k3,44,150,0.293333
7,gpt-5.6-luna,51,150,0.340000
8,gpt-5.5,53,150,0.353333
9,gemini-3.8-flash,57,150,0.380000



Most common confusions:


,Actual LLM,Predicted LLM,Errors
0,gemini-3.8-flash,gemini-3.7-flash,49
1,gemini-3.7-flash,gemini-3.8-flash,44
2,glm-5.3,glm-5.3-flash,37
3,glm-5.3-flash,glm-5.3,32
4,glm-5.3,claude-opus-5,30
5,gpt-5.6-sol,gpt-5.6-luna,30
6,gemini-3.6-flash,gemini-3.7-flash,30
7,gemini-3.6-flash,gemini-3.1-pro-preview,25
8,gpt-5.6-terra,gpt-5.5,24
9,gpt-5.6-terra,gpt-5.6-luna,23


In [2]:
import numpy as np
import pandas as pd
import torch
import kagglehub

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score
from scipy.stats import rankdata

# --------------------------------------------------
# 1. Prepare data
# --------------------------------------------------

X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)

labels = np.sort(y.unique())

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))

print("Training examples:", len(X))
print("Classes:", len(labels))


# --------------------------------------------------
# 2. Download and load Qwen3 Embedding
# --------------------------------------------------

model_dir = kagglehub.model_download(
    "qwen-lm/qwen-3-embedding/transformers/0.6b"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer(
    str(model_dir),
    device=device,
)

# Keep the initial experiment manageable.
embedder.max_seq_length = 512

print("Embedding model loaded.")
print("Device:", device)


# --------------------------------------------------
# 3. Convert passages into embeddings
# --------------------------------------------------

embeddings = embedder.encode(
    X.tolist(),
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

print("Embedding shape:", embeddings.shape)


# --------------------------------------------------
# 4. Evaluate Logistic Regression on embeddings
# --------------------------------------------------

results = []
best_score = -1
best_c = None
best_oof = None

for c in [0.03, 0.1, 0.3, 1.0, 3.0]:

    print(f"\nTesting Logistic Regression C={c}")

    classifier = LogisticRegression(
        C=c,
        max_iter=3000,
    )

    oof_probs = cross_val_predict(
        classifier,
        embeddings,
        y,
        cv=cv_splits,
        method="predict_proba",
        n_jobs=-1,
    )

    predictions = labels[np.argmax(oof_probs, axis=1)]
    score = balanced_accuracy_score(y, predictions)

    results.append({
        "C": c,
        "Balanced accuracy": score,
    })

    print(f"Balanced accuracy: {score:.4f}")

    if score > best_score:
        best_score = score
        best_c = c
        best_oof = oof_probs.copy()

results_df = pd.DataFrame(results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

print("\n========== QWEN EMBEDDING RESULTS ==========")
display(results_df)

print(f"Best C: {best_c}")
print(f"Best embedding score: {best_score:.4f}")


# --------------------------------------------------
# 5. Compare with the existing ensemble
# --------------------------------------------------

def rank_scores(scores):
    ranks = rankdata(
        scores,
        axis=1,
        method="average",
    )
    return (ranks - 1) / (scores.shape[1] - 1)


if (
    "svm_oof" in globals()
    and "baseline_oof" in globals()
    and np.asarray(svm_oof).shape == (len(X), len(labels))
    and np.asarray(baseline_oof).shape == (len(X), len(labels))
):

    # Current best blend: 70% SVM + 30% Logistic ensemble
    current_blend = (
        0.7 * rank_scores(svm_oof)
        + 0.3 * rank_scores(baseline_oof)
    )

    current_predictions = labels[
        np.argmax(current_blend, axis=1)
    ]

    current_score = balanced_accuracy_score(
        y, current_predictions
    )

    qwen_ranks = rank_scores(best_oof)

    blend_results = [{
        "Method": "Existing SVM + Logistic blend",
        "Qwen weight": 0.0,
        "Balanced accuracy": current_score,
    }, {
        "Method": "Qwen embeddings alone",
        "Qwen weight": 1.0,
        "Balanced accuracy": best_score,
    }]

    # Test different Qwen contributions
    for qwen_weight in [0.1, 0.2, 0.3, 0.4, 0.5]:

        combined = (
            (1 - qwen_weight) * current_blend
            + qwen_weight * qwen_ranks
        )

        predictions = labels[np.argmax(combined, axis=1)]

        score = balanced_accuracy_score(y, predictions)

        blend_results.append({
            "Method": "Rank blend",
            "Qwen weight": qwen_weight,
            "Balanced accuracy": score,
        })

    blend_df = pd.DataFrame(blend_results).sort_values(
        "Balanced accuracy",
        ascending=False,
    ).reset_index(drop=True)

    print("\n========== BLEND COMPARISON ==========")
    display(blend_df)

else:
    print(
        "\nPrevious SVM/Logistic out-of-fold arrays were not found. "
        "Qwen was evaluated independently; rerun the earlier "
        "OOF prediction cells to compare ensembles."
    )

Training examples: 2699
Classes: 18


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Embedding model loaded.
Device: cuda


Batches:   0%|          | 0/338 [00:00<?, ?it/s]

Embedding shape: (2699, 1024)

Testing Logistic Regression C=0.03
Balanced accuracy: 0.1270

Testing Logistic Regression C=0.1
Balanced accuracy: 0.1326

Testing Logistic Regression C=0.3
Balanced accuracy: 0.1452

Testing Logistic Regression C=1.0
Balanced accuracy: 0.1526

Testing Logistic Regression C=3.0
Balanced accuracy: 0.1630

========== QWEN EMBEDDING RESULTS ==========


,C,Balanced accuracy
0,3.00,0.162970
1,1.00,0.152598
2,0.30,0.145185
3,0.10,0.132593
4,0.03,0.127037


Best C: 3.0
Best embedding score: 0.1630

Previous SVM/Logistic out-of-fold arrays were not found. Qwen was evaluated independently; rerun the earlier OOF prediction cells to compare ensembles.


In [5]:
# ============================================================
# FINAL INFERENCE CELL
#   - Fit best configuration (70% SVM / 30% LR rank blend) on all train
#   - Predict test scores
#   - Cluster test passages to recover prompt groups
#   - Hungarian-constrain any cluster of size 17 or 18
#   - Save submission.parquet
# ============================================================

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from scipy.stats import rankdata

from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

assert "StyleFeatures" in globals(), (
    "Run the earlier cells that define StyleFeatures before this cell."
)

# --------------------------------------------------
# 1. Prepare data
# --------------------------------------------------

X_train = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y_train = train_df["model"].astype(str).reset_index(drop=True)

labels = np.sort(y_train.unique())
n_classes = len(labels)

X_test = test_df["text"].fillna("").astype(str).reset_index(drop=True)
test_ids = test_df["id"].reset_index(drop=True)

print(f"Train: {len(X_train)} | Test: {len(X_test)} | Classes: {n_classes}")


# --------------------------------------------------
# 2. Rebuild the best models
# --------------------------------------------------

def build_word_lr():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=25000,
            sublinear_tf=True,
        )),
        ("clf", LogisticRegression(C=1.0, max_iter=1500)),
    ])


def build_char_lr():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(1, 5),
            max_features=100000,
            sublinear_tf=True,
        )),
        ("clf", LogisticRegression(C=1.0, max_iter=1500)),
    ])


def build_style_lr():
    return Pipeline([
        ("features", StyleFeatures()),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(C=1.0, max_iter=1500)),
    ])


def build_svm():
    return Pipeline([
        ("features", FeatureUnion(
            transformer_list=[
                ("word", TfidfVectorizer(
                    ngram_range=(1, 2),
                    max_features=25000,
                    sublinear_tf=True,
                )),
                ("char", TfidfVectorizer(
                    analyzer="char",
                    ngram_range=(2, 5),
                    max_features=100000,
                    sublinear_tf=True,
                )),
                ("style", Pipeline([
                    ("extract", StyleFeatures()),
                    ("scale", StandardScaler()),
                ])),
            ],
            transformer_weights={
                "word": 0.75,
                "char": 1.0,
                "style": 0.1,
            },
        )),
        ("classifier", LinearSVC(C=3.0, max_iter=10000)),
    ])


# --------------------------------------------------
# 3. Fit on full training data
# --------------------------------------------------

print("\nFitting 4 models on full training data...")

word_model = build_word_lr().fit(X_train, y_train)
char_model = build_char_lr().fit(X_train, y_train)
style_model = build_style_lr().fit(X_train, y_train)
svm_model = build_svm().fit(X_train, y_train)

print("Done.")


# --------------------------------------------------
# 4. Predict on test
# --------------------------------------------------

print("\nPredicting test...")

word_p = word_model.predict_proba(X_test)
char_p = char_model.predict_proba(X_test)
style_p = style_model.predict_proba(X_test)


def geometric_blend(probs, weights):
    log_probs = np.zeros_like(probs[0], dtype=np.float64)
    for p, w in zip(probs, weights):
        log_probs += w * np.log(np.clip(p, 1e-12, 1.0))
    log_probs -= log_probs.max(axis=1, keepdims=True)
    combined = np.exp(log_probs)
    return combined / combined.sum(axis=1, keepdims=True)


lr_probs = geometric_blend(
    [word_p, char_p, style_p],
    [0.40, 0.50, 0.10],
)
svm_scores = svm_model.decision_function(X_test)


# --------------------------------------------------
# 5. Align column ordering, rank-normalize, blend
# --------------------------------------------------

def align_columns(matrix, classes):
    classes_list = list(classes)
    idx = [classes_list.index(lbl) for lbl in labels]
    return matrix[:, idx]


lr_probs = align_columns(lr_probs, word_model.classes_)
svm_scores = align_columns(svm_scores, svm_model.classes_)


def rank_scores(scores):
    ranks = rankdata(scores, axis=1, method="average")
    return (ranks - 1) / (scores.shape[1] - 1)


lr_ranks = rank_scores(lr_probs)
svm_ranks = rank_scores(svm_scores)

# 70% SVM / 30% LR (best CV configuration)
blended = 0.7 * svm_ranks + 0.3 * lr_ranks
print("Blended scores:", blended.shape)


# --------------------------------------------------
# 6. Cluster test passages to recover prompts
# --------------------------------------------------

def cluster_test_passages(texts, target_size=18):
    n = len(texts)
    if n < target_size * 2:
        return np.arange(n)

    n_clusters = max(1, round(n / target_size))

    vec = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=40000,
        min_df=1,
        stop_words="english",
        sublinear_tf=True,
    )
    feats = vec.fit_transform(texts)
    sim = (feats @ feats.T).toarray()
    dist = np.clip(1.0 - sim, 0.0, 2.0)
    np.fill_diagonal(dist, 0.0)

    clusterer = AgglomerativeClustering(
        n_clusters=n_clusters,
        metric="precomputed",
        linkage="average",
    )
    return clusterer.fit_predict(dist)


print("\nClustering test passages...")
cluster_ids = cluster_test_passages(X_test.tolist(), target_size=n_classes)

n_clusters_found = len(np.unique(cluster_ids))
size_counts = pd.Series(cluster_ids).value_counts().sort_index().to_dict()
print(f"Found {n_clusters_found} clusters from {len(X_test)} passages")
print(f"Cluster sizes: {size_counts}")


# --------------------------------------------------
# 7. Hungarian-constrain valid clusters
# --------------------------------------------------

base_predictions = labels[np.argmax(blended, axis=1)].copy()
final_predictions = base_predictions.copy()

n_constrained = 0

for cid in np.unique(cluster_ids):
    rows = np.where(cluster_ids == cid)[0]
    size = len(rows)

    # Only enforce the constraint for prompt-sized clusters
    if size not in (17, 18):
        continue

    block = blended[rows].copy()

    if size == 17:
        # Add a neutral dummy row so Hungarian returns a full 18x18 matching.
        block = np.vstack([block, np.zeros((1, n_classes))])

    row_ind, col_ind = linear_sum_assignment(-block)

    for r, c in zip(row_ind, col_ind):
        if r < size:
            final_predictions[rows[r]] = labels[c]

    n_constrained += 1

n_changed = int((final_predictions != base_predictions).sum())

print(f"\nConstrained {n_constrained} clusters via Hungarian assignment.")
print(f"Changed {n_changed}/{len(final_predictions)} predictions vs baseline.")



Train: 2699 | Test: 3 | Classes: 18

Fitting 4 models on full training data...
Done.

Predicting test...
Blended scores: (3, 18)

Clustering test passages...
Found 3 clusters from 3 passages
Cluster sizes: {0: 1, 1: 1, 2: 1}

Constrained 0 clusters via Hungarian assignment.
Changed 0/3 predictions vs baseline.


In [1]:
import pathlib
import re
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from IPython.display import display

# ------------------------------------------------------------
# 1. Load data (works in Kaggle; or reuse train_df if already set)
# ------------------------------------------------------------
if "train_df" not in globals():
    CANDIDATES = [
        pathlib.Path("/kaggle/input/competitions/name-that-llm"),
        pathlib.Path("/kaggle/input/name-that-llm"),
    ]
    DATA_DIR = next((p for p in CANDIDATES if (p / "train.csv").exists()), None)
    if DATA_DIR is None:
        raise FileNotFoundError(
            "Could not find train.csv. Attach the Name That LLM competition dataset."
        )
    train_df = pd.read_csv(DATA_DIR / "train.csv")

X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].astype(str).reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)
labels = np.sort(y.unique())
label_to_col = {label: i for i, label in enumerate(labels)}

# Exact family mapping used in the existing experiments.
MODEL_TO_FAMILY = {
    "gpt-5.5": "OpenAI",
    "gpt-5.6-luna": "OpenAI",
    "gpt-5.6-sol": "OpenAI",
    "gpt-5.6-terra": "OpenAI",
    "gpt-6-astra": "OpenAI",
    "gemini-3.1-pro-preview": "Google",
    "gemini-3.6-flash": "Google",
    "gemini-3.7-flash": "Google",
    "gemini-3.8-flash": "Google",
    "claude-fable-5": "Anthropic",
    "claude-opus-4-6": "Anthropic",
    "claude-opus-4-8": "Anthropic",
    "claude-opus-5": "Anthropic",
    "claude-sonnet-5": "Anthropic",
    "glm-5.3": "ZAI",
    "glm-5.3-flash": "ZAI",
    "qwen3.8-max": "Qwen",
    "kimi-k3": "Moonshot",
}

unknown_models = sorted(set(labels) - set(MODEL_TO_FAMILY))
if unknown_models:
    raise ValueError(f"Add family mappings for: {unknown_models}")

family_by_label = MODEL_TO_FAMILY
family_pairs = []
for family in sorted(set(family_by_label.values())):
    family_labels = sorted([m for m in labels if family_by_label[m] == family])
    family_pairs.extend(combinations(family_labels, 2))

print(f"Rows: {len(X):,} | prompts: {groups.nunique()} | models: {len(labels)}")
print(f"Same-family candidate pairs available: {len(family_pairs)}")

# ------------------------------------------------------------
# 2. Style features — same feature design as the existing notebook
# ------------------------------------------------------------
class StyleFeatures(BaseEstimator, TransformerMixin):
    punctuation = [
        ",", ";", ":", ".", "!", "?", "-", "—", "–",
        "'", "’", '"', "“", "”", "(", ")", "[", "]",
        "{", "}", "/", "\\", "*", "_", "#", "`", "…", "•"
    ]

    phrases = [
        "certainly", "of course", "for example",
        "for instance", "in summary", "in conclusion",
        "however", "therefore", "moreover",
        "furthermore", "it is important", "overall",
        "hope this helps", "let's", "here is",
        "as a language model", "note that",
    ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.asarray([self.extract(text) for text in X], dtype=np.float32)

    def extract(self, text):
        text = str(text)
        lower = text.lower()
        length = max(len(text), 1)
        words = re.findall(r"\b\w+\b", lower)
        word_count = max(len(words), 1)
        word_lengths = [len(w) for w in words]
        sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
        sentence_lengths = [len(re.findall(r"\b\w+\b", s)) for s in sentences]
        lines = text.splitlines() or [text]
        paragraphs = [p for p in re.split(r"\n\s*\n", text) if p.strip()]

        features = [
            np.log1p(len(text)), np.log1p(len(words)), np.log1p(len(sentences)),
            np.log1p(len(lines)), np.log1p(len(paragraphs)),
            np.mean(word_lengths) if word_lengths else 0,
            np.std(word_lengths) if word_lengths else 0,
            np.max(word_lengths) if word_lengths else 0,
            len(set(words)) / word_count,
            sum(len(w) <= 3 for w in words) / word_count,
            sum(len(w) >= 8 for w in words) / word_count,
            np.mean(sentence_lengths) if sentence_lengths else 0,
            np.std(sentence_lengths) if sentence_lengths else 0,
            np.max(sentence_lengths) if sentence_lengths else 0,
            sum(c.isupper() for c in text) / length,
            sum(c.islower() for c in text) / length,
            sum(c.isdigit() for c in text) / length,
            sum(c.isspace() for c in text) / length,
            sum(ord(c) > 127 for c in text) / length,
        ]
        features.extend(text.count(mark) * 1000 / length for mark in self.punctuation)

        bullet_count = sum(
            bool(re.match(r"^\s*(?:[-*•]\s+|\d+[.)]\s+)", line))
            for line in lines
        )
        features.extend([
            bullet_count / max(len(lines), 1),
            float("\n" in text),
            float("```" in text),
            float("##" in text),
            float(text.rstrip().endswith(("!", "?", "."))),
        ])
        features.extend(float(phrase in lower) for phrase in self.phrases)
        return features

# ------------------------------------------------------------
# 3. Helpers
# ------------------------------------------------------------
def align_probabilities(model, probabilities, all_labels):
    """Place estimator probabilities into a stable, global label order."""
    aligned = np.zeros((probabilities.shape[0], len(all_labels)), dtype=np.float64)
    global_cols = {label: i for i, label in enumerate(all_labels)}
    for local_col, label in enumerate(model.classes_):
        aligned[:, global_cols[label]] = probabilities[:, local_col]
    return aligned


def geometric_blend(probability_views, weights):
    log_probs = np.zeros_like(probability_views[0], dtype=np.float64)
    for probabilities, weight in zip(probability_views, weights):
        log_probs += weight * np.log(np.clip(probabilities, 1e-12, 1.0))
    log_probs -= log_probs.max(axis=1, keepdims=True)
    combined = np.exp(log_probs)
    return combined / combined.sum(axis=1, keepdims=True)


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def sigmoid(z):
    # Numerically stable enough for the clipped logit values above.
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

# Blend weight applied to the sibling specialist's within-pair odds.
# 0.0 = global baseline only; 1.0 = specialist only for that pair.
SPECIALIST_WEIGHT = 0.50
N_SPLITS = 5
cv_splits = list(GroupKFold(n_splits=N_SPLITS).split(X, y, groups))

baseline_oof = np.zeros((len(X), len(labels)), dtype=np.float64)
exp_b_oof = np.zeros((len(X), len(labels)), dtype=np.float64)
fold_results = []
pair_activations = Counter()
pair_prediction_changes = Counter()

# ------------------------------------------------------------
# 4. Outer grouped CV: baseline ensemble + pairwise specialists
# ------------------------------------------------------------
for fold, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
    print(f"\n========== FOLD {fold}/{N_SPLITS} ==========")
    X_train_text = X.iloc[train_idx]
    X_valid_text = X.iloc[valid_idx]
    y_train = y.iloc[train_idx].to_numpy()
    y_valid = y.iloc[valid_idx].to_numpy()

    # Fit all vocabularies on the outer training fold only.
    word_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True)
    char_vec = TfidfVectorizer(analyzer="char", ngram_range=(1, 5), max_features=100000, sublinear_tf=True)
    Xw_train = word_vec.fit_transform(X_train_text)
    Xw_valid = word_vec.transform(X_valid_text)
    Xc_train = char_vec.fit_transform(X_train_text)
    Xc_valid = char_vec.transform(X_valid_text)

    style = StyleFeatures()
    S_train = style.fit_transform(X_train_text)
    S_valid = style.transform(X_valid_text)
    scaler = StandardScaler()
    Ss_train = scaler.fit_transform(S_train)
    Ss_valid = scaler.transform(S_valid)

    # Train the same three global LR views used by the existing ensemble.
    word_clf = LogisticRegression(C=1.0, max_iter=1500)
    char_clf = LogisticRegression(C=1.0, max_iter=1500)
    style_clf = LogisticRegression(C=1.0, max_iter=1500)
    word_clf.fit(Xw_train, y_train)
    char_clf.fit(Xc_train, y_train)
    style_clf.fit(Ss_train, y_train)

    word_probs = align_probabilities(word_clf, word_clf.predict_proba(Xw_valid), labels)
    char_probs = align_probabilities(char_clf, char_clf.predict_proba(Xc_valid), labels)
    style_probs = align_probabilities(style_clf, style_clf.predict_proba(Ss_valid), labels)
    base_probs = geometric_blend([word_probs, char_probs, style_probs], [0.40, 0.50, 0.10])
    enhanced_probs = base_probs.copy()

    # Pairwise specialists use combined word + character features.
    # Both TF-IDF vectorizers were fit only on X_train_text above.
    Xt_train = hstack([Xw_train, Xc_train], format="csr")
    Xt_valid = hstack([Xw_valid, Xc_valid], format="csr")

    # Activate only pairs that the global model itself puts in the top two
    # and that belong to the same family. This selection uses no validation labels.
    rows_by_pair = {}
    row_top_pair = {}
    for local_row, probs in enumerate(base_probs):
        top_two = np.argsort(probs)[-2:]
        top1_col = int(top_two[-1])
        top2_col = int(top_two[-2])
        model1 = labels[top1_col]
        model2 = labels[top2_col]
        if MODEL_TO_FAMILY[model1] == MODEL_TO_FAMILY[model2]:
            pair = tuple(sorted((model1, model2)))
            rows_by_pair.setdefault(pair, []).append(local_row)
            row_top_pair[local_row] = (top1_col, top2_col)

    for pair, local_rows in rows_by_pair.items():
        pair_mask = np.isin(y_train, pair)
        pair_y = y_train[pair_mask]
        if len(np.unique(pair_y)) != 2:
            continue

        specialist = LogisticRegression(C=1.0, max_iter=1500)
        specialist.fit(Xt_train[pair_mask], pair_y)
        specialist_probs = specialist.predict_proba(Xt_valid[local_rows])
        specialist_col = {label: col for col, label in enumerate(specialist.classes_)}
        pair_activations[pair] += len(local_rows)

        for batch_idx, local_row in enumerate(local_rows):
            top1_col, top2_col = row_top_pair[local_row]
            model1 = labels[top1_col]
            model2 = labels[top2_col]
            p1 = enhanced_probs[local_row, top1_col]
            p2 = enhanced_probs[local_row, top2_col]
            mass = p1 + p2
            if mass <= 0:
                continue

            base_conditional = p1 / mass
            specialist_conditional = specialist_probs[batch_idx, specialist_col[model1]]
            blended_logit = (
                (1 - SPECIALIST_WEIGHT) * logit(base_conditional)
                + SPECIALIST_WEIGHT * logit(specialist_conditional)
            )
            new_conditional = sigmoid(blended_logit)
            enhanced_probs[local_row, top1_col] = mass * new_conditional
            enhanced_probs[local_row, top2_col] = mass * (1 - new_conditional)

            if labels[np.argmax(base_probs[local_row])] != labels[np.argmax(enhanced_probs[local_row])]:
                pair_prediction_changes[pair] += 1

    baseline_oof[valid_idx] = base_probs
    exp_b_oof[valid_idx] = enhanced_probs

    baseline_pred = labels[np.argmax(base_probs, axis=1)]
    enhanced_pred = labels[np.argmax(enhanced_probs, axis=1)]
    baseline_score = balanced_accuracy_score(y_valid, baseline_pred)
    enhanced_score = balanced_accuracy_score(y_valid, enhanced_pred)
    activated_rows = sum(len(rows) for rows in rows_by_pair.values())
    changed_rows = int(np.sum(baseline_pred != enhanced_pred))

    fold_results.append({
        "Fold": fold,
        "Baseline balanced accuracy": baseline_score,
        "Experiment B balanced accuracy": enhanced_score,
        "Delta": enhanced_score - baseline_score,
        "Rows with sibling top-2": activated_rows,
        "Predictions changed": changed_rows,
    })
    print(f"Baseline: {baseline_score:.4f} | Exp B: {enhanced_score:.4f} | Delta: {enhanced_score - baseline_score:+.4f}")
    print(f"Rows with same-family top-2: {activated_rows} | Final predictions changed: {changed_rows}")

# ------------------------------------------------------------
# 5. Results and diagnostic pair report
# ------------------------------------------------------------
results_df = pd.DataFrame(fold_results)
print("\n========== EXPERIMENT B RESULTS ==========")
display(results_df)

baseline_oof_pred = labels[np.argmax(baseline_oof, axis=1)]
exp_b_oof_pred = labels[np.argmax(exp_b_oof, axis=1)]
base_pooled = balanced_accuracy_score(y, baseline_oof_pred)
exp_b_pooled = balanced_accuracy_score(y, exp_b_oof_pred)

print(f"\nPooled OOF baseline balanced accuracy: {base_pooled:.4f}")
print(f"Pooled OOF Experiment B balanced accuracy: {exp_b_pooled:.4f}")
print(f"Pooled OOF delta: {exp_b_pooled - base_pooled:+.4f}")
print(f"Mean fold baseline: {results_df['Baseline balanced accuracy'].mean():.4f} +/- {results_df['Baseline balanced accuracy'].std(ddof=0):.4f}")
print(f"Mean fold Experiment B: {results_df['Experiment B balanced accuracy'].mean():.4f} +/- {results_df['Experiment B balanced accuracy'].std(ddof=0):.4f}")
print(f"Rows whose final label changed: {int(np.sum(baseline_oof_pred != exp_b_oof_pred))} / {len(y)}")

pair_report = pd.DataFrame([
    {
        "Sibling pair": f"{a}  vs  {b}",
        "Times pair was top-2": count,
        "Changed final prediction": pair_prediction_changes[(a, b)],
        "Family": MODEL_TO_FAMILY[a],
    }
    for (a, b), count in pair_activations.most_common()
])
print("\nPairs most often handled by specialists:")
display(pair_report.head(20))

print("\nInterpretation: keep Experiment B only if it improves pooled OOF balanced accuracy and the fold-level results are not driven by one fold.")


Rows: 2,699 | prompts: 150 | models: 18
Same-family candidate pairs available: 27

========== FOLD 1/5 ==========
Baseline: 0.3519 | Exp B: 0.3519 | Delta: +0.0000
Rows with same-family top-2: 314 | Final predictions changed: 32

========== FOLD 2/5 ==========
Baseline: 0.3241 | Exp B: 0.3241 | Delta: -0.0000
Rows with same-family top-2: 338 | Final predictions changed: 37

========== FOLD 3/5 ==========
Baseline: 0.4000 | Exp B: 0.3963 | Delta: -0.0037
Rows with same-family top-2: 314 | Final predictions changed: 37

========== FOLD 4/5 ==========
Baseline: 0.3833 | Exp B: 0.3815 | Delta: -0.0019
Rows with same-family top-2: 320 | Final predictions changed: 27

========== FOLD 5/5 ==========
Baseline: 0.3391 | Exp B: 0.3317 | Delta: -0.0074
Rows with same-family top-2: 318 | Final predictions changed: 28

========== EXPERIMENT B RESULTS ==========


,Fold,Baseline balanced accuracy,Experiment B balanced accuracy,Delta,Rows with sibling top-2,Predictions changed
0,1,0.351852,0.351852,0.000000e+00,314,32
1,2,0.324074,0.324074,-5.551115e-17,338,37
2,3,0.400000,0.396296,-3.703704e-03,314,37
3,4,0.383333,0.381481,-1.851852e-03,320,27
4,5,0.339080,0.331673,-7.407407e-03,318,28



Pooled OOF baseline balanced accuracy: 0.3597
Pooled OOF Experiment B balanced accuracy: 0.3571
Pooled OOF delta: -0.0026
Mean fold baseline: 0.3597 +/- 0.0281
Mean fold Experiment B: 0.3571 +/- 0.0279
Rows whose final label changed: 161 / 2699

Pairs most often handled by specialists:


,Sibling pair,Times pair was top-2,Changed final prediction,Family
0,gemini-3.7-flash vs gemini-3.8-flash,170,7,Google
1,gemini-3.6-flash vs gemini-3.7-flash,131,15,Google
2,gemini-3.1-pro-preview vs gemini-3.6-flash,125,11,Google
3,gpt-5.6-luna vs gpt-5.6-terra,119,16,OpenAI
4,claude-opus-4-6 vs claude-sonnet-5,104,7,Anthropic
5,gemini-3.6-flash vs gemini-3.8-flash,93,8,Google
6,gpt-5.6-luna vs gpt-6-astra,64,10,OpenAI
7,gpt-5.6-luna vs gpt-5.6-sol,62,7,OpenAI
8,gpt-5.6-sol vs gpt-5.6-terra,61,5,OpenAI
9,glm-5.3 vs glm-5.3-flash,59,10,ZAI



Interpretation: keep Experiment B only if it improves pooled OOF balanced accuracy and the fold-level results are not driven by one fold.


In [4]:
import os
import re
import hashlib
import pathlib
import warnings

import numpy as np
import pandas as pd
import torch

from scipy.sparse import hstack, csr_matrix
from scipy.stats import rankdata

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold

# ============================================================
# 1. CONFIGURATION
# ============================================================

N_SPLITS = 5

QWEN_MODEL_GPU = "Qwen/Qwen3-1.7B"
QWEN_MODEL_CPU = "Qwen/Qwen3-0.6B"

MAX_LENGTH = 512
GPU_BATCH_SIZE = 4
CPU_BATCH_SIZE = 1

# Existing text ensemble: 70% SVM ranks + 30% Logistic ranks.
SVM_WEIGHT = 0.70
LOGISTIC_WEIGHT = 0.30

# Fixed primary comparison: equal contribution from text and Qwen.
FIXED_QWEN_WEIGHT = 0.50

# Diagnostic sweep. Choosing the winner introduces tuning bias.
QWEN_WEIGHT_SWEEP = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]

# ============================================================
# 2. LOAD DATA
# ============================================================

if "train_df" not in globals():
    candidate_dirs = [
        pathlib.Path("/kaggle/input/competitions/name-that-llm"),
        pathlib.Path("/kaggle/input/name-that-llm"),
    ]

    data_dir = next(
        (p for p in candidate_dirs if (p / "train.csv").exists()),
        None
    )

    if data_dir is None:
        raise FileNotFoundError(
            "Attach the Name That LLM competition dataset."
        )

    train_df = pd.read_csv(data_dir / "train.csv")

    test_path = data_dir / "test.csv"
    test_df = (
        pd.read_csv(test_path)
        if test_path.exists()
        else pd.DataFrame(columns=["id", "text"])
    )

elif "test_df" not in globals():
    candidate_dirs = [
        pathlib.Path("/kaggle/input/competitions/name-that-llm"),
        pathlib.Path("/kaggle/input/name-that-llm"),
    ]

    data_dir = next(
        (p for p in candidate_dirs if (p / "test.csv").exists()),
        None
    )

    test_df = (
        pd.read_csv(data_dir / "test.csv")
        if data_dir
        else pd.DataFrame(columns=["id", "text"])
    )

required = {"text", "model", "prompt_id"}
missing = required - set(train_df.columns)

if missing:
    raise ValueError(f"Missing training columns: {missing}")

X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].astype(str).reset_index(drop=True)
groups = train_df["prompt_id"].astype(str).reset_index(drop=True)

if "text" not in test_df.columns:
    test_df["text"] = ""

X_test = test_df["text"].fillna("").astype(str).reset_index(drop=True)

labels = np.sort(y.unique())
n_classes = len(labels)
label_to_col = {label: i for i, label in enumerate(labels)}

print("Rows:", len(X))
print("Prompts:", groups.nunique())
print("Models:", n_classes)
print("Test rows:", len(X_test))

# ============================================================
# 3. HANDCRAFTED STYLE FEATURES
# ============================================================

PUNCTUATION = [
    ",", ";", ":", ".", "!", "?", "-", "—", "–", "'", "’",
    '"', "“", "”", "(", ")", "[", "]", "{", "}", "/",
    "\\", "*", "_", "#", "`", "…", "•"
]

PHRASES = [
    "certainly", "of course", "for example", "for instance",
    "in summary", "in conclusion", "however", "therefore",
    "moreover", "furthermore", "it is important", "overall",
    "hope this helps", "let's", "here is",
    "as a language model", "note that"
]


def extract_style(text):
    text = str(text)
    lower = text.lower()

    length = max(len(text), 1)
    words = re.findall(r"\b\w+\b", lower)
    word_count = max(len(words), 1)

    word_lengths = [len(w) for w in words]

    sentences = [
        s.strip()
        for s in re.split(r"[.!?]+", text)
        if s.strip()
    ]

    sentence_lengths = [
        len(re.findall(r"\b\w+\b", s))
        for s in sentences
    ]

    lines = text.splitlines() or [text]
    paragraphs = [
        p for p in re.split(r"\n\s*\n", text)
        if p.strip()
    ]

    features = [
        np.log1p(len(text)),
        np.log1p(len(words)),
        np.log1p(len(sentences)),
        np.log1p(len(lines)),
        np.log1p(len(paragraphs)),

        np.mean(word_lengths) if word_lengths else 0,
        np.std(word_lengths) if word_lengths else 0,
        np.max(word_lengths) if word_lengths else 0,

        len(set(words)) / word_count,
        sum(len(w) <= 3 for w in words) / word_count,
        sum(len(w) >= 8 for w in words) / word_count,

        np.mean(sentence_lengths) if sentence_lengths else 0,
        np.std(sentence_lengths) if sentence_lengths else 0,
        np.max(sentence_lengths) if sentence_lengths else 0,

        sum(c.isupper() for c in text) / length,
        sum(c.islower() for c in text) / length,
        sum(c.isdigit() for c in text) / length,
        sum(c.isspace() for c in text) / length,
        sum(ord(c) > 127 for c in text) / length,
    ]

    features.extend(
        text.count(p) / length
        for p in PUNCTUATION
    )

    bullet_count = sum(
        bool(re.match(r"^\s*(?:[-*•]\s+|\d+[.)]\s+)", line))
        for line in lines
    )

    features.extend([
        bullet_count / max(len(lines), 1),
        float("\n" in text),
        float("```" in text),
        float("##" in text),
        float(text.rstrip().endswith(("!", "?", "."))),
    ])

    features.extend(
        float(phrase in lower)
        for phrase in PHRASES
    )

    return features


def make_style_matrix(texts):
    return np.asarray(
        [extract_style(t) for t in texts],
        dtype=np.float32
    )


# ============================================================
# 4. SCORE ALIGNMENT AND RANK HELPERS
# ============================================================

def scores_to_ranks(scores, estimator_classes):
    scores = np.asarray(scores)

    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])

    aligned = np.full(
        (scores.shape[0], n_classes),
        -1e12,
        dtype=np.float64
    )

    for j, label in enumerate(estimator_classes):
        aligned[:, label_to_col[label]] = scores[:, j]

    ranks = rankdata(aligned, axis=1, method="average")

    return (ranks - 1) / max(n_classes - 1, 1)


def probabilities_to_ranks(probabilities):
    ranks = rankdata(
        probabilities,
        axis=1,
        method="average"
    )

    return (ranks - 1) / max(probabilities.shape[1] - 1, 1)


def align_probabilities(estimator, probabilities):
    aligned = np.zeros(
        (probabilities.shape[0], n_classes),
        dtype=np.float64
    )

    for j, label in enumerate(estimator.classes_):
        aligned[:, label_to_col[label]] = probabilities[:, j]

    return aligned


def geometric_blend(probability_views, weights):
    log_probabilities = np.zeros_like(
        probability_views[0],
        dtype=np.float64
    )

    for probabilities, weight in zip(probability_views, weights):
        log_probabilities += weight * np.log(
            np.clip(probabilities, 1e-12, 1.0)
        )

    log_probabilities -= log_probabilities.max(
        axis=1,
        keepdims=True
    )

    blended = np.exp(log_probabilities)

    return blended / blended.sum(axis=1, keepdims=True)


# ============================================================
# 5. EXTRACT FROZEN QWEN FEATURES
# ============================================================

# We extract features without using model labels.
# Only the lightweight classifiers later see y.

if torch.cuda.is_available():
    model_id = QWEN_MODEL_GPU
    device = torch.device("cuda")
    batch_size = GPU_BATCH_SIZE
    torch_dtype = torch.float16
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    model_id = QWEN_MODEL_CPU
    device = torch.device("cpu")
    batch_size = CPU_BATCH_SIZE
    torch_dtype = torch.float32
    MAX_LENGTH = 384

    warnings.warn(
        "No GPU detected. Falling back to Qwen3-0.6B on CPU. "
        "This is not the same experiment as Qwen3-1.7B."
    )

from transformers import AutoTokenizer, AutoModel
import transformers

print("Transformers version:", transformers.__version__)

if tuple(map(int, transformers.__version__.split(".")[:2])) < (4, 51):
    raise RuntimeError(
        "Qwen3 requires a newer Transformers version. "
        "Install transformers>=4.51.0 and restart the session."
    )

all_texts = X.tolist() + X_test.tolist()

digest = hashlib.sha256()
digest.update(f"{model_id}|{MAX_LENGTH}|last+mean-v1".encode())

for text in all_texts:
    digest.update(text.encode("utf-8", errors="replace"))
    digest.update(b"\0")

cache_key = digest.hexdigest()

output_dir = pathlib.Path("/kaggle/working")
output_dir.mkdir(parents=True, exist_ok=True)

cache_path = output_dir / "experiment_c_qwen_features.npz"

qwen_last = None
qwen_mean = None

if cache_path.exists():
    try:
        cached = np.load(cache_path, allow_pickle=False)

        if str(cached["cache_key"].item()) == cache_key:
            qwen_last = cached["qwen_last"].astype(np.float32)
            qwen_mean = cached["qwen_mean"].astype(np.float32)

            print("Loaded cached Qwen features.")
    except Exception:
        qwen_last = None
        qwen_mean = None

if qwen_last is None:
    print(f"Extracting features from {model_id} on {device}.")

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        use_fast=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Right padding makes last-token pooling straightforward.
    tokenizer.padding_side = "right"

    qwen_model = AutoModel.from_pretrained(
        model_id,
        torch_dtype=torch_dtype
    )

    qwen_model.to(device)
    qwen_model.eval()
    qwen_model.config.use_cache = False

    last_parts = []
    mean_parts = []

    with torch.inference_mode():
        for start in range(0, len(all_texts), batch_size):
            batch_texts = all_texts[start:start + batch_size]

            encoded = tokenizer(
                batch_texts,
                max_length=MAX_LENGTH,
                truncation=True,
                padding=True,
                return_tensors="pt"
            )

            encoded = {
                k: v.to(device)
                for k, v in encoded.items()
            }

            outputs = qwen_model(
                **encoded,
                use_cache=False,
                return_dict=True
            )

            hidden = outputs.last_hidden_state

            mask = encoded["attention_mask"].to(hidden.dtype)
            token_counts = mask.sum(dim=1).clamp(min=1)

            # Mean pooling across non-padding tokens.
            mean_pool = (
                hidden * mask.unsqueeze(-1)
            ).sum(dim=1) / token_counts.unsqueeze(-1)

            # Last actual token, not the final padding token.
            last_positions = token_counts.long() - 1

            row_indices = torch.arange(
                hidden.shape[0],
                device=device
            )

            last_pool = hidden[row_indices, last_positions]

            # Normalize embeddings before training linear classifiers.
            last_pool = torch.nn.functional.normalize(
                last_pool.float(),
                p=2,
                dim=1
            )

            mean_pool = torch.nn.functional.normalize(
                mean_pool.float(),
                p=2,
                dim=1
            )

            last_parts.append(
                last_pool.cpu().numpy().astype(np.float16)
            )

            mean_parts.append(
                mean_pool.cpu().numpy().astype(np.float16)
            )

            if start % (batch_size * 50) == 0:
                print(
                    f"Encoded {min(start + batch_size, len(all_texts))}"
                    f" / {len(all_texts)} texts"
                )

    qwen_last = np.concatenate(last_parts).astype(np.float32)
    qwen_mean = np.concatenate(mean_parts).astype(np.float32)

    np.savez_compressed(
        cache_path,
        cache_key=np.asarray(cache_key),
        qwen_last=qwen_last.astype(np.float16),
        qwen_mean=qwen_mean.astype(np.float16)
    )

    del qwen_model, tokenizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Cached features:", cache_path)

n_train = len(X)

qwen_train_last = qwen_last[:n_train]
qwen_train_mean = qwen_mean[:n_train]

qwen_test_last = qwen_last[n_train:]
qwen_test_mean = qwen_mean[n_train:]

print("Qwen last-token shape:", qwen_train_last.shape)
print("Qwen mean-pooling shape:", qwen_train_mean.shape)


# ============================================================
# 6. FIVE-FOLD GROUPED CROSS-VALIDATION
# ============================================================

cv = GroupKFold(n_splits=N_SPLITS)

splits = list(cv.split(X, y, groups))

baseline_oof = np.zeros(
    (len(X), n_classes),
    dtype=np.float32
)

qwen_oof = np.zeros(
    (len(X), n_classes),
    dtype=np.float32
)

fold_results = []

test_baseline_folds = []
test_qwen_folds = []

for fold, (train_idx, valid_idx) in enumerate(splits, start=1):
    print(f"\n========== FOLD {fold}/{N_SPLITS} ==========")

    X_train = X.iloc[train_idx].tolist()
    X_valid = X.iloc[valid_idx].tolist()

    y_train = y.iloc[train_idx].to_numpy()
    y_valid = y.iloc[valid_idx].to_numpy()

    Xte = X_test.tolist()

    # --------------------------------------------------------
    # A. Word and character TF-IDF
    # --------------------------------------------------------

    word_vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=25000,
        sublinear_tf=True
    )

    char_vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(1, 5),
        max_features=100000,
        sublinear_tf=True
    )

    Xw_train = word_vectorizer.fit_transform(X_train)
    Xw_valid = word_vectorizer.transform(X_valid)

    Xc_train = char_vectorizer.fit_transform(X_train)
    Xc_valid = char_vectorizer.transform(X_valid)

    if len(Xte):
        Xw_test = word_vectorizer.transform(Xte)
        Xc_test = char_vectorizer.transform(Xte)
    else:
        Xw_test = None
        Xc_test = None

    # --------------------------------------------------------
    # B. Style features
    # --------------------------------------------------------

    S_train_raw = make_style_matrix(X_train)
    S_valid_raw = make_style_matrix(X_valid)

    scaler = StandardScaler()

    S_train = scaler.fit_transform(S_train_raw)
    S_valid = scaler.transform(S_valid_raw)

    if len(Xte):
        S_test = scaler.transform(make_style_matrix(Xte))
    else:
        S_test = np.empty((0, S_train.shape[1]))

    # --------------------------------------------------------
    # C. Logistic regression ensemble
    # --------------------------------------------------------

    word_lr = LogisticRegression(
        C=1.0,
        max_iter=1500
    )

    char_lr = LogisticRegression(
        C=1.0,
        max_iter=1500
    )

    style_lr = LogisticRegression(
        C=1.0,
        max_iter=1500
    )

    word_lr.fit(Xw_train, y_train)
    char_lr.fit(Xc_train, y_train)
    style_lr.fit(S_train, y_train)

    word_prob = align_probabilities(
        word_lr,
        word_lr.predict_proba(Xw_valid)
    )

    char_prob = align_probabilities(
        char_lr,
        char_lr.predict_proba(Xc_valid)
    )

    style_prob = align_probabilities(
        style_lr,
        style_lr.predict_proba(S_valid)
    )

    logistic_prob = geometric_blend(
        [word_prob, char_prob, style_prob],
        [0.40, 0.50, 0.10]
    )

    logistic_ranks = probabilities_to_ranks(logistic_prob)

    # --------------------------------------------------------
    # D. Multi-view Linear SVM
    # --------------------------------------------------------

    Xsvm_train = hstack(
        [
            Xw_train,
            Xc_train,
            csr_matrix(0.1 * S_train)
        ],
        format="csr"
    )

    Xsvm_valid = hstack(
        [
            Xw_valid,
            Xc_valid,
            csr_matrix(0.1 * S_valid)
        ],
        format="csr"
    )

    svm = LinearSVC(
        C=1.5,
        max_iter=10000
    )

    svm.fit(Xsvm_train, y_train)

    svm_ranks = scores_to_ranks(
        svm.decision_function(Xsvm_valid),
        svm.classes_
    )

    # Existing text ensemble.
    baseline_ranks = (
        SVM_WEIGHT * svm_ranks
        + LOGISTIC_WEIGHT * logistic_ranks
    )

    baseline_oof[valid_idx] = baseline_ranks

    # --------------------------------------------------------
    # E. Qwen feature classifiers
    # --------------------------------------------------------

    qwen_last_clf = LinearSVC(C=1.0)
    qwen_mean_clf = LinearSVC(C=1.0)

    qwen_last_clf.fit(
        qwen_train_last[train_idx],
        y_train
    )

    qwen_mean_clf.fit(
        qwen_train_mean[train_idx],
        y_train
    )

    qwen_last_ranks = scores_to_ranks(
        qwen_last_clf.decision_function(
            qwen_train_last[valid_idx]
        ),
        qwen_last_clf.classes_
    )

    qwen_mean_ranks = scores_to_ranks(
        qwen_mean_clf.decision_function(
            qwen_train_mean[valid_idx]
        ),
        qwen_mean_clf.classes_
    )

    qwen_ranks = (
        0.50 * qwen_last_ranks
        + 0.50 * qwen_mean_ranks
    )

    qwen_oof[valid_idx] = qwen_ranks

    # --------------------------------------------------------
    # F. Compare predictions
    # --------------------------------------------------------

    baseline_pred = labels[np.argmax(baseline_ranks, axis=1)]
    qwen_pred = labels[np.argmax(qwen_ranks, axis=1)]

    hybrid_ranks = (
        (1 - FIXED_QWEN_WEIGHT) * baseline_ranks
        + FIXED_QWEN_WEIGHT * qwen_ranks
    )

    hybrid_pred = labels[np.argmax(hybrid_ranks, axis=1)]

    baseline_score = balanced_accuracy_score(
        y_valid,
        baseline_pred
    )

    qwen_score = balanced_accuracy_score(
        y_valid,
        qwen_pred
    )

    hybrid_score = balanced_accuracy_score(
        y_valid,
        hybrid_pred
    )

    fold_results.append({
        "Fold": fold,
        "Text baseline": baseline_score,
        "Qwen only": qwen_score,
        "Hybrid": hybrid_score,
        "Delta": hybrid_score - baseline_score
    })

    print(f"Text baseline: {baseline_score:.4f}")
    print(f"Qwen only:     {qwen_score:.4f}")
    print(f"Hybrid:        {hybrid_score:.4f}")
    print(f"Delta:         {hybrid_score - baseline_score:+.4f}")

    # Test predictions for the eventual submission.
    if len(Xte):
        word_prob_test = align_probabilities(
            word_lr,
            word_lr.predict_proba(Xw_test)
        )

        char_prob_test = align_probabilities(
            char_lr,
            char_lr.predict_proba(Xc_test)
        )

        style_prob_test = align_probabilities(
            style_lr,
            style_lr.predict_proba(S_test)
        )

        logistic_prob_test = geometric_blend(
            [word_prob_test, char_prob_test, style_prob_test],
            [0.40, 0.50, 0.10]
        )

        logistic_ranks_test = probabilities_to_ranks(
            logistic_prob_test
        )

        Xsvm_test = hstack(
            [
                Xw_test,
                Xc_test,
                csr_matrix(0.1 * S_test)
            ],
            format="csr"
        )

        svm_ranks_test = scores_to_ranks(
            svm.decision_function(Xsvm_test),
            svm.classes_
        )

        test_baseline_folds.append(
            SVM_WEIGHT * svm_ranks_test
            + LOGISTIC_WEIGHT * logistic_ranks_test
        )

        qwen_last_test_ranks = scores_to_ranks(
            qwen_last_clf.decision_function(qwen_test_last),
            qwen_last_clf.classes_
        )

        qwen_mean_test_ranks = scores_to_ranks(
            qwen_mean_clf.decision_function(qwen_test_mean),
            qwen_mean_clf.classes_
        )

        test_qwen_folds.append(
            0.50 * qwen_last_test_ranks
            + 0.50 * qwen_mean_test_ranks
        )


# ============================================================
# 7. RESULTS AND WEIGHT SWEEP
# ============================================================

results = pd.DataFrame(fold_results)

print("\n========== FOLD RESULTS ==========")
display(results)

baseline_pred = labels[np.argmax(baseline_oof, axis=1)]
qwen_pred = labels[np.argmax(qwen_oof, axis=1)]

hybrid_oof = (
    (1 - FIXED_QWEN_WEIGHT) * baseline_oof
    + FIXED_QWEN_WEIGHT * qwen_oof
)

hybrid_pred = labels[np.argmax(hybrid_oof, axis=1)]

baseline_score = balanced_accuracy_score(y, baseline_pred)
qwen_score = balanced_accuracy_score(y, qwen_pred)
hybrid_score = balanced_accuracy_score(y, hybrid_pred)

print("\n========== POOLED OOF RESULTS ==========")
print(f"Text baseline: {baseline_score:.4f}")
print(f"Qwen only:     {qwen_score:.4f}")
print(f"Hybrid:        {hybrid_score:.4f}")
print(f"Hybrid delta:  {hybrid_score - baseline_score:+.4f}")

# Diagnostic weight sweep.
weight_rows = []

for weight in [0.0] + QWEN_WEIGHT_SWEEP:
    combined = (
        (1 - weight) * baseline_oof
        + weight * qwen_oof
    )

    pred = labels[np.argmax(combined, axis=1)]

    score = balanced_accuracy_score(y, pred)

    weight_rows.append({
        "Qwen weight": weight,
        "Pooled OOF balanced accuracy": score,
        "Delta vs baseline": score - baseline_score
    })

weight_results = pd.DataFrame(weight_rows)

print("\n========== WEIGHT SWEEP ==========")
display(weight_results)

print("\n========== COMPLEMENTARITY ==========")

baseline_correct = baseline_pred == y.to_numpy()
qwen_correct = qwen_pred == y.to_numpy()
hybrid_correct = hybrid_pred == y.to_numpy()

print(
    "Baseline wrong, Qwen correct:",
    int(np.sum((~baseline_correct) & qwen_correct))
)

print(
    "Baseline correct, Qwen wrong:",
    int(np.sum(baseline_correct & (~qwen_correct)))
)

print(
    "Baseline errors fixed by hybrid:",
    int(np.sum((~baseline_correct) & hybrid_correct))
)

print(
    "Baseline correct predictions broken by hybrid:",
    int(np.sum(baseline_correct & (~hybrid_correct)))
)


# ============================================================
# 8. SAVE OOF ERROR ANALYSIS
# ============================================================

oof_output = pd.DataFrame({
    "prompt_id": groups,
    "true_model": y,
    "baseline_prediction": baseline_pred,
    "qwen_prediction": qwen_pred,
    "hybrid_prediction": hybrid_pred,
    "baseline_correct": baseline_correct,
    "qwen_correct": qwen_correct,
    "hybrid_correct": hybrid_correct
})

oof_path = output_dir / "experiment_c_oof_predictions.csv"
oof_output.to_csv(oof_path, index=False)

print("\nSaved OOF diagnostics:", oof_path)




Rows: 2699
Prompts: 150
Models: 18
Test rows: 3
GPU detected: Tesla T4
Transformers version: 5.16.1
Extracting features from Qwen/Qwen3-1.7B on cuda.


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Encoded 4 / 2702 texts
Encoded 204 / 2702 texts
Encoded 404 / 2702 texts
Encoded 604 / 2702 texts
Encoded 804 / 2702 texts
Encoded 1004 / 2702 texts
Encoded 1204 / 2702 texts
Encoded 1404 / 2702 texts
Encoded 1604 / 2702 texts
Encoded 1804 / 2702 texts
Encoded 2004 / 2702 texts
Encoded 2204 / 2702 texts
Encoded 2404 / 2702 texts
Encoded 2604 / 2702 texts
Cached features: /kaggle/working/experiment_c_qwen_features.npz
Qwen last-token shape: (2699, 2048)
Qwen mean-pooling shape: (2699, 2048)

========== FOLD 1/5 ==========
Text baseline: 0.3889
Qwen only:     0.2759
Hybrid:        0.3704
Delta:         -0.0185

========== FOLD 2/5 ==========
Text baseline: 0.3463
Qwen only:     0.2741
Hybrid:        0.3741
Delta:         +0.0278

========== FOLD 3/5 ==========
Text baseline: 0.3815
Qwen only:     0.2815
Hybrid:        0.3981
Delta:         +0.0167

========== FOLD 4/5 ==========
Text baseline: 0.3796
Qwen only:     0.2630
Hybrid:        0.3759
Delta:         -0.0037

========== FOLD 5/5 

,Fold,Text baseline,Qwen only,Hybrid,Delta
0,1,0.388889,0.275926,0.370370,-0.018519
1,2,0.346296,0.274074,0.374074,0.027778
2,3,0.381481,0.281481,0.398148,0.016667
3,4,0.379630,0.262963,0.375926,-0.003704
4,5,0.352107,0.281737,0.355875,0.003768



========== POOLED OOF RESULTS ==========
Text baseline: 0.3697
Qwen only:     0.2760
Hybrid:        0.3748
Hybrid delta:  +0.0052

========== WEIGHT SWEEP ==========


,Qwen weight,Pooled OOF balanced accuracy,Delta vs baseline
0,0.0,0.369672,0.000000
1,0.1,0.380413,0.010741
2,0.2,0.388929,0.019257
3,0.3,0.385961,0.016289
4,0.4,0.384850,0.015178
5,0.5,0.374847,0.005175
6,0.6,0.362255,-0.007417



========== COMPLEMENTARITY ==========
Baseline wrong, Qwen correct: 299
Baseline correct, Qwen wrong: 552
Baseline errors fixed by hybrid: 272
Baseline correct predictions broken by hybrid: 258

Saved OOF diagnostics: /kaggle/working/experiment_c_oof_predictions.csv


In [5]:
weights = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

rows = []

for w in weights:
    combined = (1 - w) * baseline_oof + w * qwen_oof
    pooled_pred = labels[np.argmax(combined, axis=1)]

    fold_scores = []

    for fold, (_, valid_idx) in enumerate(splits, start=1):
        fold_pred = labels[np.argmax(combined[valid_idx], axis=1)]

        score = balanced_accuracy_score(
            y.iloc[valid_idx],
            fold_pred
        )

        fold_scores.append(score)

    rows.append({
        "Qwen weight": w,
        "Pooled OOF": balanced_accuracy_score(y, pooled_pred),
        "Mean fold score": np.mean(fold_scores),
        "Fold std": np.std(fold_scores),
        "Fold scores": [
            round(s, 4) for s in fold_scores
        ]
    })

weight_check = pd.DataFrame(rows)

display(
    weight_check.sort_values(
        "Pooled OOF",
        ascending=False
    ).reset_index(drop=True)
)

,Qwen weight,Pooled OOF,Mean fold score,Fold std,Fold scores
0,0.20,0.388929,0.388940,0.018548,"[0.387, 0.3815, 0.413, 0.4037, 0.3595]"
1,0.25,0.388558,0.388570,0.022256,"[0.3741, 0.3926, 0.4241, 0.3944, 0.3577]"
2,0.30,0.385961,0.385977,0.020774,"[0.3778, 0.3759, 0.4204, 0.3963, 0.3595]"
3,0.35,0.385958,0.385977,0.019186,"[0.3815, 0.3833, 0.4148, 0.3944, 0.3558]"
4,0.40,0.384850,0.384866,0.018779,"[0.3852, 0.3833, 0.413, 0.3889, 0.354]"
5,0.15,0.383371,0.383384,0.019631,"[0.3944, 0.3648, 0.4074, 0.3944, 0.3558]"
6,0.10,0.380413,0.380421,0.019625,"[0.3889, 0.3611, 0.4093, 0.387, 0.3558]"
7,0.05,0.375600,0.375607,0.020538,"[0.3889, 0.3519, 0.4, 0.387, 0.3503]"
8,0.00,0.369672,0.369681,0.017105,"[0.3889, 0.3463, 0.3815, 0.3796, 0.3521]"


In [7]:
import numpy as np
import pandas as pd

from sklearn.metrics import balanced_accuracy_score

# ============================================================
# EXPERIMENT D1: FAMILY-WISE QWEN ERROR ANALYSIS
# Requires Experiment C variables to exist in memory.
# ============================================================

QWEN_WEIGHT = 0.20

# ------------------------------------------------------------
# 1. Prepare labels and validate inputs
# ------------------------------------------------------------

y_true = np.asarray(y, dtype=str)

# IMPORTANT: NumPy arrays support array-based indexing.
model_labels = np.asarray(labels, dtype=str)

baseline_oof = np.asarray(baseline_oof)
qwen_oof = np.asarray(qwen_oof)

if baseline_oof.shape != qwen_oof.shape:
    raise ValueError("Baseline and Qwen OOF shapes do not match.")

if baseline_oof.shape != (len(y_true), len(model_labels)):
    raise ValueError(
        f"Shape mismatch: baseline={baseline_oof.shape}, "
        f"expected={(len(y_true), len(model_labels))}"
    )

# ------------------------------------------------------------
# 2. Map each model to its family
# ------------------------------------------------------------

def get_family(model):
    name = str(model).lower()

    prefixes = {
        "gpt": "OpenAI",
        "claude": "Anthropic",
        "gemini": "Google",
        "glm": "ZAI",
        "qwen": "Qwen",
        "kimi": "Moonshot",
        "deepseek": "DeepSeek",
        "llama": "Meta",
        "mistral": "Mistral",
    }

    for prefix, family in prefixes.items():
        if name.startswith(prefix):
            return family

    return "Other"


family_map = {
    model: get_family(model)
    for model in model_labels
}

print("MODEL → FAMILY MAPPING")

mapping_df = pd.DataFrame({
    "Model": model_labels,
    "Family": [family_map[m] for m in model_labels]
})

display(
    mapping_df.sort_values(["Family", "Model"]).reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Generate baseline, Qwen and hybrid predictions
# ------------------------------------------------------------

baseline_pred = model_labels[np.argmax(baseline_oof, axis=1)]

qwen_pred = model_labels[np.argmax(qwen_oof, axis=1)]

hybrid_oof = (
    (1.0 - QWEN_WEIGHT) * baseline_oof
    + QWEN_WEIGHT * qwen_oof
)

hybrid_pred = model_labels[np.argmax(hybrid_oof, axis=1)]

baseline_correct = baseline_pred == y_true
qwen_correct = qwen_pred == y_true
hybrid_correct = hybrid_pred == y_true

print("\nOVERALL OOF RESULTS")
print(
    f"Baseline: {balanced_accuracy_score(y_true, baseline_pred):.4f}"
)
print(
    f"Qwen:     {balanced_accuracy_score(y_true, qwen_pred):.4f}"
)
print(
    f"Hybrid:   {balanced_accuracy_score(y_true, hybrid_pred):.4f}"
)

print("\nERROR COMPLEMENTARITY")
print(
    "Baseline wrong, Qwen correct:",
    int(np.sum((~baseline_correct) & qwen_correct))
)
print(
    "Baseline correct, Qwen wrong:",
    int(np.sum(baseline_correct & (~qwen_correct)))
)
print(
    "Hybrid fixes baseline errors:",
    int(np.sum((~baseline_correct) & hybrid_correct))
)
print(
    "Hybrid breaks baseline correct predictions:",
    int(np.sum(baseline_correct & (~hybrid_correct)))
)

# ------------------------------------------------------------
# 4. Analyze by TRUE model family
# ------------------------------------------------------------

true_families = np.array(
    [family_map.get(model, "Other") for model in y_true]
)

family_rows = []

for family in sorted(set(true_families)):
    mask = true_families == family

    family_models = [
        model for model in model_labels
        if family_map[model] == family
    ]

    rescued = (~baseline_correct[mask]) & qwen_correct[mask]
    harmed = baseline_correct[mask] & (~qwen_correct[mask])

    family_rows.append({
        "True family": family,
        "Rows": int(mask.sum()),
        "Models": len(family_models),

        "Baseline balanced accuracy": balanced_accuracy_score(
            y_true[mask], baseline_pred[mask]
        ),

        "Qwen balanced accuracy": balanced_accuracy_score(
            y_true[mask], qwen_pred[mask]
        ),

        "Hybrid balanced accuracy": balanced_accuracy_score(
            y_true[mask], hybrid_pred[mask]
        ),

        "Baseline errors Qwen gets right": int(rescued.sum()),
        "Baseline correct Qwen gets wrong": int(harmed.sum()),

        "Hybrid fixes baseline errors": int(
            ((~baseline_correct[mask]) & hybrid_correct[mask]).sum()
        ),

        "Hybrid breaks baseline correct": int(
            (baseline_correct[mask] & (~hybrid_correct[mask])).sum()
        ),
    })

family_results = pd.DataFrame(family_rows)

family_results["Hybrid delta"] = (
    family_results["Hybrid balanced accuracy"]
    - family_results["Baseline balanced accuracy"]
)

print("\nRESULTS BY TRUE FAMILY")
display(
    family_results.sort_values(
        "Hybrid delta", ascending=False
    ).reset_index(drop=True).round(4)
)

# ------------------------------------------------------------
# 5. Analyze by INDIVIDUAL MODEL
# ------------------------------------------------------------

model_rows = []

for model in model_labels:
    mask = y_true == model

    rescued = (~baseline_correct[mask]) & qwen_correct[mask]
    harmed = baseline_correct[mask] & (~qwen_correct[mask])

    model_rows.append({
        "True model": model,
        "Family": family_map[model],
        "Rows": int(mask.sum()),

        "Baseline recall": float(baseline_correct[mask].mean()),
        "Qwen recall": float(qwen_correct[mask].mean()),
        "Hybrid recall": float(hybrid_correct[mask].mean()),

        "Qwen rescues": int(rescued.sum()),
        "Qwen harms": int(harmed.sum()),

        "Hybrid delta": float(
            hybrid_correct[mask].mean()
            - baseline_correct[mask].mean()
        ),
    })

model_results = pd.DataFrame(model_rows)

print("\nRESULTS BY INDIVIDUAL MODEL")
display(
    model_results.sort_values(
        "Hybrid delta", ascending=False
    ).reset_index(drop=True).round(4)
)

# ------------------------------------------------------------
# 6. Analyze by BASELINE-PREDICTED FAMILY
# This is potentially useful for inference-time routing.
# ------------------------------------------------------------

predicted_families = np.array(
    [family_map.get(model, "Other") for model in baseline_pred]
)

routing_rows = []

for family in sorted(set(predicted_families)):
    mask = predicted_families == family

    rescued = (~baseline_correct[mask]) & qwen_correct[mask]
    harmed = baseline_correct[mask] & (~qwen_correct[mask])

    routing_rows.append({
        "Baseline predicted family": family,
        "Rows routed": int(mask.sum()),

        "Baseline accuracy": float(baseline_correct[mask].mean()),
        "Qwen accuracy": float(qwen_correct[mask].mean()),
        "Hybrid accuracy": float(hybrid_correct[mask].mean()),

        "Qwen rescues": int(rescued.sum()),
        "Qwen harms": int(harmed.sum()),

        "Hybrid delta": float(
            hybrid_correct[mask].mean()
            - baseline_correct[mask].mean()
        ),
    })

routing_results = pd.DataFrame(routing_rows)

print("\nRESULTS BY BASELINE-PREDICTED FAMILY")
display(
    routing_results.sort_values(
        "Hybrid delta", ascending=False
    ).reset_index(drop=True).round(4)
)

# ------------------------------------------------------------
# 7. Save all analysis tables
# ------------------------------------------------------------

output_dir = "/kaggle/working"

family_results.to_csv(
    f"{output_dir}/experiment_d1_family_results.csv",
    index=False
)

model_results.to_csv(
    f"{output_dir}/experiment_d1_model_results.csv",
    index=False
)

routing_results.to_csv(
    f"{output_dir}/experiment_d1_routing_results.csv",
    index=False
)

print("\nSaved analysis CSV files in /kaggle/working/")


MODEL → FAMILY MAPPING


,Model,Family
0,claude-fable-5,Anthropic
1,claude-opus-4-6,Anthropic
2,claude-opus-4-8,Anthropic
3,claude-opus-5,Anthropic
4,claude-sonnet-5,Anthropic
5,gemini-3.1-pro-preview,Google
6,gemini-3.6-flash,Google
7,gemini-3.7-flash,Google
8,gemini-3.8-flash,Google
9,kimi-k3,Moonshot



OVERALL OOF RESULTS
Baseline: 0.3697
Qwen:     0.2760
Hybrid:   0.3889

ERROR COMPLEMENTARITY
Baseline wrong, Qwen correct: 299
Baseline correct, Qwen wrong: 552
Hybrid fixes baseline errors: 117
Hybrid breaks baseline correct predictions: 65

RESULTS BY TRUE FAMILY


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics

,True family,Rows,Models,Baseline balanced accuracy,Qwen balanced accuracy,Hybrid balanced accuracy,Baseline errors Qwen gets right,Baseline correct Qwen gets wrong,Hybrid fixes baseline errors,Hybrid breaks baseline correct,Hybrid delta
0,Moonshot,150,1,0.2667,0.3400,0.3267,29,18,10,1,0.0600
1,OpenAI,750,5,0.3440,0.2373,0.3693,76,156,38,19,0.0253
2,Google,600,4,0.5000,0.3650,0.5167,69,150,19,9,0.0167
3,Anthropic,749,5,0.3788,0.3202,0.3948,93,137,36,24,0.0160
4,ZAI,300,2,0.1867,0.1167,0.1933,23,44,8,6,0.0067
5,Qwen,150,1,0.4000,0.1467,0.4000,9,47,6,6,0.0000



RESULTS BY INDIVIDUAL MODEL


,True model,Family,Rows,Baseline recall,Qwen recall,Hybrid recall,Qwen rescues,Qwen harms,Hybrid delta
0,claude-opus-5,Anthropic,150,0.6667,0.7733,0.7333,30,14,0.0667
1,kimi-k3,Moonshot,150,0.2667,0.3400,0.3267,29,18,0.0600
2,gpt-5.6-terra,OpenAI,150,0.2667,0.1867,0.3200,14,26,0.0533
3,gemini-3.8-flash,Google,150,0.3600,0.2400,0.4000,17,35,0.0400
4,gpt-5.6-luna,OpenAI,150,0.3333,0.3533,0.3600,20,17,0.0267
5,gpt-5.5,OpenAI,150,0.3600,0.1400,0.3867,16,49,0.0267
6,claude-opus-4-6,Anthropic,150,0.4867,0.3333,0.5133,22,45,0.0267
7,gpt-6-astra,OpenAI,150,0.6067,0.4467,0.6267,17,41,0.0200
8,glm-5.3,ZAI,150,0.1600,0.0933,0.1800,12,22,0.0200
9,gemini-3.7-flash,Google,150,0.3733,0.4267,0.3933,28,20,0.0200



RESULTS BY BASELINE-PREDICTED FAMILY


,Baseline predicted family,Rows routed,Baseline accuracy,Qwen accuracy,Hybrid accuracy,Qwen rescues,Qwen harms,Hybrid delta
0,Moonshot,140,0.2857,0.2286,0.3429,10,18,0.0571
1,Qwen,139,0.4317,0.2158,0.4532,17,47,0.0216
2,Google,667,0.4498,0.3448,0.4708,80,150,0.0210
3,Anthropic,738,0.3848,0.2927,0.4024,69,137,0.0176
4,ZAI,258,0.2171,0.1899,0.2326,37,44,0.0155
5,OpenAI,757,0.3408,0.2483,0.3540,86,156,0.0132



Saved analysis CSV files in /kaggle/working/


In [8]:
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix
from scipy.stats import rankdata

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score

# ============================================================
# D2: FAMILY-ADAPTIVE QWEN WEIGHTING
# Requires Experiment C feature arrays in the current session.
# ============================================================

if "qwen_train_last" not in globals() or "qwen_train_mean" not in globals():
    raise RuntimeError(
        "Qwen features are missing. Re-run Experiment C's feature "
        "extraction/loading cells first."
    )

if "make_style_matrix" not in globals():
    raise RuntimeError(
        "Run Experiment C's style-feature definition cell first."
    )

X = pd.Series(X).reset_index(drop=True)
y = pd.Series(y).astype(str).reset_index(drop=True)
groups = pd.Series(groups).reset_index(drop=True)

labels = np.asarray(labels, dtype=str)
n_classes = len(labels)
label_to_col = {label: i for i, label in enumerate(labels)}

assert len(X) == len(y) == len(groups)
assert qwen_train_last.shape[0] == len(X)
assert qwen_train_mean.shape[0] == len(X)

# ------------------------------------------------------------
# 1. Model-family mapping
# ------------------------------------------------------------

def get_family(model):
    model = str(model).lower()

    prefixes = {
        "gpt": "OpenAI",
        "claude": "Anthropic",
        "gemini": "Google",
        "glm": "ZAI",
        "qwen": "Qwen",
        "kimi": "Moonshot",
        "deepseek": "DeepSeek",
        "llama": "Meta",
        "mistral": "Mistral",
    }

    for prefix, family in prefixes.items():
        if model.startswith(prefix):
            return family

    return "Other"


family_map = {model: get_family(model) for model in labels}


# ------------------------------------------------------------
# 2. Score/rank helpers
# ------------------------------------------------------------

def scores_to_ranks(scores, estimator_classes):
    scores = np.asarray(scores)

    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])

    aligned = np.full(
        (scores.shape[0], n_classes),
        -1e12,
        dtype=np.float64
    )

    for j, label in enumerate(estimator_classes):
        aligned[:, label_to_col[str(label)]] = scores[:, j]

    ranks = rankdata(aligned, axis=1, method="average")
    return (ranks - 1) / max(n_classes - 1, 1)


def probabilities_to_ranks(probabilities):
    ranks = rankdata(
        probabilities,
        axis=1,
        method="average"
    )
    return (ranks - 1) / max(n_classes - 1, 1)


def align_probabilities(estimator, probabilities):
    aligned = np.zeros(
        (probabilities.shape[0], n_classes),
        dtype=np.float64
    )

    for j, label in enumerate(estimator.classes_):
        aligned[:, label_to_col[str(label)]] = probabilities[:, j]

    return aligned


def geometric_blend(probability_views, weights):
    log_p = np.zeros_like(probability_views[0], dtype=np.float64)

    for probabilities, weight in zip(probability_views, weights):
        log_p += weight * np.log(
            np.clip(probabilities, 1e-12, 1.0)
        )

    log_p -= log_p.max(axis=1, keepdims=True)
    result = np.exp(log_p)

    return result / result.sum(axis=1, keepdims=True)


# ------------------------------------------------------------
# 3. Fit both systems on a train split, predict a validation split
# ------------------------------------------------------------

def fit_predict_pair(train_idx, valid_idx):
    X_train = X.iloc[train_idx].tolist()
    X_valid = X.iloc[valid_idx].tolist()
    y_train = y.iloc[train_idx].to_numpy()

    # Word TF-IDF
    word_vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=25000,
        sublinear_tf=True
    )

    Xw_train = word_vectorizer.fit_transform(X_train)
    Xw_valid = word_vectorizer.transform(X_valid)

    # Character TF-IDF
    char_vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(1, 5),
        max_features=100000,
        sublinear_tf=True
    )

    Xc_train = char_vectorizer.fit_transform(X_train)
    Xc_valid = char_vectorizer.transform(X_valid)

    # Style features; fit scaling on training data only.
    S_train_raw = make_style_matrix(X_train)
    S_valid_raw = make_style_matrix(X_valid)

    scaler = StandardScaler()
    S_train = scaler.fit_transform(S_train_raw)
    S_valid = scaler.transform(S_valid_raw)

    # Logistic regression on each view.
    word_lr = LogisticRegression(C=1.0, max_iter=1500)
    char_lr = LogisticRegression(C=1.0, max_iter=1500)
    style_lr = LogisticRegression(C=1.0, max_iter=1500)

    word_lr.fit(Xw_train, y_train)
    char_lr.fit(Xc_train, y_train)
    style_lr.fit(S_train, y_train)

    word_prob = align_probabilities(
        word_lr, word_lr.predict_proba(Xw_valid)
    )

    char_prob = align_probabilities(
        char_lr, char_lr.predict_proba(Xc_valid)
    )

    style_prob = align_probabilities(
        style_lr, style_lr.predict_proba(S_valid)
    )

    logistic_prob = geometric_blend(
        [word_prob, char_prob, style_prob],
        [0.40, 0.50, 0.10]
    )

    logistic_ranks = probabilities_to_ranks(logistic_prob)

    # Multi-view SVM.
    Xsvm_train = hstack(
        [
            Xw_train,
            Xc_train,
            csr_matrix(0.1 * S_train)
        ],
        format="csr"
    )

    Xsvm_valid = hstack(
        [
            Xw_valid,
            Xc_valid,
            csr_matrix(0.1 * S_valid)
        ],
        format="csr"
    )

    svm = LinearSVC(C=1.5, max_iter=10000)
    svm.fit(Xsvm_train, y_train)

    svm_ranks = scores_to_ranks(
        svm.decision_function(Xsvm_valid),
        svm.classes_
    )

    baseline_ranks = 0.70 * svm_ranks + 0.30 * logistic_ranks

    # Qwen feature specialists.
    qwen_last_clf = LinearSVC(C=1.0)
    qwen_mean_clf = LinearSVC(C=1.0)

    qwen_last_clf.fit(qwen_train_last[train_idx], y_train)
    qwen_mean_clf.fit(qwen_train_mean[train_idx], y_train)

    qwen_last_ranks = scores_to_ranks(
        qwen_last_clf.decision_function(
            qwen_train_last[valid_idx]
        ),
        qwen_last_clf.classes_
    )

    qwen_mean_ranks = scores_to_ranks(
        qwen_mean_clf.decision_function(
            qwen_train_mean[valid_idx]
        ),
        qwen_mean_clf.classes_
    )

    qwen_ranks = 0.50 * qwen_last_ranks + 0.50 * qwen_mean_ranks

    return baseline_ranks, qwen_ranks


# ------------------------------------------------------------
# 4. Split by prompt, not by individual response
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

dev_idx, holdout_idx = next(
    splitter.split(
        np.zeros(len(X)),
        y,
        groups
    )
)

print("Development prompts:", groups.iloc[dev_idx].nunique())
print("Holdout prompts:", groups.iloc[holdout_idx].nunique())
print("Development rows:", len(dev_idx))
print("Holdout rows:", len(holdout_idx))


# ------------------------------------------------------------
# 5. Generate development OOF predictions
#    No holdout prompts enter these model fits.
# ------------------------------------------------------------

inner_cv = GroupKFold(n_splits=5)

dev_baseline_oof = np.zeros((len(dev_idx), n_classes))
dev_qwen_oof = np.zeros((len(dev_idx), n_classes))

dev_y = y.iloc[dev_idx].to_numpy()

for fold, (train_local, valid_local) in enumerate(
    inner_cv.split(
        dev_idx,
        dev_y,
        groups.iloc[dev_idx]
    ),
    start=1
):
    train_global = dev_idx[train_local]
    valid_global = dev_idx[valid_local]

    baseline_ranks, qwen_ranks = fit_predict_pair(
        train_global,
        valid_global
    )

    dev_baseline_oof[valid_local] = baseline_ranks
    dev_qwen_oof[valid_local] = qwen_ranks

    print(f"Completed development fold {fold}/5")


# ------------------------------------------------------------
# 6. Route rows according to the baseline's top prediction
# ------------------------------------------------------------

dev_baseline_pred = labels[
    np.argmax(dev_baseline_oof, axis=1)
]

dev_routes = np.array([
    family_map[model]
    for model in dev_baseline_pred
])

candidate_weights = np.round(
    np.arange(0.0, 0.501, 0.05),
    2
)

GLOBAL_WEIGHT = 0.20

# Start with a uniform 20% Qwen contribution.
family_weights = {
    family: GLOBAL_WEIGHT
    for family in np.unique(dev_routes)
}


def combine_by_family(baseline, qwen, routes, weights):
    combined = np.empty_like(baseline)

    for family in np.unique(routes):
        mask = routes == family
        weight = weights.get(family, GLOBAL_WEIGHT)

        combined[mask] = (
            (1.0 - weight) * baseline[mask]
            + weight * qwen[mask]
        )

    return combined


# ------------------------------------------------------------
# 7. Tune weights using development OOF predictions only
# ------------------------------------------------------------

# Each candidate is assessed on the overall balanced accuracy.
# Other families retain their current weights during each search.
# Ties prefer the global 20% weight.

for family in sorted(family_weights):
    best_weight = GLOBAL_WEIGHT
    best_score = -np.inf

    for weight in candidate_weights:
        trial_weights = family_weights.copy()
        trial_weights[family] = float(weight)

        trial_ranks = combine_by_family(
            dev_baseline_oof,
            dev_qwen_oof,
            dev_routes,
            trial_weights
        )

        trial_pred = labels[np.argmax(trial_ranks, axis=1)]

        score = balanced_accuracy_score(dev_y, trial_pred)

        if (
            score > best_score + 1e-10
            or (
                abs(score - best_score) <= 1e-10
                and abs(weight - GLOBAL_WEIGHT)
                < abs(best_weight - GLOBAL_WEIGHT)
            )
        ):
            best_score = score
            best_weight = float(weight)

    family_weights[family] = best_weight


print("\nWEIGHTS SELECTED ON DEVELOPMENT OOF")
display(
    pd.DataFrame([
        {
            "Baseline predicted family": family,
            "Development rows": int((dev_routes == family).sum()),
            "Selected Qwen weight": weight
        }
        for family, weight in sorted(family_weights.items())
    ])
)

# Development score is a tuning diagnostic, not the final estimate.
dev_adaptive = combine_by_family(
    dev_baseline_oof,
    dev_qwen_oof,
    dev_routes,
    family_weights
)

print(
    "Development OOF baseline:",
    round(
        balanced_accuracy_score(
            dev_y,
            labels[np.argmax(dev_baseline_oof, axis=1)]
        ),
        4
    )
)

print(
    "Development OOF adaptive:",
    round(
        balanced_accuracy_score(
            dev_y,
            labels[np.argmax(dev_adaptive, axis=1)]
        ),
        4
    )
)


# ------------------------------------------------------------
# 8. Evaluate on untouched holdout prompts
# ------------------------------------------------------------

holdout_baseline, holdout_qwen = fit_predict_pair(
    dev_idx,
    holdout_idx
)

holdout_y = y.iloc[holdout_idx].to_numpy()

holdout_baseline_pred = labels[
    np.argmax(holdout_baseline, axis=1)
]

holdout_routes = np.array([
    family_map[model]
    for model in holdout_baseline_pred
])

# Control: baseline only.
baseline_score = balanced_accuracy_score(
    holdout_y,
    holdout_baseline_pred
)

# Control: one fixed 20% Qwen weight for everyone.
global_ranks = (
    (1.0 - GLOBAL_WEIGHT) * holdout_baseline
    + GLOBAL_WEIGHT * holdout_qwen
)

global_pred = labels[np.argmax(global_ranks, axis=1)]

global_score = balanced_accuracy_score(holdout_y, global_pred)

# Candidate: family-specific weights selected on development prompts.
adaptive_ranks = combine_by_family(
    holdout_baseline,
    holdout_qwen,
    holdout_routes,
    family_weights
)

adaptive_pred = labels[np.argmax(adaptive_ranks, axis=1)]

adaptive_score = balanced_accuracy_score(
    holdout_y,
    adaptive_pred
)

holdout_results = pd.DataFrame([
    {
        "Method": "Text baseline",
        "Holdout balanced accuracy": baseline_score
    },
    {
        "Method": "Global 20% Qwen blend",
        "Holdout balanced accuracy": global_score
    },
    {
        "Method": "Family-adaptive Qwen blend",
        "Holdout balanced accuracy": adaptive_score
    }
])

holdout_results["Delta vs baseline"] = (
    holdout_results["Holdout balanced accuracy"] - baseline_score
)

print("\nFINAL HELD-OUT RESULTS")
display(holdout_results.round(4))

print(
    "\nAdaptive minus global blend:",
    round(adaptive_score - global_score, 4)
)

Development prompts: 120
Holdout prompts: 30
Development rows: 2159
Holdout rows: 540
Completed development fold 1/5
Completed development fold 2/5
Completed development fold 3/5
Completed development fold 4/5
Completed development fold 5/5

WEIGHTS SELECTED ON DEVELOPMENT OOF


,Baseline predicted family,Development rows,Selected Qwen weight
0,Anthropic,588,0.25
1,Google,540,0.20
2,Moonshot,112,0.40
3,OpenAI,601,0.25
4,Qwen,113,0.20
5,ZAI,205,0.10


Development OOF baseline: 0.3491
Development OOF adaptive: 0.3746

FINAL HELD-OUT RESULTS


,Method,Holdout balanced accuracy,Delta vs baseline
0,Text baseline,0.3574,0.0000
1,Global 20% Qwen blend,0.3722,0.0148
2,Family-adaptive Qwen blend,0.3722,0.0148



Adaptive minus global blend: 0.0


In [9]:
import numpy as np
import pandas as pd

from sklearn.metrics import balanced_accuracy_score

# ============================================================
# EXPERIMENT D3: CLASS-WISE SCORE CALIBRATION
# ============================================================

QWEN_WEIGHT = 0.20

labels_arr = np.asarray(labels, dtype=str)
dev_y_arr = np.asarray(dev_y, dtype=str)
holdout_y_arr = np.asarray(holdout_y, dtype=str)

# Use the same global Qwen weight for both datasets.
dev_scores = (
    (1 - QWEN_WEIGHT) * dev_baseline_oof
    + QWEN_WEIGHT * dev_qwen_oof
)

holdout_scores = (
    (1 - QWEN_WEIGHT) * holdout_baseline
    + QWEN_WEIGHT * holdout_qwen
)

n_classes = len(labels_arr)

assert dev_scores.shape[1] == n_classes
assert holdout_scores.shape[1] == n_classes


def predict_with_offsets(scores, offsets):
    return labels_arr[np.argmax(scores + offsets, axis=1)]


# ------------------------------------------------------------
# 1. Optimize class offsets on development OOF predictions
# ------------------------------------------------------------

offsets = np.zeros(n_classes, dtype=float)

# Restrict adjustments to small values to limit overfitting.
candidate_offsets = np.round(
    np.arange(-0.06, 0.061, 0.01), 3
)

# Mild regularization discourages unnecessarily large adjustments.
REGULARIZATION = 0.05


def calibration_objective(scores, y_true, candidate_biases):
    predictions = predict_with_offsets(scores, candidate_biases)

    accuracy = balanced_accuracy_score(y_true, predictions)

    penalty = REGULARIZATION * np.mean(candidate_biases ** 2)

    return accuracy - penalty


initial_predictions = predict_with_offsets(
    dev_scores, offsets
)

initial_dev_score = balanced_accuracy_score(
    dev_y_arr, initial_predictions
)

# Coordinate search: adjust one model's score at a time.
for iteration in range(4):
    changes = 0

    for class_idx in range(n_classes):
        best_offset = offsets[class_idx]

        best_objective = calibration_objective(
            dev_scores, dev_y_arr, offsets
        )

        for candidate in candidate_offsets:
            trial_offsets = offsets.copy()
            trial_offsets[class_idx] = candidate

            objective = calibration_objective(
                dev_scores, dev_y_arr, trial_offsets
            )

            if objective > best_objective + 1e-12:
                best_objective = objective
                best_offset = candidate

        if best_offset != offsets[class_idx]:
            offsets[class_idx] = best_offset
            changes += 1

    print(
        f"Pass {iteration + 1}: "
        f"{changes} model offsets changed"
    )

    if changes == 0:
        break


# ------------------------------------------------------------
# 2. Compare development scores
# ------------------------------------------------------------

dev_calibrated_pred = predict_with_offsets(
    dev_scores, offsets
)

dev_calibrated_score = balanced_accuracy_score(
    dev_y_arr, dev_calibrated_pred
)

print("\nDEVELOPMENT RESULTS")
print(f"Before calibration: {initial_dev_score:.4f}")
print(f"After calibration:  {dev_calibrated_score:.4f}")


# ------------------------------------------------------------
# 3. Evaluate on untouched holdout prompts
# ------------------------------------------------------------

holdout_baseline_pred = labels_arr[
    np.argmax(holdout_baseline, axis=1)
]

holdout_global_pred = labels_arr[
    np.argmax(holdout_scores, axis=1)
]

holdout_calibrated_pred = predict_with_offsets(
    holdout_scores, offsets
)

baseline_score = balanced_accuracy_score(
    holdout_y_arr, holdout_baseline_pred
)

global_score = balanced_accuracy_score(
    holdout_y_arr, holdout_global_pred
)

calibrated_score = balanced_accuracy_score(
    holdout_y_arr, holdout_calibrated_pred
)

results = pd.DataFrame([
    {
        "Method": "Text baseline",
        "Holdout balanced accuracy": baseline_score,
    },
    {
        "Method": "Global 20% Qwen blend",
        "Holdout balanced accuracy": global_score,
    },
    {
        "Method": "Qwen blend + class calibration",
        "Holdout balanced accuracy": calibrated_score,
    },
])

results["Delta vs global blend"] = (
    results["Holdout balanced accuracy"] - global_score
)

print("\nHELD-OUT RESULTS")
display(results.round(4))


# ------------------------------------------------------------
# 4. Inspect learned offsets
# ------------------------------------------------------------

offset_table = pd.DataFrame({
    "Model": labels_arr,
    "Family": [get_family(m) for m in labels_arr],
    "Learned score offset": offsets,
})

print("\nLEARNED MODEL OFFSETS")
display(
    offset_table.sort_values(
        "Learned score offset",
        ascending=False
    ).reset_index(drop=True).round(3)
)

print(
    "\nHoldout predictions changed by calibration:",
    int(np.sum(holdout_global_pred != holdout_calibrated_pred))
)

Pass 1: 11 model offsets changed
Pass 2: 4 model offsets changed
Pass 3: 0 model offsets changed

DEVELOPMENT RESULTS
Before calibration: 0.3695
After calibration:  0.3839

HELD-OUT RESULTS


,Method,Holdout balanced accuracy,Delta vs global blend
0,Text baseline,0.3574,-0.0148
1,Global 20% Qwen blend,0.3722,0.0000
2,Qwen blend + class calibration,0.3833,0.0111



LEARNED MODEL OFFSETS


,Model,Family,Learned score offset
0,claude-opus-4-8,Anthropic,0.06
1,claude-opus-4-6,Anthropic,0.04
2,gemini-3.7-flash,Google,0.01
3,claude-fable-5,Anthropic,0.00
4,gemini-3.6-flash,Google,0.00
5,gemini-3.8-flash,Google,0.00
6,gpt-5.6-terra,OpenAI,0.00
7,kimi-k3,Moonshot,0.00
8,gpt-5.6-luna,OpenAI,0.00
9,glm-5.3-flash,ZAI,0.00



Holdout predictions changed by calibration: 74


In [10]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score

# ============================================================
# EXPERIMENT D4: OUTER GROUPED CV FOR CLASS CALIBRATION
# ============================================================

N_OUTER = 5
QWEN_WEIGHT = 0.20

labels_arr = np.asarray(labels, dtype=str)
y_arr = np.asarray(y, dtype=str)
groups_arr = np.asarray(groups)

n_classes = len(labels_arr)

assert len(y_arr) == len(groups_arr) == len(X)
assert qwen_train_last.shape[0] == len(y_arr)

# Keep the same calibration search as D3.
candidate_offsets = np.round(
    np.arange(-0.06, 0.061, 0.01), 3
)
REGULARIZATION = 0.05


def predict_offsets(scores, offsets):
    return labels_arr[np.argmax(scores + offsets, axis=1)]


def calibration_objective(scores, targets, offsets):
    predictions = predict_offsets(scores, offsets)

    score = balanced_accuracy_score(targets, predictions)
    penalty = REGULARIZATION * np.mean(offsets ** 2)

    return score - penalty


def learn_class_offsets(scores, targets):
    offsets = np.zeros(n_classes, dtype=float)

    for _ in range(4):
        changes = 0

        for class_idx in range(n_classes):
            best_offset = offsets[class_idx]
            best_objective = calibration_objective(
                scores, targets, offsets
            )

            for candidate in candidate_offsets:
                trial = offsets.copy()
                trial[class_idx] = candidate

                objective = calibration_objective(
                    scores, targets, trial
                )

                if objective > best_objective + 1e-12:
                    best_objective = objective
                    best_offset = candidate

            if best_offset != offsets[class_idx]:
                offsets[class_idx] = best_offset
                changes += 1

        if changes == 0:
            break

    return offsets


# Out-of-fold prediction arrays cover every training row.
oof_baseline = np.zeros((len(y_arr), n_classes))
oof_qwen = np.zeros((len(y_arr), n_classes))
oof_calibrated = np.zeros((len(y_arr), n_classes))

outer_cv = GroupKFold(n_splits=N_OUTER)
outer_results = []

for fold, (outer_train, outer_valid) in enumerate(
    outer_cv.split(np.zeros(len(y_arr)), y_arr, groups_arr),
    start=1
):
    print(f"\n========== OUTER FOLD {fold}/{N_OUTER} ==========")

    # --------------------------------------------------------
    # A. Generate inner OOF predictions from outer-train only.
    # These are used to learn class-specific offsets.
    # --------------------------------------------------------

    inner_cv = GroupKFold(n_splits=5)

    inner_baseline = np.zeros((len(outer_train), n_classes))
    inner_qwen = np.zeros((len(outer_train), n_classes))

    outer_train_y = y_arr[outer_train]
    outer_train_groups = groups_arr[outer_train]

    for inner_fold, (inner_train_local, inner_valid_local) in enumerate(
        inner_cv.split(
            np.zeros(len(outer_train)),
            outer_train_y,
            outer_train_groups
        ),
        start=1
    ):
        inner_train_global = outer_train[inner_train_local]
        inner_valid_global = outer_train[inner_valid_local]

        b_scores, q_scores = fit_predict_pair(
            inner_train_global,
            inner_valid_global
        )

        inner_baseline[inner_valid_local] = b_scores
        inner_qwen[inner_valid_local] = q_scores

        print(f"  Inner fold {inner_fold}/5 complete")

    # Learn offsets without seeing outer-validation labels.
    inner_combined = (
        (1 - QWEN_WEIGHT) * inner_baseline
        + QWEN_WEIGHT * inner_qwen
    )

    offsets = learn_class_offsets(
        inner_combined,
        outer_train_y
    )

    # --------------------------------------------------------
    # B. Train on all outer-train prompts and predict held-out.
    # --------------------------------------------------------

    outer_baseline, outer_qwen_scores = fit_predict_pair(
        outer_train,
        outer_valid
    )

    global_scores = (
        (1 - QWEN_WEIGHT) * outer_baseline
        + QWEN_WEIGHT * outer_qwen_scores
    )

    calibrated_scores = global_scores + offsets

    oof_baseline[outer_valid] = outer_baseline
    oof_qwen[outer_valid] = outer_qwen_scores
    oof_calibrated[outer_valid] = calibrated_scores

    # --------------------------------------------------------
    # C. Evaluate outer-validation prompts only.
    # --------------------------------------------------------

    targets = y_arr[outer_valid]

    pred_baseline = labels_arr[
        np.argmax(outer_baseline, axis=1)
    ]

    pred_global = labels_arr[
        np.argmax(global_scores, axis=1)
    ]

    pred_calibrated = labels_arr[
        np.argmax(calibrated_scores, axis=1)
    ]

    score_baseline = balanced_accuracy_score(
        targets, pred_baseline
    )

    score_global = balanced_accuracy_score(
        targets, pred_global
    )

    score_calibrated = balanced_accuracy_score(
        targets, pred_calibrated
    )

    outer_results.append({
        "Fold": fold,
        "Baseline": score_baseline,
        "Global Qwen blend": score_global,
        "Calibrated blend": score_calibrated,
        "Calibration delta": score_calibrated - score_global,
        "Changed by calibration": int(
            np.sum(pred_global != pred_calibrated)
        )
    })

    print(f"Baseline:        {score_baseline:.4f}")
    print(f"Global Qwen:     {score_global:.4f}")
    print(f"Calibrated:      {score_calibrated:.4f}")
    print(f"Delta:           {score_calibrated - score_global:+.4f}")


# ============================================================
# D. FINAL OUT-OF-FOLD SUMMARY
# ============================================================

results = pd.DataFrame(outer_results)

print("\n========== OUTER FOLD RESULTS ==========")
display(results.round(4))

pred_baseline = labels_arr[np.argmax(oof_baseline, axis=1)]

pred_global = labels_arr[
    np.argmax(
        (1 - QWEN_WEIGHT) * oof_baseline
        + QWEN_WEIGHT * oof_qwen,
        axis=1
    )
]

pred_calibrated = labels_arr[np.argmax(oof_calibrated, axis=1)]

summary = pd.DataFrame([
    {
        "Method": "Text baseline",
        "Pooled OOF balanced accuracy":
            balanced_accuracy_score(y_arr, pred_baseline)
    },
    {
        "Method": "Global 20% Qwen blend",
        "Pooled OOF balanced accuracy":
            balanced_accuracy_score(y_arr, pred_global)
    },
    {
        "Method": "Calibrated Qwen blend",
        "Pooled OOF balanced accuracy":
            balanced_accuracy_score(y_arr, pred_calibrated)
    }
])

print("\n========== POOLED OUTER OOF RESULTS ==========")
display(summary.round(4))

print("\nMean fold scores:")
display(
    results[[
        "Baseline", "Global Qwen blend", "Calibrated blend"
    ]].agg(["mean", "std"]).round(4)
)


========== OUTER FOLD 1/5 ==========
  Inner fold 1/5 complete
  Inner fold 2/5 complete
  Inner fold 3/5 complete
  Inner fold 4/5 complete
  Inner fold 5/5 complete
Baseline:        0.3889
Global Qwen:     0.3870
Calibrated:      0.3852
Delta:           -0.0019

========== OUTER FOLD 2/5 ==========
  Inner fold 1/5 complete
  Inner fold 2/5 complete
  Inner fold 3/5 complete
  Inner fold 4/5 complete
  Inner fold 5/5 complete
Baseline:        0.3463
Global Qwen:     0.3796
Calibrated:      0.3704
Delta:           -0.0093

========== OUTER FOLD 3/5 ==========
  Inner fold 1/5 complete
  Inner fold 2/5 complete
  Inner fold 3/5 complete
  Inner fold 4/5 complete
  Inner fold 5/5 complete
Baseline:        0.3815
Global Qwen:     0.4167
Calibrated:      0.3944
Delta:           -0.0222

========== OUTER FOLD 4/5 ==========
  Inner fold 1/5 complete
  Inner fold 2/5 complete
  Inner fold 3/5 complete
  Inner fold 4/5 complete
  Inner fold 5/5 complete
Baseline:        0.3796
Global Qwen: 

,Fold,Baseline,Global Qwen blend,Calibrated blend,Calibration delta,Changed by calibration
0,1,0.3889,0.3870,0.3852,-0.0019,54
1,2,0.3463,0.3796,0.3704,-0.0093,76
2,3,0.3815,0.4167,0.3944,-0.0222,81
3,4,0.3796,0.4037,0.3963,-0.0074,68
4,5,0.3521,0.3595,0.3559,-0.0036,73



========== POOLED OUTER OOF RESULTS ==========


,Method,Pooled OOF balanced accuracy
0,Text baseline,0.3697
1,Global 20% Qwen blend,0.3893
2,Calibrated Qwen blend,0.3804



Mean fold scores:


,Baseline,Global Qwen blend,Calibrated blend
mean,0.3697,0.3893,0.3804
std,0.0191,0.0220,0.0171


In [11]:
import numpy as np
import pandas as pd

from scipy.stats import rankdata
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score

# ============================================================
# EXPERIMENT E: QWEN REPRESENTATION ABLATION
# ============================================================

labels_arr = np.asarray(labels, dtype=str)
y_arr = np.asarray(y, dtype=str)
groups_arr = np.asarray(groups)

N_CLASSES = len(labels_arr)
N_FOLDS = 5

if "oof_baseline" not in globals():
    raise RuntimeError("Run Experiment D4 first to create oof_baseline.")

assert oof_baseline.shape == (len(y_arr), N_CLASSES)
assert qwen_train_last.shape[0] == len(y_arr)
assert qwen_train_mean.shape[0] == len(y_arr)

# Fixed global Qwen weights to compare.
QWEN_WEIGHTS = [0.10, 0.15, 0.20, 0.25, 0.30]

# Each row is normalized before feature extraction.
representations = {
    "Last token": qwen_train_last,
    "Mean pooling": qwen_train_mean,
    "Concatenated": np.concatenate(
        [qwen_train_last, qwen_train_mean],
        axis=1
    ),
}

# Convert model scores to comparable within-row ranks.
def aligned_ranks(scores, estimator_classes):
    scores = np.asarray(scores)

    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])

    aligned = np.full(
        (len(scores), N_CLASSES),
        -1e12,
        dtype=np.float64
    )

    label_to_col = {
        label: idx for idx, label in enumerate(labels_arr)
    }

    for j, label in enumerate(estimator_classes):
        aligned[:, label_to_col[str(label)]] = scores[:, j]

    ranks = rankdata(aligned, axis=1, method="average")

    return (ranks - 1) / max(N_CLASSES - 1, 1)


# The baseline OOF predictions come from the same 5-fold
# GroupKFold scheme used to generate these Qwen predictions.
qwen_oof_variants = {
    name: np.zeros((len(y_arr), N_CLASSES))
    for name in representations
}

cv = GroupKFold(n_splits=N_FOLDS)

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(np.zeros(len(y_arr)), y_arr, groups_arr),
    start=1
):
    print(f"Fold {fold}/{N_FOLDS}")

    for name, features in representations.items():
        clf = LinearSVC(C=1.0, max_iter=10000)

        clf.fit(features[train_idx], y_arr[train_idx])

        scores = clf.decision_function(features[valid_idx])

        qwen_oof_variants[name][valid_idx] = aligned_ranks(
            scores, clf.classes_
        )

# Also evaluate the existing average of last-token and mean ranks.
qwen_oof_variants["Last + mean rank blend"] = (
    0.5 * qwen_oof_variants["Last token"]
    + 0.5 * qwen_oof_variants["Mean pooling"]
)

# ------------------------------------------------------------
# Score every representation and its text/Qwen blends
# ------------------------------------------------------------

baseline_pred = labels_arr[np.argmax(oof_baseline, axis=1)]

baseline_pooled = balanced_accuracy_score(y_arr, baseline_pred)

fold_splits = list(
    GroupKFold(n_splits=N_FOLDS).split(
        np.zeros(len(y_arr)), y_arr, groups_arr
    )
)

results = []

for name, qwen_scores in qwen_oof_variants.items():
    qwen_pred = labels_arr[np.argmax(qwen_scores, axis=1)]

    qwen_pooled = balanced_accuracy_score(y_arr, qwen_pred)

    results.append({
        "Representation": name,
        "Qwen weight": 1.0,
        "Pooled OOF": qwen_pooled,
        "Mean fold score": np.mean([
            balanced_accuracy_score(
                y_arr[val_idx],
                qwen_pred[val_idx]
            )
            for _, val_idx in fold_splits
        ]),
        "Delta vs text baseline": qwen_pooled - baseline_pooled,
    })

    for weight in QWEN_WEIGHTS:
        combined = (
            (1 - weight) * oof_baseline
            + weight * qwen_scores
        )

        pred = labels_arr[np.argmax(combined, axis=1)]

        pooled = balanced_accuracy_score(y_arr, pred)

        fold_scores = [
            balanced_accuracy_score(
                y_arr[val_idx],
                pred[val_idx]
            )
            for _, val_idx in fold_splits
        ]

        results.append({
            "Representation": name,
            "Qwen weight": weight,
            "Pooled OOF": pooled,
            "Mean fold score": np.mean(fold_scores),
            "Delta vs text baseline": pooled - baseline_pooled,
        })

results_df = pd.DataFrame(results)

print(f"\nText baseline pooled OOF: {baseline_pooled:.4f}")

print("\nTOP CONFIGURATIONS")
display(
    results_df.sort_values(
        "Pooled OOF",
        ascending=False
    ).head(15).reset_index(drop=True).round(4)
)

print("\nBEST CONFIGURATION PER REPRESENTATION")
display(
    results_df.loc[
        results_df.groupby("Representation")["Pooled OOF"].idxmax()
    ].sort_values("Pooled OOF", ascending=False).reset_index(drop=True).round(4)
)

Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5

Text baseline pooled OOF: 0.3697

TOP CONFIGURATIONS


,Representation,Qwen weight,Pooled OOF,Mean fold score,Delta vs text baseline
0,Concatenated,0.30,0.3926,0.3927,0.0230
1,Concatenated,0.25,0.3919,0.3919,0.0222
2,Concatenated,0.20,0.3893,0.3893,0.0196
3,Last + mean rank blend,0.20,0.3893,0.3893,0.0196
4,Concatenated,0.15,0.3886,0.3886,0.0189
5,Last + mean rank blend,0.25,0.3882,0.3882,0.0185
6,Last token,0.20,0.3874,0.3875,0.0178
7,Last token,0.25,0.3874,0.3875,0.0178
8,Mean pooling,0.20,0.3867,0.3867,0.0170
9,Last + mean rank blend,0.30,0.3860,0.3860,0.0163



BEST CONFIGURATION PER REPRESENTATION


,Representation,Qwen weight,Pooled OOF,Mean fold score,Delta vs text baseline
0,Concatenated,0.3,0.3926,0.3927,0.0230
1,Last + mean rank blend,0.2,0.3893,0.3893,0.0196
2,Last token,0.2,0.3874,0.3875,0.0178
3,Mean pooling,0.2,0.3867,0.3867,0.0170


In [12]:
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix
from scipy.stats import rankdata

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score

# ============================================================
# EXPERIMENT F: NESTED CV FOR QWEN REPRESENTATION SELECTION
# ============================================================

X = pd.Series(X).reset_index(drop=True)
y_arr = np.asarray(y, dtype=str)
groups_arr = np.asarray(groups)
labels_arr = np.asarray(labels, dtype=str)

N_CLASSES = len(labels_arr)
label_to_col = {label: i for i, label in enumerate(labels_arr)}

assert len(X) == len(y_arr) == len(groups_arr)
assert qwen_train_last.shape[0] == len(X)
assert qwen_train_mean.shape[0] == len(X)

# Qwen features are frozen and precomputed.
qwen_features = {
    "Last token": qwen_train_last,
    "Mean pooling": qwen_train_mean,
    "Concatenated": np.concatenate(
        [qwen_train_last, qwen_train_mean], axis=1
    ),
}

WEIGHTS = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]
N_OUTER = 5
N_INNER = 4


def score_to_ranks(scores, classes):
    scores = np.asarray(scores)

    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])

    aligned = np.full(
        (len(scores), N_CLASSES), -1e12, dtype=float
    )

    for j, label in enumerate(classes):
        aligned[:, label_to_col[str(label)]] = scores[:, j]

    ranks = rankdata(aligned, axis=1, method="average")
    return (ranks - 1) / max(N_CLASSES - 1, 1)


def probability_to_ranks(probabilities):
    ranks = rankdata(
        probabilities, axis=1, method="average"
    )
    return (ranks - 1) / max(N_CLASSES - 1, 1)


def align_proba(estimator, probabilities):
    aligned = np.zeros(
        (len(probabilities), N_CLASSES), dtype=float
    )

    for j, label in enumerate(estimator.classes_):
        aligned[:, label_to_col[str(label)]] = probabilities[:, j]

    return aligned


def geometric_blend(probabilities, weights):
    logp = np.zeros_like(probabilities[0], dtype=float)

    for p, weight in zip(probabilities, weights):
        logp += weight * np.log(np.clip(p, 1e-12, 1.0))

    logp -= logp.max(axis=1, keepdims=True)
    result = np.exp(logp)

    return result / result.sum(axis=1, keepdims=True)


def fit_all_views(train_idx, valid_idx):
    """Fit models only on train_idx and predict valid_idx."""

    X_train = X.iloc[train_idx].tolist()
    X_valid = X.iloc[valid_idx].tolist()
    y_train = y_arr[train_idx]

    # Word TF-IDF
    word_vec = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=25000,
        sublinear_tf=True
    )

    Xw_train = word_vec.fit_transform(X_train)
    Xw_valid = word_vec.transform(X_valid)

    # Character TF-IDF
    char_vec = TfidfVectorizer(
        analyzer="char",
        ngram_range=(1, 5),
        max_features=100000,
        sublinear_tf=True
    )

    Xc_train = char_vec.fit_transform(X_train)
    Xc_valid = char_vec.transform(X_valid)

    # Style features
    style_train_raw = make_style_matrix(X_train)
    style_valid_raw = make_style_matrix(X_valid)

    scaler = StandardScaler()
    style_train = scaler.fit_transform(style_train_raw)
    style_valid = scaler.transform(style_valid_raw)

    # Logistic regression views
    word_lr = LogisticRegression(C=1.0, max_iter=1500)
    char_lr = LogisticRegression(C=1.0, max_iter=1500)
    style_lr = LogisticRegression(C=1.0, max_iter=1500)

    word_lr.fit(Xw_train, y_train)
    char_lr.fit(Xc_train, y_train)
    style_lr.fit(style_train, y_train)

    word_p = align_proba(
        word_lr, word_lr.predict_proba(Xw_valid)
    )
    char_p = align_proba(
        char_lr, char_lr.predict_proba(Xc_valid)
    )
    style_p = align_proba(
        style_lr, style_lr.predict_proba(style_valid)
    )

    logistic_p = geometric_blend(
        [word_p, char_p, style_p],
        [0.40, 0.50, 0.10]
    )

    logistic_ranks = probability_to_ranks(logistic_p)

    # Multi-view SVM
    svm_train = hstack([
        Xw_train,
        Xc_train,
        csr_matrix(0.1 * style_train)
    ], format="csr")

    svm_valid = hstack([
        Xw_valid,
        Xc_valid,
        csr_matrix(0.1 * style_valid)
    ], format="csr")

    svm = LinearSVC(C=1.5, max_iter=10000)
    svm.fit(svm_train, y_train)

    svm_ranks = score_to_ranks(
        svm.decision_function(svm_valid),
        svm.classes_
    )

    baseline = 0.70 * svm_ranks + 0.30 * logistic_ranks

    # Qwen classifiers for each representation
    qwen_ranks = {}

    for name, features in qwen_features.items():
        clf = LinearSVC(C=1.0, max_iter=10000)
        clf.fit(features[train_idx], y_train)

        qwen_ranks[name] = score_to_ranks(
            clf.decision_function(features[valid_idx]),
            clf.classes_
        )

    # Alternative: average last-token and mean-pooled rankings
    qwen_ranks["Last + mean rank blend"] = (
        0.50 * qwen_ranks["Last token"]
        + 0.50 * qwen_ranks["Mean pooling"]
    )

    return {"Baseline": baseline, **qwen_ranks}


def combine_view(outputs, representation, weight):
    if weight == 0:
        return outputs["Baseline"]

    return (
        (1.0 - weight) * outputs["Baseline"]
        + weight * outputs[representation]
    )


# ============================================================
# OUTER CV: INNER CV SELECTS, OUTER FOLD EVALUATES
# ============================================================

outer_cv = GroupKFold(n_splits=N_OUTER)

oof_baseline = np.zeros((len(X), N_CLASSES))
oof_selected = np.zeros((len(X), N_CLASSES))

fold_results = []

for outer_fold, (outer_train, outer_valid) in enumerate(
    outer_cv.split(np.zeros(len(X)), y_arr, groups_arr),
    start=1
):
    print(f"\n========== OUTER FOLD {outer_fold}/{N_OUTER} ==========")

    inner_cv = GroupKFold(n_splits=N_INNER)

    # Inner OOF arrays: only outer-training prompts are used.
    inner_oof = {
        "Baseline": np.zeros((len(outer_train), N_CLASSES)),
        **{
            name: np.zeros((len(outer_train), N_CLASSES))
            for name in [
                "Last token",
                "Mean pooling",
                "Concatenated",
                "Last + mean rank blend"
            ]
        }
    }

    for inner_fold, (inner_train_local, inner_valid_local) in enumerate(
        inner_cv.split(
            np.zeros(len(outer_train)),
            y_arr[outer_train],
            groups_arr[outer_train]
        ),
        start=1
    ):
        inner_train = outer_train[inner_train_local]
        inner_valid = outer_train[inner_valid_local]

        predictions = fit_all_views(inner_train, inner_valid)

        for name in inner_oof:
            inner_oof[name][inner_valid_local] = predictions[name]

        print(f"  Inner fold {inner_fold}/{N_INNER} complete")

    # Select representation and weight using inner OOF only.
    inner_y = y_arr[outer_train]

    best_score = balanced_accuracy_score(
        inner_y,
        labels_arr[np.argmax(inner_oof["Baseline"], axis=1)]
    )

    best_representation = "Baseline"
    best_weight = 0.0

    # Include the unmodified baseline as a valid candidate.
    candidates = [
        (name, weight)
        for name in [
            "Last token",
            "Mean pooling",
            "Concatenated",
            "Last + mean rank blend"
        ]
        for weight in WEIGHTS
    ]

    for representation, weight in candidates:
        candidate_scores = (
            (1.0 - weight) * inner_oof["Baseline"]
            + weight * inner_oof[representation]
        )

        candidate_pred = labels_arr[
            np.argmax(candidate_scores, axis=1)
        ]

        score = balanced_accuracy_score(inner_y, candidate_pred)

        if score > best_score + 1e-12:
            best_score = score
            best_representation = representation
            best_weight = weight

    print(
        f"Selected: {best_representation}, "
        f"Qwen weight={best_weight:.2f}, "
        f"inner OOF={best_score:.4f}"
    )

    # Fit on all outer-training prompts and evaluate outer holdout.
    outer_predictions = fit_all_views(outer_train, outer_valid)

    baseline_outer = outer_predictions["Baseline"]

    selected_outer = combine_view(
        outer_predictions,
        best_representation,
        best_weight
    )

    oof_baseline[outer_valid] = baseline_outer
    oof_selected[outer_valid] = selected_outer

    baseline_pred = labels_arr[np.argmax(baseline_outer, axis=1)]
    selected_pred = labels_arr[np.argmax(selected_outer, axis=1)]

    baseline_score = balanced_accuracy_score(
        y_arr[outer_valid], baseline_pred
    )

    selected_score = balanced_accuracy_score(
        y_arr[outer_valid], selected_pred
    )

    fold_results.append({
        "Outer fold": outer_fold,
        "Selected representation": best_representation,
        "Selected Qwen weight": best_weight,
        "Inner OOF score": best_score,
        "Outer baseline score": baseline_score,
        "Outer selected score": selected_score,
        "Outer delta": selected_score - baseline_score
    })

    print(f"Outer baseline: {baseline_score:.4f}")
    print(f"Outer selected: {selected_score:.4f}")
    print(f"Outer delta:    {selected_score - baseline_score:+.4f}")


# ============================================================
# FINAL RESULTS
# ============================================================

fold_results = pd.DataFrame(fold_results)

print("\n========== NESTED CV RESULTS ==========")
display(fold_results.round(4))

baseline_oof_score = balanced_accuracy_score(
    y_arr, labels_arr[np.argmax(oof_baseline, axis=1)]
)

selected_oof_score = balanced_accuracy_score(
    y_arr, labels_arr[np.argmax(oof_selected, axis=1)]
)

print("\nPooled outer OOF baseline:", round(baseline_oof_score, 4))
print("Pooled outer OOF selected:", round(selected_oof_score, 4))
print(
    "Improvement:",
    round(selected_oof_score - baseline_oof_score, 4)
)

print(
    "Outer-fold mean baseline:",
    round(fold_results["Outer baseline score"].mean(), 4)
)

print(
    "Outer-fold mean selected:",
    round(fold_results["Outer selected score"].mean(), 4)
)


========== OUTER FOLD 1/5 ==========
  Inner fold 1/4 complete
  Inner fold 2/4 complete
  Inner fold 3/4 complete
  Inner fold 4/4 complete
Selected: Last + mean rank blend, Qwen weight=0.20, inner OOF=0.3700
Outer baseline: 0.3889
Outer selected: 0.3870
Outer delta:    -0.0019

========== OUTER FOLD 2/5 ==========
  Inner fold 1/4 complete
  Inner fold 2/4 complete
  Inner fold 3/4 complete
  Inner fold 4/4 complete
Selected: Concatenated, Qwen weight=0.25, inner OOF=0.3801
Outer baseline: 0.3463
Outer selected: 0.3870
Outer delta:    +0.0407

========== OUTER FOLD 3/5 ==========
  Inner fold 1/4 complete
  Inner fold 2/4 complete
  Inner fold 3/4 complete
  Inner fold 4/4 complete
Selected: Concatenated, Qwen weight=0.25, inner OOF=0.3686
Outer baseline: 0.3815
Outer selected: 0.4204
Outer delta:    +0.0389

========== OUTER FOLD 4/5 ==========
  Inner fold 1/4 complete
  Inner fold 2/4 complete
  Inner fold 3/4 complete
  Inner fold 4/4 complete
Selected: Concatenated, Qwen weight

,Outer fold,Selected representation,Selected Qwen weight,Inner OOF score,Outer baseline score,Outer selected score,Outer delta
0,1,Last + mean rank blend,0.20,0.3700,0.3889,0.3870,-0.0019
1,2,Concatenated,0.25,0.3801,0.3463,0.3870,0.0407
2,3,Concatenated,0.25,0.3686,0.3815,0.4204,0.0389
3,4,Concatenated,0.15,0.3727,0.3796,0.4000,0.0204
4,5,Mean pooling,0.20,0.3819,0.3521,0.3521,-0.0000



Pooled outer OOF baseline: 0.3697
Pooled outer OOF selected: 0.3893
Improvement: 0.0196
Outer-fold mean baseline: 0.3697
Outer-fold mean selected: 0.3893


In [13]:
import numpy as np
import pandas as pd

from scipy.stats import rankdata
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score

# ============================================================
# EXPERIMENT G: TUNE LINEAR SVM C FOR CONCATENATED QWEN FEATURES
# Reuses fit_all_views() and qwen_features from Experiment F.
# ============================================================

X = pd.Series(X).reset_index(drop=True)
y_arr = np.asarray(y, dtype=str)
groups_arr = np.asarray(groups)
labels_arr = np.asarray(labels, dtype=str)

N_CLASSES = len(labels_arr)
label_to_col = {
    label: i for i, label in enumerate(labels_arr)
}

features = qwen_features["Concatenated"]

assert len(X) == len(y_arr) == len(groups_arr)
assert features.shape[0] == len(y_arr)

C_VALUES = [0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
QWEN_WEIGHTS = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]


def qwen_scores_to_ranks(scores, classes):
    scores = np.asarray(scores)

    aligned = np.full(
        (len(scores), N_CLASSES),
        -1e12,
        dtype=float
    )

    for j, label in enumerate(classes):
        aligned[:, label_to_col[str(label)]] = scores[:, j]

    ranks = rankdata(
        aligned,
        axis=1,
        method="average"
    )

    return (ranks - 1) / max(N_CLASSES - 1, 1)


def predict_labels(scores):
    return labels_arr[np.argmax(scores, axis=1)]


# ============================================================
# OUTER CV: estimate generalization of the selection procedure
# ============================================================

outer_cv = GroupKFold(n_splits=5)

oof_baseline_g = np.zeros((len(X), N_CLASSES))
oof_selected_g = np.zeros((len(X), N_CLASSES))

fold_results_g = []

for outer_fold, (outer_train, outer_valid) in enumerate(
    outer_cv.split(np.zeros(len(X)), y_arr, groups_arr),
    start=1
):
    print(f"\n========== OUTER FOLD {outer_fold}/5 ==========")

    inner_cv = GroupKFold(n_splits=4)

    inner_baseline = np.zeros(
        (len(outer_train), N_CLASSES)
    )

    inner_qwen = {
        C: np.zeros((len(outer_train), N_CLASSES))
        for C in C_VALUES
    }

    # --------------------------------------------------------
    # 1. Produce inner OOF predictions on outer-training prompts
    # --------------------------------------------------------

    for inner_fold, (tr_local, va_local) in enumerate(
        inner_cv.split(
            np.zeros(len(outer_train)),
            y_arr[outer_train],
            groups_arr[outer_train]
        ),
        start=1
    ):
        tr_global = outer_train[tr_local]
        va_global = outer_train[va_local]

        # Existing text baseline.
        outputs = fit_all_views(tr_global, va_global)

        inner_baseline[va_local] = outputs["Baseline"]

        # Qwen classifiers at different C values.
        for C in C_VALUES:
            clf = LinearSVC(C=C, max_iter=10000)

            clf.fit(
                features[tr_global],
                y_arr[tr_global]
            )

            scores = clf.decision_function(
                features[va_global]
            )

            inner_qwen[C][va_local] = qwen_scores_to_ranks(
                scores,
                clf.classes_
            )

        print(f"Inner fold {inner_fold}/4 complete")

    # --------------------------------------------------------
    # 2. Select C and Qwen weight using inner OOF only
    # --------------------------------------------------------

    inner_y = y_arr[outer_train]

    best_score = balanced_accuracy_score(
        inner_y,
        predict_labels(inner_baseline)
    )

    best_C = None
    best_weight = 0.0

    # Baseline is the initial candidate.
    for C in C_VALUES:
        for weight in QWEN_WEIGHTS:
            combined = (
                (1 - weight) * inner_baseline
                + weight * inner_qwen[C]
            )

            score = balanced_accuracy_score(
                inner_y,
                predict_labels(combined)
            )

            if score > best_score + 1e-12:
                best_score = score
                best_C = C
                best_weight = weight

    if best_C is None:
        print("Selected: text baseline; no Qwen blend improved inner CV")
    else:
        print(
            f"Selected C={best_C}, Qwen weight={best_weight:.2f}, "
            f"inner OOF={best_score:.4f}"
        )

    # --------------------------------------------------------
    # 3. Evaluate the chosen configuration on outer holdout
    # --------------------------------------------------------

    outputs_outer = fit_all_views(
        outer_train,
        outer_valid
    )

    baseline_outer = outputs_outer["Baseline"]

    if best_C is None:
        selected_outer = baseline_outer.copy()
    else:
        clf_outer = LinearSVC(
            C=best_C,
            max_iter=10000
        )

        clf_outer.fit(
            features[outer_train],
            y_arr[outer_train]
        )

        qwen_outer = qwen_scores_to_ranks(
            clf_outer.decision_function(
                features[outer_valid]
            ),
            clf_outer.classes_
        )

        selected_outer = (
            (1 - best_weight) * baseline_outer
            + best_weight * qwen_outer
        )

    oof_baseline_g[outer_valid] = baseline_outer
    oof_selected_g[outer_valid] = selected_outer

    y_valid = y_arr[outer_valid]

    baseline_score = balanced_accuracy_score(
        y_valid,
        predict_labels(baseline_outer)
    )

    selected_score = balanced_accuracy_score(
        y_valid,
        predict_labels(selected_outer)
    )

    fold_results_g.append({
        "Fold": outer_fold,
        "Selected C": best_C,
        "Selected Qwen weight": best_weight,
        "Inner OOF score": best_score,
        "Outer baseline": baseline_score,
        "Outer selected": selected_score,
        "Delta": selected_score - baseline_score
    })

    print(f"Outer baseline: {baseline_score:.4f}")
    print(f"Outer selected: {selected_score:.4f}")
    print(f"Delta: {selected_score - baseline_score:+.4f}")


# ============================================================
# 4. Summarize results
# ============================================================

results_g = pd.DataFrame(fold_results_g)

print("\n========== EXPERIMENT G: FOLD RESULTS ==========")
display(results_g.round(4))

baseline_oof_score = balanced_accuracy_score(
    y_arr,
    predict_labels(oof_baseline_g)
)

selected_oof_score = balanced_accuracy_score(
    y_arr,
    predict_labels(oof_selected_g)
)

summary_g = pd.DataFrame([
    {
        "Method": "Text baseline",
        "Pooled OOF balanced accuracy": baseline_oof_score
    },
    {
        "Method": "Nested C + weight selection",
        "Pooled OOF balanced accuracy": selected_oof_score
    }
])

print("\n========== POOLED OOF RESULTS ==========")
display(summary_g.round(4))

print(
    "\nGain over baseline:",
    round(selected_oof_score - baseline_oof_score, 4)
)


========== OUTER FOLD 1/5 ==========
Inner fold 1/4 complete
Inner fold 2/4 complete
Inner fold 3/4 complete
Inner fold 4/4 complete
Selected C=10.0, Qwen weight=0.25, inner OOF=0.3802
Outer baseline: 0.3889
Outer selected: 0.3907
Delta: +0.0019

========== OUTER FOLD 2/5 ==========
Inner fold 1/4 complete
Inner fold 2/4 complete
Inner fold 3/4 complete
Inner fold 4/4 complete
Selected C=3.0, Qwen weight=0.35, inner OOF=0.3834
Outer baseline: 0.3463
Outer selected: 0.3796
Delta: +0.0333

========== OUTER FOLD 3/5 ==========
Inner fold 1/4 complete
Inner fold 2/4 complete
Inner fold 3/4 complete
Inner fold 4/4 complete
Selected C=3.0, Qwen weight=0.35, inner OOF=0.3723
Outer baseline: 0.3815
Outer selected: 0.4185
Delta: +0.0370

========== OUTER FOLD 4/5 ==========
Inner fold 1/4 complete
Inner fold 2/4 complete
Inner fold 3/4 complete
Inner fold 4/4 complete
Selected C=3.0, Qwen weight=0.25, inner OOF=0.3751
Outer baseline: 0.3796
Outer selected: 0.4056
Delta: +0.0259

========== OUT

,Fold,Selected C,Selected Qwen weight,Inner OOF score,Outer baseline,Outer selected,Delta
0,1,10.0,0.25,0.3802,0.3889,0.3907,0.0019
1,2,3.0,0.35,0.3834,0.3463,0.3796,0.0333
2,3,3.0,0.35,0.3723,0.3815,0.4185,0.0370
3,4,3.0,0.25,0.3751,0.3796,0.4056,0.0259
4,5,10.0,0.25,0.3861,0.3521,0.3725,0.0204



========== POOLED OOF RESULTS ==========


,Method,Pooled OOF balanced accuracy
0,Text baseline,0.3697
1,Nested C + weight selection,0.3934



Gain over baseline: 0.0237


In [1]:
# ============================================================
# SESSION RECOVERY: RESTORE DATA, HELPERS AND QWEN FEATURES
# Run this cell before the final training/submission cell.
# ============================================================

import hashlib
import pathlib
import re
import warnings

import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. Find and load competition data
# ------------------------------------------------------------

input_root = pathlib.Path("/kaggle/input")
train_path = None

for path in input_root.rglob("train.csv"):
    try:
        columns = set(pd.read_csv(path, nrows=0).columns)
        if {"text", "model", "prompt_id"}.issubset(columns):
            train_path = path
            break
    except Exception:
        continue

if train_path is None:
    raise FileNotFoundError(
        "Could not find train.csv containing text, model, prompt_id. "
        "Attach the Name That LLM competition dataset."
    )

test_path = train_path.parent / "test.csv"

train_df = pd.read_csv(train_path)
test_df = (
    pd.read_csv(test_path)
    if test_path.exists()
    else pd.DataFrame(columns=["id", "text"])
)

X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].astype(str).reset_index(drop=True)
groups = train_df["prompt_id"].astype(str).reset_index(drop=True)

labels = np.sort(y.unique())

if "text" not in test_df.columns:
    raise ValueError("test.csv is missing the text column.")

test_df["text"] = test_df["text"].fillna("").astype(str)

print("Training file:", train_path)
print("Rows:", len(X))
print("Prompts:", groups.nunique())
print("Models:", len(labels))
print("Test rows:", len(test_df))

# ------------------------------------------------------------
# 2. Restore style feature extraction
# ------------------------------------------------------------

PUNCTUATION = [
    ",", ";", ":", ".", "!", "?", "-", "—", "–", "'", "’",
    '"', "“", "”", "(", ")", "[", "]", "{", "}", "/",
    "\\", "*", "_", "#", "`", "…", "•"
]

PHRASES = [
    "certainly", "of course", "for example", "for instance",
    "in summary", "in conclusion", "however", "therefore",
    "moreover", "furthermore", "it is important", "overall",
    "hope this helps", "let's", "here is",
    "as a language model", "note that"
]


def extract_style(text):
    text = str(text)
    lower = text.lower()
    length = max(len(text), 1)

    words = re.findall(r"\b\w+\b", lower)
    word_count = max(len(words), 1)
    word_lengths = [len(w) for w in words]

    sentences = [
        s.strip() for s in re.split(r"[.!?]+", text) if s.strip()
    ]

    sentence_lengths = [
        len(re.findall(r"\b\w+\b", s)) for s in sentences
    ]

    lines = text.splitlines() or [text]
    paragraphs = [
        p for p in re.split(r"\n\s*\n", text) if p.strip()
    ]

    features = [
        np.log1p(len(text)),
        np.log1p(len(words)),
        np.log1p(len(sentences)),
        np.log1p(len(lines)),
        np.log1p(len(paragraphs)),
        np.mean(word_lengths) if word_lengths else 0,
        np.std(word_lengths) if word_lengths else 0,
        np.max(word_lengths) if word_lengths else 0,
        len(set(words)) / word_count,
        sum(len(w) <= 3 for w in words) / word_count,
        sum(len(w) >= 8 for w in words) / word_count,
        np.mean(sentence_lengths) if sentence_lengths else 0,
        np.std(sentence_lengths) if sentence_lengths else 0,
        np.max(sentence_lengths) if sentence_lengths else 0,
        sum(c.isupper() for c in text) / length,
        sum(c.islower() for c in text) / length,
        sum(c.isdigit() for c in text) / length,
        sum(c.isspace() for c in text) / length,
        sum(ord(c) > 127 for c in text) / length,
    ]

    features.extend(text.count(p) / length for p in PUNCTUATION)

    bullet_count = sum(
        bool(re.match(r"^\s*(?:[-*•]\s+|\d+[.)]\s+)", line))
        for line in lines
    )

    features.extend([
        bullet_count / max(len(lines), 1),
        float("\n" in text),
        float("```" in text),
        float("##" in text),
        float(text.rstrip().endswith(("!", "?", "."))),
    ])

    features.extend(float(phrase in lower) for phrase in PHRASES)

    return features


def make_style_matrix(texts):
    return np.asarray(
        [extract_style(t) for t in texts],
        dtype=np.float32
    )


# ------------------------------------------------------------
# 3. Check cached Qwen features
# ------------------------------------------------------------

MODEL_ID = "Qwen/Qwen3-1.7B"
MAX_LENGTH = 512

all_texts = X.tolist() + test_df["text"].tolist()

digest = hashlib.sha256()
digest.update(
    f"{MODEL_ID}|{MAX_LENGTH}|last+mean-v1".encode()
)

for text in all_texts:
    digest.update(text.encode("utf-8", errors="replace"))
    digest.update(b"\0")

cache_key = digest.hexdigest()

cache_path = pathlib.Path(
    "/kaggle/working/experiment_c_qwen_features.npz"
)

qwen_last = None
qwen_mean = None

if cache_path.exists():
    try:
        cached = np.load(cache_path, allow_pickle=False)

        if str(cached["cache_key"].item()) == cache_key:
            qwen_last = cached["qwen_last"].astype(np.float32)
            qwen_mean = cached["qwen_mean"].astype(np.float32)

            print("Reusing cached Qwen features.")

    except Exception as exc:
        print("Cache could not be reused:", exc)

# ------------------------------------------------------------
# 4. Re-extract features only if the matching cache is missing
# ------------------------------------------------------------

if qwen_last is None:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Qwen3-1.7B features are not cached and no GPU is available. "
            "Enable a Kaggle GPU and rerun this cell."
        )

    from transformers import AutoTokenizer, AutoModel

    device = torch.device("cuda")
    batch_size = 4

    print("Extracting Qwen3-1.7B features on GPU...")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        use_fast=True
    )

    tokenizer.padding_side = "right"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    qwen_model = AutoModel.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16
    ).to(device)

    qwen_model.eval()
    qwen_model.config.use_cache = False

    last_parts = []
    mean_parts = []

    with torch.inference_mode():
        for start in range(0, len(all_texts), batch_size):
            batch = tokenizer(
                all_texts[start:start + batch_size],
                max_length=MAX_LENGTH,
                truncation=True,
                padding=True,
                return_tensors="pt"
            )

            batch = {k: v.to(device) for k, v in batch.items()}

            hidden = qwen_model(
                **batch,
                use_cache=False,
                return_dict=True
            ).last_hidden_state

            mask = batch["attention_mask"].to(hidden.dtype)
            token_counts = mask.sum(dim=1).clamp(min=1)

            mean_pool = (
                (hidden * mask.unsqueeze(-1)).sum(dim=1)
                / token_counts.unsqueeze(-1)
            )

            last_positions = token_counts.long() - 1
            row_indices = torch.arange(hidden.shape[0], device=device)

            last_pool = hidden[row_indices, last_positions]

            last_pool = torch.nn.functional.normalize(
                last_pool.float(), p=2, dim=1
            )

            mean_pool = torch.nn.functional.normalize(
                mean_pool.float(), p=2, dim=1
            )

            last_parts.append(last_pool.cpu().numpy().astype(np.float16))
            mean_parts.append(mean_pool.cpu().numpy().astype(np.float16))

            if (start // batch_size) % 50 == 0:
                print(f"Processed {min(start + batch_size, len(all_texts))}/{len(all_texts)}")

    qwen_last = np.concatenate(last_parts).astype(np.float32)
    qwen_mean = np.concatenate(mean_parts).astype(np.float32)

    np.savez_compressed(
        cache_path,
        cache_key=np.asarray(cache_key),
        qwen_last=qwen_last.astype(np.float16),
        qwen_mean=qwen_mean.astype(np.float16)
    )

    del qwen_model, tokenizer
    torch.cuda.empty_cache()

    print("Qwen features extracted and cached.")

# ------------------------------------------------------------
# 5. Restore train/test feature arrays
# ------------------------------------------------------------

n_train = len(X)

if len(qwen_last) != len(X) + len(test_df):
    raise ValueError(
        "Qwen feature count does not match the current dataset. "
        "Check the training/test CSVs and feature cache."
    )

qwen_train_last = qwen_last[:n_train]
qwen_train_mean = qwen_mean[:n_train]

qwen_test_last = qwen_last[n_train:]
qwen_test_mean = qwen_mean[n_train:]

print("\nSETUP COMPLETE")
print("Training rows:", len(X))
print("Test rows:", len(test_df))
print("Qwen feature shape:", qwen_train_last.shape)
print("Style function:", callable(make_style_matrix))
print("Ready to run the final training/submission cell.")

Training file: /kaggle/input/competitions/name-that-llm/train.csv
Rows: 2699
Prompts: 150
Models: 18
Test rows: 3


Extracting Qwen3-1.7B features on GPU...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] Qwen3Model LOAD REPORT from: Qwen/Qwen3-1.7B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processed 4/2702
Processed 204/2702
Processed 404/2702
Processed 604/2702
Processed 804/2702
Processed 1004/2702
Processed 1204/2702
Processed 1404/2702
Processed 1604/2702
Processed 1804/2702
Processed 2004/2702
Processed 2204/2702
Processed 2404/2702
Processed 2604/2702
Qwen features extracted and cached.

SETUP COMPLETE
Training rows: 2699
Test rows: 3
Qwen feature shape: (2699, 2048)
Style function: True
Ready to run the final training/submission cell.


In [2]:
# ============================================================
# SESSION RECOVERY: RESTORE DATA, HELPERS AND QWEN FEATURES
# Run this cell before the final training/submission cell.
# ============================================================

import hashlib
import pathlib
import re
import warnings

import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. Find and load competition data
# ------------------------------------------------------------

input_root = pathlib.Path("/kaggle/input")
train_path = None

for path in input_root.rglob("train.csv"):
    try:
        columns = set(pd.read_csv(path, nrows=0).columns)
        if {"text", "model", "prompt_id"}.issubset(columns):
            train_path = path
            break
    except Exception:
        continue

if train_path is None:
    raise FileNotFoundError(
        "Could not find train.csv containing text, model, prompt_id. "
        "Attach the Name That LLM competition dataset."
    )

test_path = train_path.parent / "test.csv"

train_df = pd.read_csv(train_path)
test_df = (
    pd.read_csv(test_path)
    if test_path.exists()
    else pd.DataFrame(columns=["id", "text"])
)

X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].astype(str).reset_index(drop=True)
groups = train_df["prompt_id"].astype(str).reset_index(drop=True)

labels = np.sort(y.unique())

if "text" not in test_df.columns:
    raise ValueError("test.csv is missing the text column.")

test_df["text"] = test_df["text"].fillna("").astype(str)

print("Training file:", train_path)
print("Rows:", len(X))
print("Prompts:", groups.nunique())
print("Models:", len(labels))
print("Test rows:", len(test_df))

# ------------------------------------------------------------
# 2. Restore style feature extraction
# ------------------------------------------------------------

PUNCTUATION = [
    ",", ";", ":", ".", "!", "?", "-", "—", "–", "'", "’",
    '"', "“", "”", "(", ")", "[", "]", "{", "}", "/",
    "\\", "*", "_", "#", "`", "…", "•"
]

PHRASES = [
    "certainly", "of course", "for example", "for instance",
    "in summary", "in conclusion", "however", "therefore",
    "moreover", "furthermore", "it is important", "overall",
    "hope this helps", "let's", "here is",
    "as a language model", "note that"
]


def extract_style(text):
    text = str(text)
    lower = text.lower()
    length = max(len(text), 1)

    words = re.findall(r"\b\w+\b", lower)
    word_count = max(len(words), 1)
    word_lengths = [len(w) for w in words]

    sentences = [
        s.strip() for s in re.split(r"[.!?]+", text) if s.strip()
    ]

    sentence_lengths = [
        len(re.findall(r"\b\w+\b", s)) for s in sentences
    ]

    lines = text.splitlines() or [text]
    paragraphs = [
        p for p in re.split(r"\n\s*\n", text) if p.strip()
    ]

    features = [
        np.log1p(len(text)),
        np.log1p(len(words)),
        np.log1p(len(sentences)),
        np.log1p(len(lines)),
        np.log1p(len(paragraphs)),
        np.mean(word_lengths) if word_lengths else 0,
        np.std(word_lengths) if word_lengths else 0,
        np.max(word_lengths) if word_lengths else 0,
        len(set(words)) / word_count,
        sum(len(w) <= 3 for w in words) / word_count,
        sum(len(w) >= 8 for w in words) / word_count,
        np.mean(sentence_lengths) if sentence_lengths else 0,
        np.std(sentence_lengths) if sentence_lengths else 0,
        np.max(sentence_lengths) if sentence_lengths else 0,
        sum(c.isupper() for c in text) / length,
        sum(c.islower() for c in text) / length,
        sum(c.isdigit() for c in text) / length,
        sum(c.isspace() for c in text) / length,
        sum(ord(c) > 127 for c in text) / length,
    ]

    features.extend(text.count(p) / length for p in PUNCTUATION)

    bullet_count = sum(
        bool(re.match(r"^\s*(?:[-*•]\s+|\d+[.)]\s+)", line))
        for line in lines
    )

    features.extend([
        bullet_count / max(len(lines), 1),
        float("\n" in text),
        float("```" in text),
        float("##" in text),
        float(text.rstrip().endswith(("!", "?", "."))),
    ])

    features.extend(float(phrase in lower) for phrase in PHRASES)

    return features


def make_style_matrix(texts):
    return np.asarray(
        [extract_style(t) for t in texts],
        dtype=np.float32
    )


# ------------------------------------------------------------
# 3. Check cached Qwen features
# ------------------------------------------------------------

MODEL_ID = "Qwen/Qwen3-1.7B"
MAX_LENGTH = 512

all_texts = X.tolist() + test_df["text"].tolist()

digest = hashlib.sha256()
digest.update(
    f"{MODEL_ID}|{MAX_LENGTH}|last+mean-v1".encode()
)

for text in all_texts:
    digest.update(text.encode("utf-8", errors="replace"))
    digest.update(b"\0")

cache_key = digest.hexdigest()

cache_path = pathlib.Path(
    "/kaggle/working/experiment_c_qwen_features.npz"
)

qwen_last = None
qwen_mean = None

if cache_path.exists():
    try:
        cached = np.load(cache_path, allow_pickle=False)

        if str(cached["cache_key"].item()) == cache_key:
            qwen_last = cached["qwen_last"].astype(np.float32)
            qwen_mean = cached["qwen_mean"].astype(np.float32)

            print("Reusing cached Qwen features.")

    except Exception as exc:
        print("Cache could not be reused:", exc)

# ------------------------------------------------------------
# 4. Re-extract features only if the matching cache is missing
# ------------------------------------------------------------

if qwen_last is None:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Qwen3-1.7B features are not cached and no GPU is available. "
            "Enable a Kaggle GPU and rerun this cell."
        )

    from transformers import AutoTokenizer, AutoModel

    device = torch.device("cuda")
    batch_size = 4

    print("Extracting Qwen3-1.7B features on GPU...")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        use_fast=True
    )

    tokenizer.padding_side = "right"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    qwen_model = AutoModel.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16
    ).to(device)

    qwen_model.eval()
    qwen_model.config.use_cache = False

    last_parts = []
    mean_parts = []

    with torch.inference_mode():
        for start in range(0, len(all_texts), batch_size):
            batch = tokenizer(
                all_texts[start:start + batch_size],
                max_length=MAX_LENGTH,
                truncation=True,
                padding=True,
                return_tensors="pt"
            )

            batch = {k: v.to(device) for k, v in batch.items()}

            hidden = qwen_model(
                **batch,
                use_cache=False,
                return_dict=True
            ).last_hidden_state

            mask = batch["attention_mask"].to(hidden.dtype)
            token_counts = mask.sum(dim=1).clamp(min=1)

            mean_pool = (
                (hidden * mask.unsqueeze(-1)).sum(dim=1)
                / token_counts.unsqueeze(-1)
            )

            last_positions = token_counts.long() - 1
            row_indices = torch.arange(hidden.shape[0], device=device)

            last_pool = hidden[row_indices, last_positions]

            last_pool = torch.nn.functional.normalize(
                last_pool.float(), p=2, dim=1
            )

            mean_pool = torch.nn.functional.normalize(
                mean_pool.float(), p=2, dim=1
            )

            last_parts.append(last_pool.cpu().numpy().astype(np.float16))
            mean_parts.append(mean_pool.cpu().numpy().astype(np.float16))

            if (start // batch_size) % 50 == 0:
                print(f"Processed {min(start + batch_size, len(all_texts))}/{len(all_texts)}")

    qwen_last = np.concatenate(last_parts).astype(np.float32)
    qwen_mean = np.concatenate(mean_parts).astype(np.float32)

    np.savez_compressed(
        cache_path,
        cache_key=np.asarray(cache_key),
        qwen_last=qwen_last.astype(np.float16),
        qwen_mean=qwen_mean.astype(np.float16)
    )

    del qwen_model, tokenizer
    torch.cuda.empty_cache()

    print("Qwen features extracted and cached.")

# ------------------------------------------------------------
# 5. Restore train/test feature arrays
# ------------------------------------------------------------

n_train = len(X)

if len(qwen_last) != len(X) + len(test_df):
    raise ValueError(
        "Qwen feature count does not match the current dataset. "
        "Check the training/test CSVs and feature cache."
    )

qwen_train_last = qwen_last[:n_train]
qwen_train_mean = qwen_mean[:n_train]

qwen_test_last = qwen_last[n_train:]
qwen_test_mean = qwen_mean[n_train:]

print("\nSETUP COMPLETE")
print("Training rows:", len(X))
print("Test rows:", len(test_df))
print("Qwen feature shape:", qwen_train_last.shape)
print("Style function:", callable(make_style_matrix))
print("Ready to run the final training/submission cell.")

Training file: /kaggle/input/competitions/name-that-llm/train.csv
Rows: 2699
Prompts: 150
Models: 18
Test rows: 3
Reusing cached Qwen features.

SETUP COMPLETE
Training rows: 2699
Test rows: 3
Qwen feature shape: (2699, 2048)
Style function: True
Ready to run the final training/submission cell.


In [3]:
import os
import pandas as pd

path = "/kaggle/working/submission_final.csv"

assert os.path.exists(path), "Submission file was not created."

submission = pd.read_csv(path)

print("Submission shape:", submission.shape)
display(submission)

AssertionError: Submission file was not created.